# Facility Finder · Datathon audit and reproducibility notebook

**Intended final app:** the restored Facility Finder at `d682872` on `restore-care-finder-map`, as identified in the latest Facility Finder Development Report. The project’s default branch at `af89658` carries a different Databricks entry point and the fuller research history. This notebook pins both. The offline recommendation benchmark is from an earlier main commit, `18335aa`. [Restored Facility Finder](https://github.com/augustchu/datathon-F26/tree/d68287276f3e9a275283eb005799c3be7db3bba5/first-prototype) · [Main research repository](https://github.com/augustchu/datathon-F26/tree/af8965890e4de2e5a5f2a24177425e5fdb77352d)

A care worker supplies an Indian location and a named service. The app compares documented branch-level service evidence and travel, shows the ordered shortlist with a map and contact details, and can compare projected demand or paired equity evidence where supplied. A bed estimate describes facility scale. The historical benchmark and load scenarios measure offline ranking behavior on fixed cases.

### Run this notebook in Databricks

Import this `.ipynb` as a Python notebook and attach a recent Databricks ML Runtime (Python 3.11–3.12) or a compatible serverless environment. Run installation and the restart cell first. In preflight, set the three organizer dataset paths, or use the configured read-only Unity Catalog share. A Unity Catalog volume is convenient for large files. Place **Facility Finder Audit Assets.zip** in the same accessible folder as the notebook or supply its absolute path in preflight. It contains exact public-source snapshots and the local reports and workbooks. Keep the notebook and companion in the project workspace.

The primary GitHub repository is private. Use an authenticated Git checkout visible to the driver or Databricks secret scope/key with a GitHub read token. The standalone verification tool is public. Poppler and Node are used for raw PDF parsing and browser-tool tests when available on the driver. The supplied extracted tables let the data pipeline continue without Poppler. The notebook writes analysis files to driver scratch by default, with an optional writable volume path. The archived plots and receipts are available without live clinical data, API calls or training. Set `download_all_data` to `yes` to also stage the large public raster and request cited publisher documents. Set `replay_level` to `full` for budgeted retraining, the population raster and slower replays. The strict capacity replays check that NumPy, scikit-learn and XGBoost use a single OpenMP runtime. If the driver packages load separate runtimes, set `capacity_python` to an existing clean CPU environment with those packages linked to one OpenMP runtime. Live routing and historical model tracking have separate opt-ins where used.

[Import a Databricks notebook](https://docs.databricks.com/aws/en/notebooks/notebook-export-import) · [Notebook-scoped dependencies](https://docs.databricks.com/aws/en/libraries/notebooks-python-libraries) · [Workspace files and volumes](https://docs.databricks.com/aws/en/files/files-recommendations)

| Read in order | Evidence and outputs |
|---|---|
| 01 · Setup and data lineage | Checked paths and source types, verified code pins, external snapshots |
| 02 · Collection, cleaning and supplementation | Coverage, scope gates, branch linkage, review samples and selector pilots |
| Model development | Count EDA, model controls, failed experiments, ablations, final model selection |
| Capacity publication | Rebuild conditions, delivery hash checks, row alignment and confidence |
| Side projects | Recommendation research, geography, load, equity, IVR and the verification app |
| App, benchmarks and audit | Intended branch, main comparison, replay, captured evidence and test matrix |
| Document index | Repository history, workshop reports, workbooks and reproducibility limitations |

In [ ]:
%pip install "numpy==2.1.3" "pandas==2.2.3" "scipy==1.15.3" "scikit-learn==1.7.2" "xgboost==3.4.1" "matplotlib==3.10.1" "seaborn==0.13.2" "torch==2.7.1" "pyarrow==18.1.0" "nbformat==5.10.4" "joblib==1.5.2" "psutil==7.0.0" "cloudpickle==3.1.1" "openpyxl==3.1.5" "beautifulsoup4==4.13.4" "requests==2.32.5" "pdfplumber==0.11.11" "geopandas==1.0.1" "rasterio==1.4.3" "exactextract==0.2.2" "rapidfuzz==3.13.0" "streamlit==1.45.1" "folium==0.19.7" "databricks-sdk==0.54.0" "databricks-sql-connector==4.0.5" "websocket-client==1.8.0" "pytest==8.3.5" "wandb==0.31.2" "skops==0.16.0" "mlflow==2.22.0"

In [ ]:
if "dbutils" in globals():
    dbutils.library.restartPython()

## 01 · Validate the organizer’s datasets

The three original tables are the Virtue Foundation facility directory (10,088 source records × 51 fields), India Post PIN directory (165,627 records in the frozen research export), and NFHS-5 district indicators (706 records). A different edition can change IDs, matching, and benchmark hashes. Use JSON exports with `columns` and `rows`, Parquet, or CSV with the original headers and facility row order. A Unity Catalog table can be materialized to scratch. Use ordered exported files for exact row-order checks because a table scan can produce a different order. The organizer’s NFHS indicators and the later WorldPop-calibrated district population table have separate roles.

Set `input_folder` to a workspace folder or volume with the named files. You can override the paths individually. An empty widget uses a detected workspace file, a readable organizer catalog table, or the pinned GitHub CSV in that order. A specified file that fails validation is flagged for correction. Dataset-dependent replays require the indicated files. The published result and audit sections can still be viewed.

[Organizer table names and configuration](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/non-ml/config/project.json)

In [ ]:
import csv
import gzip
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tarfile
import tempfile
import zipfile

import matplotlib.pyplot as plt
import pandas as pd
import requests
from IPython.display import Image, Markdown, display

WIDGETS = {
    "input_folder": "", "facilities_path": "", "pin_path": "", "nfhs_path": "",
    "catalog_schema": "databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset",
    "main_git_checkout": "", "restored_git_checkout": "", "verification_git_checkout": "",
    "github_secret_scope": "", "github_secret_key": "",
    "assets_zip": "", "output_folder": "", "density_path": "", "capacity_archive": "",
    "replay_level": "standard", "download_all_data": "no", "capacity_python": "",
}

if "dbutils" in globals():
    for name, default in WIDGETS.items():
        try:
            dbutils.widgets.get(name)
        except Exception:
            if name == "replay_level":
                dbutils.widgets.dropdown(name, default, ["standard", "full"])
            elif name == "download_all_data":
                dbutils.widgets.dropdown(name, default, ["no", "yes"])
            else:
                dbutils.widgets.text(name, default)

def setting(key):
    if "dbutils" in globals():
        return dbutils.widgets.get(key).strip()
    return os.environ.get(f"DATATHON_AUDIT_{key.upper()}", WIDGETS[key]).strip()

def local_path(path):
    value = str(path)
    if value.startswith("dbfs:/Volumes/"):
        value = value.removeprefix("dbfs:")
    elif value.startswith("dbfs:/"):
        value = "/dbfs/" + value.removeprefix("dbfs:/")
    elif value.startswith("file:"):
        value = value.removeprefix("file:")
    return Path(value).expanduser().resolve()

work_folder = setting("output_folder")
if work_folder:
    local_path(work_folder).mkdir(parents=True, exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix="facility-finder-audit-", dir=local_path(work_folder) if work_folder else None))
FULL_REPLAY = setting("replay_level") == "full"
CPU_PYTHON = local_path(setting("capacity_python")) if setting("capacity_python") else Path(sys.executable)
if not CPU_PYTHON.is_file():
    raise FileNotFoundError(f"capacity_python must point to a Python executable: {CPU_PYTHON}")
INPUT_FOLDER = local_path(setting("input_folder")) if setting("input_folder") else Path.cwd()
print(f"Run directory: {RUN} · Full replay: {FULL_REPLAY}")

def load_json(path):
    path = Path(path)
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as stream:
        return json.load(stream)

def read_csv(path, **kwargs):
    kwargs.setdefault("low_memory", False)
    return pd.read_csv(path, **kwargs)

def digest(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def project_frame(path):
    path = Path(path)
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".csv":
        return read_csv(path, dtype="string", keep_default_na=False)
    payload = load_json(path)
    return pd.DataFrame(payload["rows"], columns=payload["columns"], dtype=object)

def run_cmd(args, cwd=None, optional=False, *, env=None):
    argv = [str(value) for value in args]
    try:
        result = subprocess.run(argv, cwd=cwd, env=env, text=True, capture_output=True, check=False)
    except FileNotFoundError as error:
        if not optional:
            raise
        result = subprocess.CompletedProcess(argv, 127, "", str(error))
    summary = f"{Path(argv[0]).name} · exit {result.returncode}"
    print(summary)
    if result.stdout.strip():
        print(result.stdout.strip()[-2200:])
    if result.stderr.strip():
        print(result.stderr.strip()[-1600:])
    if result.returncode:
        if not optional:
            result.check_returncode()
    return result

SOURCE_TABLES = {
    "facilities": ("facilities", ("facilities dataset",), {"unique_id", "countries", "capacity", "numberDoctors"}, 10088),
    "pin": ("india_post_pincode_directory", ("pincodes",), {"pincode", "district", "statename"}, 165627),
    "nfhs": ("nfhs_5_district_health_indicators", ("NFHS-5 Dataset",), {"district_name", "state_ut"}, 706),
}
SRC = {}
source_rows = []
for key, (table, aliases, required, expected) in SOURCE_TABLES.items():
    explicit = setting(f"{key}_path")
    table_name = f"{setting('catalog_schema')}.{table}"
    candidates = [local_path(explicit)] if explicit else [
        folder / (stem + suffix)
        for folder in (INPUT_FOLDER, INPUT_FOLDER / "data", INPUT_FOLDER / "project-release")
        for stem in (table, *aliases)
        for suffix in (".json.gz", ".json", ".parquet", ".csv")
    ]
    path = next((candidate for candidate in candidates if candidate.is_file()), None)
    row = {"dataset": key, "frozen_rows": expected, "input": str(path or explicit or table_name)}
    use_catalog = path is None and not explicit and "spark" in globals() and bool(setting("catalog_schema"))
    try:
        if use_catalog:
            table_df = spark.table(table_name)
            rows = table_df.limit(max(expected * 2, 25000) + 1).collect()
            if len(rows) > max(expected * 2, 25000):
                raise ValueError("Row count exceeds the expected project snapshot range")
            path = RUN / "organizer" / f"{table}.json.gz"
            path.parent.mkdir(exist_ok=True)
            with gzip.open(path, "wt", encoding="utf-8") as stream:
                json.dump({"columns": table_df.columns, "rows": [row.asDict(recursive=True) for row in rows]},
                          stream, ensure_ascii=False, separators=(",", ":"), default=str)
        if path is None:
            raise FileNotFoundError("Set the path widget or make the organizer table readable")
        frame = project_frame(path)
        missing = required.difference(frame.columns)
        if missing:
            raise ValueError(f"Missing columns: {sorted(missing)}")
        if key == "facilities" and len(frame.columns) != 51:
            raise ValueError(f"Expected 51 original fields, received {len(frame.columns)}")
        row.update(rows=len(frame), columns=len(frame.columns),
                   status="ready" if len(frame) == expected else "ready · differs from frozen row count",
                   sha256=digest(path))
        SRC[key] = path
        del frame
    except Exception as error:
        SRC[key] = None
        row["status"] = f"{type(error).__name__}: {str(error).splitlines()[0][:250]}"
    source_rows.append(row)

SRC["facility_catalog"] = f"{setting('catalog_schema')}.facilities" if "spark" in globals() and setting("catalog_schema") else ""
SRC["density"] = local_path(setting("density_path")) if setting("density_path") else None
print("Workspace paths checked. The next section resolves any remaining datasets and displays the final paths.")

### Fetch the verified Git trees

The app branch split before later main research commits. Each tree is fetched and pinned separately. The clone uses existing Git authentication when configured. The optional Databricks secret is passed to Git via a process environment, with no credential in a repository URL. The snapshot directories are separate from checkouts and analysis outputs.

| Source | Pin | Use |
|---|---|---|
| Project `main` | `af89658` | Research, tests, most documents and the default app |
| `restore-care-finder-map` | `d682872` | Intended final app, classic UI and later scorer |
| In The Loop standalone `main` | `fe2c9db` | Deployed verification side project |
| `capacity-newest-2026-11-22` | `b3270ab` | Tagged publication and separately hosted model assets |
| Earlier release-input snapshot | `ff7dd5e` | Byte-exact dataset for published caseload receipts |

In [ ]:
import base64
from contextlib import contextmanager

MAIN_SHA = "af8965890e4de2e5a5f2a24177425e5fdb77352d"
APP_SHA = "d68287276f3e9a275283eb005799c3be7db3bba5"
VERIFY_SHA = "fe2c9db5c29e1eebacee656d9002aececdf69c1c"
PROJECT_URL = "https://github.com/augustchu/datathon-F26.git"
VERIFY_URL = "https://github.com/yana-157/in-the-loop-verification-tool.git"

def git_environment():
    result = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    scope, key = setting("github_secret_scope"), setting("github_secret_key")
    token = dbutils.secrets.get(scope=scope, key=key) if scope and key and "dbutils" in globals() else os.getenv("GH_TOKEN")
    if token:
        authorization = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        count = int(result.get("GIT_CONFIG_COUNT", "0"))
        result.update({
            "GIT_CONFIG_COUNT": str(count + 1),
            f"GIT_CONFIG_KEY_{count}": "http.https://github.com/.extraheader",
            f"GIT_CONFIG_VALUE_{count}": f"Authorization: Basic {authorization}",
        })
    return result

def git(*args, cwd=None):
    result = subprocess.run(["git", *map(str, args)], cwd=cwd, env=git_environment(), capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Git {args[0]} could not read the requested revision. Check checkout paths, GitHub access and egress.")
    return result.stdout

def get_checkout(widget, url, name, fallback=None):
    configured = setting(widget)
    if configured:
        folder = local_path(configured)
        git("rev-parse", "--git-dir", cwd=folder)
        return folder
    if fallback is not None:
        return fallback
    folder = RUN / "git" / name
    folder.parent.mkdir(exist_ok=True)
    git("clone", "--quiet", "--no-checkout", url, folder)
    return folder

def snapshot(checkout, revision, reference, url, name):
    try:
        git("cat-file", "-e", f"{revision}^{{commit}}", cwd=checkout)
    except RuntimeError:
        git("fetch", "--no-tags", url, reference, cwd=checkout)
    if git("rev-parse", f"{revision}^{{commit}}", cwd=checkout).decode().strip() != revision:
        raise ValueError(f"Unexpected revision for {name}")
    folder = RUN / "snapshots" / name
    folder.mkdir(parents=True, exist_ok=True)
    archive_path = folder.parent / (name + ".tar")
    git("archive", f"--output={archive_path}", revision, cwd=checkout)
    with tarfile.open(archive_path) as archive:
        archive.extractall(folder, filter="data")
    archive_path.unlink()
    print(f"{name} · {revision}")
    return folder

MAIN_GIT = get_checkout("main_git_checkout", PROJECT_URL, "project")
APP_GIT = get_checkout("restored_git_checkout", PROJECT_URL, "restored", fallback=MAIN_GIT)
VERIFICATION_GIT = get_checkout("verification_git_checkout", VERIFY_URL, "verification")
ROOT = snapshot(MAIN_GIT, MAIN_SHA, "refs/heads/main", PROJECT_URL, "main")
APP_ROOT = snapshot(APP_GIT, APP_SHA, "refs/heads/restore-care-finder-map", PROJECT_URL, "restored")
VERIFICATION_ROOT = snapshot(VERIFICATION_GIT, VERIFY_SHA, "refs/heads/main", VERIFY_URL, "verification")

### Resolve the organizer input paths

The private project checkout includes CSV mirrors of all three organizer tables. This cell stages a checksum-verified copy whenever the preflight could use neither a workspace file nor the Unity Catalog table. The CSVs retain the expected schemas, row counts and row keys. Some values have different CSV representations, such as a null marker or a numeric string. For an archived hash comparison, supply the original ordered exports through the filepath widgets. The manifest shows which copy each replay will read.

To use the organizer data directly, open **Catalog** in the Databricks workspace with the challenge share and verify access to `facilities`, `india_post_pincode_directory`, and `nfhs_5_district_health_indicators` under `databricks_virtue_foundation_dataset_dais_2026.virtue_foundation_dataset`. On Run All, the earlier preflight materializes readable tables as JSON files in the run directory. You can also place approved exports in a workspace folder or volume and set `input_folder` or the three individual paths.

[Data download locations](https://github.com/augustchu/datathon-F26/blob/restore-care-finder-map/facility-finder-audit/README.md#data-downloads)

In [ ]:
organizer_mirrors = {
    "facilities": ("facilities dataset.csv", "47bb1aad11f84a68aed3025d2887665bb164bc38c535c39e59eddb436c556691"),
    "pin": ("india_post_pincode_directory.csv", "47cd7285d6bf0b2aa45d0893bf46eff99ab7f0c4b84331482f4079a49c0edf38"),
    "nfhs": ("NFHS-5 Dataset.csv", "583b2fd98d60c3901a230bcce41bfddc0aea1109fc0814db0a500f7a5e9a0547"),
}
for entry in source_rows:
    key = entry["dataset"]
    if SRC[key] is not None:
        entry["resolved_path"] = str(SRC[key])
        continue
    if setting(f"{key}_path"):
        continue
    filename, expected_sha = organizer_mirrors[key]
    mirror = ROOT / "load-balancing-pipeline" / filename
    if not mirror.is_file() or digest(mirror) != expected_sha:
        raise ValueError(f"The pinned GitHub copy for {key} is missing or failed its checksum.")
    table, _, required, expected_rows = SOURCE_TABLES[key]
    path = RUN / "organizer" / f"{table}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(mirror, path)
    frame = project_frame(path)
    if len(frame) != expected_rows or not required.issubset(frame.columns):
        raise ValueError(f"The GitHub copy for {key} does not match the organizer input schema.")
    if key == "facilities" and len(frame.columns) != 51:
        raise ValueError("The GitHub facilities copy must preserve the 51 original fields.")
    SRC[key] = path
    entry.update(input=f"GitHub {MAIN_SHA[:7]} · {filename}", resolved_path=str(path),
                 status="ready · pinned GitHub CSV", rows=len(frame), columns=len(frame.columns), sha256=expected_sha)
    del frame

organizer_manifest = pd.DataFrame(source_rows)
organizer_manifest.to_csv(RUN / "organizer-data-manifest.csv", index=False)
display(organizer_manifest)

### Acquire the external inputs

The repository snapshot already downloads the archived public registry and government exports, external census and state projection tables, and historical boundary geometries. The companion archive supplies the exact insurer disclosures and HTML pages used during development, including public pages that changed or were blocked during collection. The download cell retrieves the missing public inputs from their publishers when an archived copy is unavailable and verifies frozen checksums when known. A current page whose bytes differ is labeled as a new snapshot.

The official WorldPop 2020 India raster is about 466 MiB. A full replay downloads it and rebuilds the population experiment. Standard replay uses the published district aggregates. Set `download_all_data` to `yes` to download the raster and request the cited publisher documents in either replay mode. The original teammate-provided population-density file was never supplied in the audited experiments. An optional compatible density dataset can be provided explicitly for a new comparison.

[geoBoundaries API](https://www.geoboundaries.org/api.html) · [WorldPop India constrained 2020 raster](https://data.worldpop.org/GIS/Population/Global_2000_2020_Constrained/2020/BSGM/IND/)

In [ ]:
ASSETS = RUN / "audit-assets"
ASSETS.mkdir(exist_ok=True)
assets_name = "Facility Finder Audit Assets.zip"
zip_options = ([local_path(setting("assets_zip"))] if setting("assets_zip") else
               [INPUT_FOLDER / assets_name, Path.cwd() / assets_name])
assets_zip = next((path for path in zip_options if path.is_file()), None)
if assets_zip:
    with zipfile.ZipFile(assets_zip) as saved:
        for item in saved.infolist():
            target = (ASSETS / item.filename).resolve()
            if not target.is_relative_to(ASSETS.resolve()):
                raise ValueError(f"Unsafe archive member: {item.filename}")
        saved.extractall(ASSETS)
    print(f"Loaded audit assets from {assets_zip}")
else:
    print("Companion archive absent. Missing public snapshots will be requested from their publishers.")

BACKUP_ROOT = ASSETS / "Databricks_Backup_2026-11-21"
BENCH_ROOT = RUN / "benchmark-18335aa"
BENCH_ROOT.mkdir(exist_ok=True)
PUBLIC = ASSETS / "public-inputs"
PUBLIC.mkdir(exist_ok=True)
RELEASE = ASSETS / "release-artifacts"
RELEASE.mkdir(exist_ok=True)

def download(url, target, *, expected=None, max_bytes=650_000_000):
    target = Path(target)
    if target.is_file() and (not expected or digest(target) == expected):
        return target
    target.parent.mkdir(parents=True, exist_ok=True)
    part = target.with_suffix(target.suffix + ".part")
    try:
        with requests.get(url, stream=True, timeout=(12, 120),
                          headers={"User-Agent": "FacilityFinderResearchReplay/1.0"}) as response:
            response.raise_for_status()
            nbytes = 0
            with part.open("wb") as stream:
                for block in response.iter_content(1 << 20):
                    nbytes += len(block)
                    if nbytes > max_bytes:
                        raise ValueError(f"Download exceeded the limit for {target.name}")
                    stream.write(block)
        if nbytes == 0:
            raise ValueError(f"Empty download for {target.name}")
        if expected and digest(part) != expected:
            fresh = target.with_name("refreshed-" + target.name)
            part.replace(fresh)
            raise ValueError(f"Publisher content changed. Kept separately at {fresh.name}.")
        part.replace(target)
        return target
    finally:
        part.unlink(missing_ok=True)

EXT = {
    "census": ROOT / "load-balancing-pipeline/india-districts-census-2011.csv",
    "state_population": ROOT / "load-balancing-pipeline/india_state_population_2020.csv",
    "boundaries": ROOT / "load-balancing-pipeline/outputs/geoBoundaries-IND-ADM2.geojson",
    "worldpop": PUBLIC / "ind_ppp_2020_UNadj_constrained.tif",
    "caseload_release": RUN / "release-ff7dd5e" / "facilities_newest_capacity.csv",
}
public_assets = {'iffco.html': {'url': 'https://www.iffcotokio.co.in/health-insurance/network-hospital-quality-parameters', 'sha256': 'd3be4ed333858bef644f7bba548d052bf78f05e1fbbe4d31357f2c701279e028'}, 'sbi-2022.pdf': {'url': 'https://content.sbigeneral.in/uploads/f22596609234455e8b24df7859027ce6.pdf', 'sha256': '625fcd136d9367a4c768e8f0604b051d8b62ed5ce0b38e3f72907ce9b0a4adb1'}, 'delhi.pdf': {'url': 'https://dgehs.delhi.gov.in/sites/default/files/inline-files/msnh_27.pdf', 'sha256': '070bdf1bb9c77a5fa2a6a6ff9fa2ddce0143a33cb544b6dd98ab526ec2d28ea8'}, 'kerala.pdf': {'url': 'https://dhs.kerala.gov.in/wp-content/uploads/2026/08/Sanctioned_IP_Beds_DHS_Kerala_July_2026-final.pdf', 'sha256': '76a91fa91cbc7968d7b3c06edc6b0443e3b5acb14fced7d0d92bacd10d724c75'}, 'imaging-dmer.html': {'url': 'https://dmer.maharashtra.gov.in/english/list-of-hospitals-with-ct-mri-facilities/', 'sha256': '955d56ae9122cffcc5360c48d90688a6e2f41fc7353f135ffb4158ec0df6ac58'}, 'imaging-nhm.html': {'url': 'https://nhm.maharashtra.gov.in/en/scheme/iphs-2/', 'sha256': '68ae9fb55cf1741a5c4932289d273c075890193dc4918f2cb04c1f4d934342c0'}, 'puducherry-statistics.html': {'url': 'https://health.py.gov.in/statistics-glance', 'sha256': '8c825d64bbb816e1176ac94304582ef60b47eb474d7cb6fca8ac7170533b5351'}, 'puducherry-hospitals.html': {'url': 'https://health.py.gov.in/hospitals', 'sha256': '3f1094630d2c644a9bb7454599ca599ff78b4f76ea2082df1a0d357f4a93a066'}, 'yanam-health.html': {'url': 'https://yanam.gov.in/departments/health/', 'sha256': 'ee6f2d3b27463697881eb89b8acc9711242fb0779922fd313c282cfe7f796a1d'}, 'iffco.csv': {'url': '', 'sha256': '713cb506d4c57e91e27a313ea3a87bfa0f95ccd62f5f7f316a125fa6a0816280'}, 'sbi-2022.csv': {'url': '', 'sha256': '25a65f1e9a32a3bf5d12bcc36902431110ce86f9cabc8e8ea3e9cd416b8e083d'}}
external_checks = []
for name, info in public_assets.items():
    path = PUBLIC / name
    status = "archived · SHA-256 verified"
    try:
        if not path.is_file():
            if not info["url"]:
                raise FileNotFoundError("Exact historical input is in the companion archive")
            download(info["url"], path, expected=info["sha256"] or None)
            status = "downloaded · SHA-256 verified" if info["sha256"] else "downloaded"
        if info["sha256"] and digest(path) != info["sha256"]:
            raise ValueError("Snapshot checksum differs from the audited file")
        EXT[name] = path
    except (OSError, ValueError, requests.RequestException) as error:
        status = f"unavailable · {error}"
    external_checks.append({"file": name, "status": status, "publisher": info["url"]})

caseload_rel = "max-supplemented-data/capacity_and_confidence/facilities_newest_capacity.csv"
EXT["caseload_release"].parent.mkdir(exist_ok=True)
with EXT["caseload_release"].open("wb") as stream:
    result = subprocess.run(["git", "show", f"ff7dd5edc2d976b907549cb89933305d9a3ab003:{caseload_rel}"], cwd=MAIN_GIT,
                            env=git_environment(), stdout=stream, stderr=subprocess.DEVNULL)
if result.returncode or not EXT["caseload_release"].stat().st_size:
    EXT["caseload_release"].unlink(missing_ok=True)
    print("The caseload input requires history containing ff7dd5e. Use a full Git checkout.")
if (FULL_REPLAY or setting("download_all_data") == "yes") and not EXT["worldpop"].is_file():
    try:
        download("https://data.worldpop.org/GIS/Population/Global_2000_2020_Constrained/2020/BSGM/IND/ind_ppp_2020_UNadj_constrained.tif",
                 EXT["worldpop"])
        with EXT["worldpop"].open("rb") as raster:
            tiff_magic = raster.read(4)
        if tiff_magic not in (b"II*\x00", b"MM\x00*", b"II+\x00", b"MM\x00+"):
            EXT["worldpop"].unlink()
            raise ValueError("WorldPop did not return a TIFF")
    except (OSError, ValueError, requests.RequestException) as error:
        print(f"WorldPop raster unavailable · {error}")
for key in ("census", "state_population", "boundaries", "caseload_release", "worldpop"):
    external_checks.append({"file": key, "status": "ready" if EXT[key].is_file() else "not staged",
                            "publisher": str(EXT[key])})
external_data_manifest = pd.DataFrame(external_checks)
external_data_manifest.to_csv(RUN / "external-data-manifest.csv", index=False)
display(external_data_manifest)

### Download source documents cited by the side studies

The caseload and equity replays ingest the repository’s transcribed observations and decision ledgers, which have site, scope, time period and document links. The manifest below enumerates files and public pages used to prepare those observations. Enable `download_all_data` to retrieve publisher copies into the run directory for independent checking. Publisher versions and response availability can change. The frozen data are kept alongside the download receipt.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import re
from urllib.parse import unquote, urlparse

def linked_documents(node, key=""):
    if isinstance(node, dict):
        for name, value in node.items():
            yield from linked_documents(value, name)
    elif isinstance(node, list):
        for value in node:
            yield from linked_documents(value, key)
    elif isinstance(node, str) and ("url" in key.casefold() or "source" in key.casefold()):
        if node.startswith("https://"):
            yield node.split("#", 1)[0].strip()

side_references = [
    ROOT / "non-ml/caseload/observations.json",
    ROOT / "non-ml/caseload/load_evidence.json",
    *sorted((ROOT / "non-ml/caseload/expansion").glob("*.json")),
]
documents_by_url = {}
for path in side_references:
    for url in linked_documents(load_json(path)):
        documents_by_url.setdefault(url, []).append(path.name)
for path in sorted((ROOT / "data/equity").glob("*.csv")):
    for url in re.findall(r'https://[^\s|"<>]+', path.read_text(encoding="utf-8")):
        parsed_url = url.rstrip(".,);").split("#", 1)[0]
        if parsed_url:
            documents_by_url.setdefault(parsed_url, []).append(path.name)

FETCH_REFERENCE_DOCS = setting("download_all_data") == "yes"

def retrieve_document(entry):
    url, records = entry
    row = {"url": url, "transcripts": ", ".join(sorted(set(records))), "status": "transcribed snapshot ready"}
    if FETCH_REFERENCE_DOCS:
        name = Path(unquote(urlparse(url).path)).name or "index.html"
        name = re.sub(r"[^A-Za-z0-9._-]+", "-", name)[:110]
        token = hashlib.sha256(url.encode()).hexdigest()[:12]
        try:
            path = download(url, RUN / "reference-downloads" / f"{token}-{name}", max_bytes=100_000_000)
            row.update(status="publisher copy downloaded", path=str(path), sha256=digest(path))
        except (OSError, ValueError, requests.RequestException) as error:
            row["status"] = f"publisher copy unavailable · {type(error).__name__}"
    return row

if FETCH_REFERENCE_DOCS:
    with ThreadPoolExecutor(max_workers=4) as pool:
        reference_rows = list(pool.map(retrieve_document, sorted(documents_by_url.items())))
else:
    reference_rows = list(map(retrieve_document, sorted(documents_by_url.items())))
reference_manifest = pd.DataFrame(reference_rows)
reference_manifest.to_csv(RUN / "reference-download-manifest.csv", index=False)
display(reference_manifest.head(10))
if not reference_manifest.empty:
    display(reference_manifest["status"].value_counts().rename("documents"))

## 02 · Collect, profile, clean and review facility evidence

The first experiments asked which records carried useful and attributable service evidence. NQAS contributed 22,015 certification listings, NABH 500 sampled accreditation records, and eRaktKosh 6,524 blood-service records. HFR access stalled. BIS CGHS (294), ESIC (6), Tamil Nadu CMCHIS (2,157), and Kerala SHA (11) added 2,468 government scheme records. Rajasthan RGHS timed out. These counts describe rows from the respective publishers. Registry rows can overlap and scheme eligibility can expire.

**Decision.** Keep publisher, date, status, quote, URL and branch identifiers with every claim. A scheme entry, accreditation scope, blood-storage unit and blood bank answer different service questions. The saved government tables lacked whole-facility bed totals, so they informed identity, contacts and service evidence, while the count study moved to dated insurers and official operator documents.

[Collection action log](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/ACTION_LOG.md) · [Government extraction inventory](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/data/government_sources/README.md)

In [ ]:
def chart_series(series, title, xlabel, *, color="#16766b"):
    ax = series.plot.barh(color=color, figsize=(9, max(2.5, 0.36 * len(series) + 1)))
    ax.set_title(title, loc="left")
    ax.set_xlabel(xlabel)
    ax.grid(axis="x", alpha=0.2)
    plt.tight_layout()
    plt.show()
    plt.close(ax.figure)

def jsonl_count(path, *, header=False):
    with Path(path).open(encoding="utf-8") as stream:
        return sum(bool(line.strip()) for line in stream) - int(header)

sources = {name: jsonl_count(ROOT / f"data/supplemental/{name}.jsonl") for name in ("nqas", "nabh", "eraktkosh")}
gov = {name: jsonl_count(ROOT / f"data/government_sources/{name}.jsonl")
       for name in ("bis_cghs", "esic", "cmchis", "kerala_sha", "rghs")
       if (ROOT / f"data/government_sources/{name}.jsonl").is_file()}
display(pd.Series(sources | gov, name="saved_rows").rename_axis("extract").reset_index())
chart_series(pd.Series(sources | gov).sort_values(), "Saved supplementary publisher extracts", "Rows")
government = read_csv(ROOT / "data/government_sources/government_facilities.csv")
assert len(government) == sum(gov.values()) == 2468
print(f"Combined government rows: {len(government):,} · fields: {list(government.columns)}")

### Extraction fidelity and paused statistical design

The preliminary browser audit drew three records from each directory. Six matched their attributable source. All three eRaktKosh samples remained unresolved when its UI stalled. The larger finite-population design was frozen at 95% with a five-point endpoint margin and paused before completion. Its sampling unit was the saved source row, with a separate phase for independent corroboration. An 80% pre-screen was discussed later and shelved. These proposed tests supplied no accuracy estimate.

The offline builders can replay successful government extracts from saved raw inputs. Refreshing the registries would introduce a new capture date and current source behavior. The exact per-source counts are available either way.

In [ ]:
audit_design = load_json(ROOT / "data/validation_audit/design.json")
plans = pd.DataFrame([{"source": key, "population": row["population_size"], "frozen_draw": row["sample_size"]}
                      for key, row in audit_design["sources"].items()])
display(plans)
spot = load_json(ROOT / "data/external/verification/report.json")
print((ROOT / "data/external/verification/report.txt").read_text(encoding="utf-8").splitlines()[4:7])

scraper_replay = RUN / "scraper-replay"
scraper_replay.mkdir(exist_ok=True)
for name in ("scripts", "data"):
    target = scraper_replay / name
    if not target.exists():
        shutil.copytree(ROOT / name, target)
for script, arguments in [
    ("build_supplemental_tables.py", []),
    ("report_browser_verification.py", []),
    ("scrape_government_central.py", []),
    ("scrape_government_cmchis.py", ["--offline"]),
    ("scrape_government_kerala.py", []),
    ("build_government_sources.py", []),
]:
    run_cmd([sys.executable, "scripts/" + script, *arguments], cwd=scraper_replay, optional=True)
if FULL_REPLAY:
    run_cmd([sys.executable, "scripts/design_validation_audit.py"], cwd=scraper_replay, optional=True)

### Rebuild the original audit checkpoints

Notebooks 01–03 profile original values, identify branches, and freeze the reported bed and doctor label cohorts. Exact identifiers and compatible name/address/PIN link records. Missing values, conflicting coordinates, ambiguous shared points, group scopes and extreme non-bed counts are logged as separate issues. Stable branch groups keep a branch from appearing in both training and test. The original record and its order survive the transformation.

Notebooks 04–05 extract service statements, run approved recovery, verify scope, record supported contacts and assessments, and write derived updates. Recovered claims carry exact excerpts and IDs tied to a hashed source snapshot. In the documented pilot, 2,421 candidate passages narrowed to two approved Faridabad statements, including a 35-bed total and NICU. A fixed query for PIN 121001 moved that branch from excluded to first in all three historical modes. Arti’s group-level 103 beds and shared dialysis footer stayed unresolved for Budge Budge. The separate geography audit held 839 more branches in the combined entity snapshot and reduced Delhi’s eligible blood-bank candidates from 241 to 200.

[Source-grounded recovery and fixed-query result](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/non-ml/SOURCE_GROUNDED_FIELD_RECOVERY.md) · [Location audit](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/non-ml/LOCATION_AUDIT.md)

In [ ]:
MOD_OUT = RUN / "modular"
AUDIT_SNAPSHOT = MOD_OUT / "export" / "2026-11-21"
RULE_ARCHIVE = local_path(setting("capacity_archive")) if setting("capacity_archive") else None
HAVE_PROJECT = all(SRC[key] is not None for key in ("facilities", "pin", "nfhs"))

def replay_repo_notebook(filename, extra_env=None, optional=False):
    filename = Path(filename).with_suffix(".ipynb").name
    archive = ROOT / "non-ml/notebooks" / filename
    path = RUN / "modular-code" / Path(filename).with_suffix(".py").name
    path.parent.mkdir(exist_ok=True)
    blocks = ["".join(cell["source"]) for cell in load_json(archive)["cells"] if cell["cell_type"] == "code"]
    path.write_text("\n\n".join(blocks), encoding="utf-8")
    env = {
        **os.environ, "DATATHON_REPO_ROOT": str(ROOT), "DATATHON_MODE": "export",
        "DATATHON_OUTPUT_DIR": str(MOD_OUT), "MPLBACKEND": "Agg",
        "DATATHON_LOG_COUNT_MODELS": "0", "WANDB_MODE": "offline",
        "DATATHON_RECOVERY_DOCUMENTS": str(ROOT / "data/recovery/public_documents_2026-11-21.json"),
        "DATATHON_RECOVERY_REVIEWS": str(ROOT / "data/recovery/public_reviews_2026-11-21.json"),
    }
    for key, name in (("facilities", "FACILITY"), ("pin", "PIN"), ("nfhs", "NFHS")):
        env[f"DATATHON_{name}_EXPORT"] = str(SRC[key] or "")
    env.update(extra_env or {})
    return run_cmd([sys.executable, path], cwd=RUN, optional=optional, env=env)

def checkpoint(name):
    path = AUDIT_SNAPSHOT / (name + ".jsonl")
    with path.open(encoding="utf-8") as stream:
        columns = json.loads(stream.readline())["columns"]
        return pd.DataFrame([json.loads(line) for line in stream if line.strip()], columns=columns, dtype=object)

phase_ready = False
if HAVE_PROJECT:
    phases = [
        "01_source_eda.ipynb", "02_facility_and_geography.ipynb", "03_count_field_eda.ipynb",
        "09_count_model_comparison.ipynb",
        "04_service_evidence.ipynb", "04a_source_recovery.ipynb", "04b_verify_services.ipynb",
        "04c_facility_quality.ipynb", "05_field_updates.ipynb",
    ]
    for phase in phases:
        print(f"Replaying {phase}")
        if replay_repo_notebook(phase, optional=True).returncode:
            if phase.startswith("09_"):
                print("Continuing the evidence pipeline without count-model overlays.")
                continue
            print("Stopped the dependent checkpoint chain at this failed phase.")
            break
    else:
        phase_ready = True
else:
    print("Supply the three verified organizer paths to rebuild modular checkpoints.")

In [ ]:
if phase_ready:
    facilities = project_frame(SRC["facilities"])
    coverage = facilities.apply(lambda series: series.map(
        lambda v: len(v) > 0 if isinstance(v, (list, dict, tuple)) else
        str(v).strip().casefold() not in {"", "none", "null", "nan", "<na>", "[]", "{}", "not stated", "n/a"}
    ).mean()).sort_values()
    chart_series(coverage.tail(23) * 100, "Original directory field coverage · top 23 fields", "Nonempty percent")
    display(checkpoint("source_counts"))
    display(checkpoint("recovery_summary").head(35))
    review_summary = checkpoint("service_verification")
    if {"status", "service"}.issubset(review_summary):
        chart_series(review_summary["status"].value_counts(), "Service verification outcomes", "Rows")
    del facilities

recovery_crops = {
    "Faridabad count and NICU": "faridabad-capacity.jpg",
    "Arti group capacity with uncertain branch scope": "arti-group-capacity.jpg",
}
for label, filename in recovery_crops.items():
    display(Markdown(f"**{label} · archived browser evidence**"))
    display(Image(filename=str(ROOT / "data/recovery/verification" / filename), width=620))

cache = PUBLIC / "recovery-cache"
if cache.is_dir():
    capture = RUN / "public-recovery-replay"
    capture.mkdir(exist_ok=True)
    run_cmd([sys.executable, ROOT / "scripts/scrape_field_recovery.py", "--cache-dir", cache,
             "--documents", capture / "documents.json", "--audit", capture / "capture-audit.json"], optional=True)

### Extract dated capacity and related fields

IFFCO Tokio 2025 and SBI General 2022 publish whole-facility observations. Delhi 2025 reports approved strength and Kerala 2026 reports sanctioned beds. The matcher keeps these scopes separate. Maharashtra scans describe inventory or specific service commissioning. Puducherry and Yanam supply institutional year and service/telephone claims with incomplete branch linkage. Exclude same-publisher reuse when assessing new evidence and route ambiguous identities to review.

The published audit linked 696 distinct branches and found outside whole-bed candidates for 347 branches missing beds and outside doctor totals for 270 branches missing doctors after the publisher exclusion. The consolidated replay with the later preserved location rules linked 667 branches locally, with 331 bed and 260 doctor candidates under the same publisher exclusion. The rebuild prints its own denominators. The workbook queued 493 facility–field comparisons and 3,265 further linkage leads. Holy Family’s cited Bandra page supported 268 beds. The Raigarh operator and expansion approval left Shri Balaji Metro’s operating count unresolved. The full extractor replay uses Poppler (`pdftotext` and `pdftocairo`). Standard replay reads the companion’s dated structured CSVs for PDF tables and reruns the HTML parsers. When Poppler is present, the independent fidelity replay also checks the original PDFs.

[Source tables, matching rules and adjudications](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/data/recovery/capacity-evidence/README.md)

In [ ]:
count_replay = RUN / "public-counts"
count_replay.mkdir(exist_ok=True)
scripts = ROOT / "scripts"
count_commands = [
    ("extract_iffco_capacity.py", ["iffco.html"], [PUBLIC / "iffco.html", count_replay / "iffco.csv"], "iffco.csv"),
    ("extract_sbi_capacity.py", ["sbi-2022.pdf"], [PUBLIC / "sbi-2022.pdf", "--edition", "2022", "--output",
                                                 count_replay / "sbi-2022.csv"], "sbi-2022.csv"),
    ("extract_delhi_capacity.py", ["delhi.pdf"], ["--pdf", PUBLIC / "delhi.pdf", "--output",
                                                count_replay / "delhi.csv"], "delhi.csv"),
    ("scrape_capacity_kerala.py", ["kerala.pdf"], ["--pdf", PUBLIC / "kerala.pdf", "--output",
                                                count_replay / "kerala.csv"], "kerala.csv"),
    ("extract_iffco_details.py", ["iffco.html"], [PUBLIC / "iffco.html", count_replay / "iffco-details.csv"], "iffco-details.csv"),
    ("extract_imaging_evidence.py", ["imaging-dmer.html", "imaging-nhm.html"],
     ["--dmer-html", PUBLIC / "imaging-dmer.html", "--nhm-html", PUBLIC / "imaging-nhm.html",
      "--output", count_replay / "imaging.csv"], "imaging.csv"),
    ("extract_establishment_evidence.py", ["puducherry-statistics.html", "puducherry-hospitals.html"],
     ["--statistics", PUBLIC / "puducherry-statistics.html", "--hospital-directory", PUBLIC / "puducherry-hospitals.html",
      "--output", count_replay / "years.csv"], "years.csv"),
    ("extract_yanam_evidence.py", ["yanam-health.html"], ["--health-html", PUBLIC / "yanam-health.html",
     "--years", count_replay / "years.csv", "--output", count_replay / "yanam.csv"], "yanam.csv"),
]
extract_rows = []
for script, raw_files, args, output in count_commands:
    out = count_replay / output
    out.unlink(missing_ok=True)
    installed = all(shutil.which(program) for program in ("pdftotext", "pdftocairo"))
    needs_poppler = script in {"extract_sbi_capacity.py", "extract_delhi_capacity.py", "scrape_capacity_kerala.py"}
    outcome = "missing"
    can_run_parser = not needs_poppler or (installed and (FULL_REPLAY or not (PUBLIC / "parsed" / output).is_file()))
    if all(name in EXT for name in raw_files) and can_run_parser:
        result = run_cmd([sys.executable, scripts / script, *args], optional=True)
        if result.returncode == 0 and out.is_file():
            outcome = "regenerated"
        else:
            out.unlink(missing_ok=True)
    if outcome == "missing" and (PUBLIC / "parsed" / output).is_file():
        shutil.copyfile(PUBLIC / "parsed" / output, out)
        outcome = "frozen extracted CSV"
    extract_rows.append({"extract": output, "result": outcome,
                         "rows": len(read_csv(out)) if out.is_file() else None})
display(pd.DataFrame(extract_rows))

In [ ]:
def append_sources(paths):
    return [part for path in paths for part in ("--source", path)]

matched = RUN / "count-matching"
field_matches = RUN / "field-comparison"
counts_ready = False
count_sources = [count_replay / name for name in ("iffco.csv", "sbi-2022.csv", "delhi.csv", "kerala.csv")]
field_sources = [count_replay / name for name in ("iffco-details.csv", "imaging.csv", "years.csv", "yanam.csv")]
if phase_ready and all(path.is_file() for path in count_sources + field_sources):
    primary = ["--entities", AUDIT_SNAPSHOT / "entities.jsonl", "--targets", AUDIT_SNAPSHOT / "capacity_target_audit.jsonl",
               "--location-records", AUDIT_SNAPSHOT / "location_records.jsonl"]
    count_match = run_cmd([sys.executable, scripts / "match_external_capacity.py", *primary,
                           "--doctor-targets", AUDIT_SNAPSHOT / "number_doctors_target_audit.jsonl",
                           "--output-dir", matched, *append_sources(count_sources)], optional=True)
    if count_match.returncode == 0:
        field_match = run_cmd([sys.executable, scripts / "compare_external_facility_fields.py", *primary,
                              "--raw", AUDIT_SNAPSHOT / "raw_facilities.jsonl", "--count-links",
                              matched / "matched_claims.csv", "--output-dir", field_matches,
                              *append_sources(field_sources)], optional=True)
        counts_ready = field_match.returncode == 0

historical_count_match = pd.Series({"Branches linked to an external count": 696, "Missing beds with outside total": 347,
                                    "Missing doctors with outside total": 270, "Pending field comparisons": 493})
chart_series(historical_count_match.sort_values(), "External evidence audit · published results", "Branches or comparisons")
if counts_ready:
    display(load_json(matched / "summary.json"))
    display(pd.DataFrame(load_json(field_matches / "field_summary.json")).fillna(0))

### Separate fidelity, source selection and human review

The insurer fidelity audit compared emitted positives (precision) and independently enumerated printed positives (recall), six checks in each publisher–field–direction group. All 48 random checks matched. Every six-check group has a 95% exact interval of about 54.1%–100%. A separate targeted check found IFFCO’s `04` doctors dropped by the parser. Accepting leading zeros restored one cell and left the exact FDR branch-join output unchanged.

The 36-case pilot and its 48-case extension froze disagreements by a deterministic hash, field and insurer stratum. The companion contains those exact sample manifests so the held-out selector comparison can use the original review IDs when current matching holds change the new sample. Five then seven bed cases had a supported exact total. The doctors sample resolved zero exact totals. XGBoost selects a source claim using publisher, date, age and agreement. Leave-one-facility-out scores compared it with a publisher baseline and newest-report baseline. Both versions tied the pooled-publisher rule on top-choice hits and failed the minimum eight-supported-case gate. The baseline remained in use and the model output was retained for study.

The next In The Loop desk review covered ten branches and 20 claims. It recorded two accepted, two rejected and 16 inconclusive claims, with two exact operator counts. A broader independent audit froze 130 of 195 branches at approximately ±5 percentage points and is pending. The link-only tool checks what a publisher printed. Independent branch evidence is required for a disputed total. A later targeted nine-review round directly supported three values, and the cleaner withheld the six unresolved decisions.

[Supplementation history](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/max-supplemented-data/SUPPLEMENTATION_AND_VERIFICATION_HISTORY.md) · [Fidelity check](https://github.com/augustchu/datathon-F26/tree/af8965890e4de2e5a5f2a24177425e5fdb77352d/max-supplemented-data/experiments/verification/scraper-audit)

In [ ]:
supplement = ROOT / "max-supplemented-data"
selector_folder = supplement / "experiments/source_selection"
verify_folder = supplement / "experiments/verification"
fidelity_folder = verify_folder / "scraper-audit"
fidelity_results = load_json(fidelity_folder / "results.json")
display(fidelity_results.get("metrics", fidelity_results.get("results", fidelity_results)))
if shutil.which("pdftotext") and all((PUBLIC / name).is_file()
    for name in ("iffco.html", "sbi-2022.pdf", "iffco.csv", "sbi-2022.csv")):
    run_cmd([sys.executable, fidelity_folder / "sample_audit.py", PUBLIC, count_replay / "fidelity-sample.json",
             "--observed-values", fidelity_folder / "source-observations.json", "--audit-report",
             count_replay / "fidelity-results.json"], optional=True)

archived_selectors = load_json(selector_folder / "results.json")
selector_rows = []
for epoch, evidence in archived_selectors["checkpoint_evidence"].items():
    for method, score in evidence["original_grouped_oof"].items():
        selector_rows.append({"pilot": epoch, "method": method, "reviewed": score["reviewed_cases"],
                              "hits": score["hits"], "brier": score["mean_candidate_brier"]})
selector_scores = pd.DataFrame(selector_rows)
display(selector_scores)
selector_scores.pivot(index="pilot", columns="method", values="hits").plot.bar(
    title="Held-out top-choice hits · frozen source-selector reviews", rot=0, ylabel="Hits")
plt.tight_layout()
plt.show()
display(Image(filename=str(verify_folder / "screenshots/in-the-loop-desk-review.png"), width=740))

In [ ]:
SUPPLEMENT_RUN = RUN / "supplementation"
CAPACITY_RULE_BUILD = {}
description_archive = RUN / "historical-description-capacity.json"
if counts_ready and (ASSETS / "historical-code/extract_capacity.py").is_file():
    import importlib.util
    sys.path[:0] = [str(ROOT / "non-ml/src"), str(ROOT / "ML/models with population density")]
    spec = importlib.util.spec_from_file_location("historical_description", ASSETS / "historical-code/extract_capacity.py")
    historical_description = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(historical_description)
    layout = RUN / "description-checkpoints"
    layout.mkdir(exist_ok=True)
    for name in ("full_audit", "location-audit-6y22xg3u"):
        location = layout / name / "export" / "2026-11-21"
        location.parent.mkdir(parents=True, exist_ok=True)
        if not location.exists():
            try:
                location.symlink_to(AUDIT_SNAPSHOT, target_is_directory=True)
            except OSError:
                location.mkdir()
                for stem in ("linked_fdr", "capacity_target_audit", "number_doctors_target_audit",
                             "location_records", "location_entity_review"):
                    shutil.copyfile(AUDIT_SNAPSHOT / f"{stem}.jsonl", location / f"{stem}.jsonl")
    description_result = historical_description.run(ROOT, layout, "2026-11-21")
    description_archive.write_text(json.dumps(description_result, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    display(pd.Series(description_result["summary"]).drop(["by_split", "status_counts", "status_counts_by_original_capacity"]))
    CAPACITY_RULE_BUILD = {
        "snapshot": AUDIT_SNAPSHOT, "count-links": matched / "matched_claims.csv",
        "field-links": field_matches / "field_matches.csv", "unlinked-fields": field_matches / "source_only_fields.csv",
        "approved-updates": AUDIT_SNAPSHOT / "field_updates.jsonl",
        "approved-services": AUDIT_SNAPSHOT / "recovered_service_claims.jsonl",
        "description-capacity": description_archive,
        "previous-reviews": ROOT / "data/recovery/capacity-evidence/reviews.json",
    }
else:
    print("Supplementation requires audit checkpoints, parsed public documents and the historical parser in the companion archive.")

def flags(mapping):
    return [part for key, value in mapping.items() for part in ("--" + key, value)]

supplement_ready = []
for name, extend in (("iteration-01", False), ("iteration-02", True)):
    dest = SUPPLEMENT_RUN / name
    dest.mkdir(parents=True, exist_ok=True)
    review = supplement / ("review" if extend else "review/initial")
    frozen_sample = ASSETS / "source-selector-samples" / (name + ".json")
    sample = dest / "sample.json"
    regenerated = dest / "resampled.json"
    if counts_ready:
        run_cmd([sys.executable, supplement / "src/sample_manual_checks.py",
                 "--counts", matched / "matched_claims.csv", "--other-review", field_matches / "field_review.csv",
                 "--output", regenerated, *(["--extend"] if extend else [])], optional=True)
    if frozen_sample.is_file():
        shutil.copyfile(frozen_sample, sample)
        print(f"{name} · replaying the frozen review panel")
        if regenerated.is_file():
            print("Regenerated sample matches the original:", load_json(regenerated) == load_json(sample))
    elif regenerated.is_file():
        shutil.copyfile(regenerated, sample)
    else:
        print(f"{name} requires the frozen sample or rebuilt count links.")
        continue
    selector_fit = run_cmd([sys.executable, supplement / "src/model_sources.py", "--sample", sample,
                           "--reviews", review, "--output-dir", dest / "model"], optional=True)
    if not CAPACITY_RULE_BUILD or selector_fit.returncode:
        continue
    commands = [
        [supplement / "src/build_dataset.py", *flags(CAPACITY_RULE_BUILD), "--sample", sample, "--reviews", review,
         "--model-dir", dest / "model", "--output-dir", dest / "dataset"],
        [supplement / "src/validate_run.py", "--iteration", dest, "--raw", AUDIT_SNAPSHOT / "raw_facilities.jsonl"],
    ]
    for args in commands:
        if run_cmd([sys.executable, *args], optional=True).returncode:
            print(f"{name} dataset validation requires the corresponding source and identity checkpoint.")
            break
    else:
        supplement_ready.append(name)

if len(supplement_ready) == 2:
    run_cmd([sys.executable, selector_folder / "evaluate_selectors.py", "--local", SUPPLEMENT_RUN,
             "--output", SUPPLEMENT_RUN / "selector-replay.json"], optional=True)
    run_cmd([sys.executable, verify_folder / "build_human_worklist.py", "--local", SUPPLEMENT_RUN,
             "--output", SUPPLEMENT_RUN / "human-worklist"], optional=True)

## 03 · Capacity experiments before the release

The project described which facility records contained count data, measured how well models recovered withheld reported counts, and released estimates for branches eligible for imputation. Each task uses its own denominator. In particular, the 522-facility comparison, 504-row density control, 467-branch forest study, 456-branch two-selector study and 455-branch source-rule study use different cohorts. Compare scores within each cohort.

The initial `dataCleaning` file is a one-byte placeholder. The executable cleaning moved into `non-ml` and the later evidence and release pipeline. [Action log](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/ACTION_LOG.md) · [Early model evaluation](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/ML/MODEL_EVALUATION.md)

In the modelling cells, MAE is the average absolute miss in beds, RMSE emphasizes large misses, and signed error is prediction minus observed. Compare against the training median and inspect the tail alongside MAE. A reported-count holdout measures performance on reporting facilities. Accuracy for the genuinely missing target population remains unmeasured.

In [ ]:
import json
import os
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

ml_path = ROOT / "ML"
density_path = ml_path / "models with population density"
capacity_path = ROOT / "max-supplemented-data/experiments/capacity"
sys.path[:0] = [str(ROOT / "non-ml/src"), str(ml_path / "notebooks"),
               str(ml_path / "capacity_EDA"), str(density_path), str(capacity_path)]
from facility_tool.workspace import _read_export

ml_run = RUN / "capacity"
ml_run.mkdir(exist_ok=True, parents=True)
for name in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ.setdefault(name, "2")

def ml_input(key):
    value = SRC.get(key)
    if value and Path(value).is_file():
        return _read_export(Path(value))
    catalog = SRC.get(f"{key}_catalog")
    if key == "facilities":
        catalog = catalog or SRC.get("facility_catalog")
    if catalog:
        table = spark.table(catalog)
        records = table.collect()
        return pd.DataFrame([row.asDict(recursive=True) for row in records], columns=table.columns)
    raise FileNotFoundError(f"Set the verified project input path for {key}.")

have_ml_project = bool(globals().get("HAVE_PROJECT", True))
if have_ml_project:
    facility_ml = ml_input("facilities")
    if "countries" not in facility_ml or "capacity" not in facility_ml:
        raise ValueError("Use the original ordered release schema for the modelling experiments.")
    pin_ml, nfhs_ml = ml_input("pin"), ml_input("nfhs")
else:
    print("Showing archived model evidence. Set the three organizer input paths to enable raw-data replays.")

def ml_bar(frame, label, value, title, xlabel, *, color="#417a83"):
    figure, axis = plt.subplots(figsize=(9, max(2.8, .38 * len(frame) + 1.1)))
    frame.plot.barh(x=label, y=value, ax=axis, color=color, legend=False)
    axis.invert_yaxis()
    axis.set(title=title, ylabel="", xlabel=xlabel)
    figure.tight_layout()
    plt.show()
    plt.close(figure)

def ml_picture(relative):
    image_path = capacity_path / relative
    if image_path.is_file():
        display(Image(filename=str(image_path)))
    else:
        print("Missing archived figure", relative)

def original_cells(notebook, indexes, namespace):
    document = json.loads(notebook.read_text(encoding="utf-8"))
    for index in indexes:
        cell = document["cells"][index]
        if cell["cell_type"] != "code":
            raise ValueError(f"{notebook.name} cell {index} is not executable.")
        source = cell["source"]
        source = "".join(source) if isinstance(source, list) else source
        exec(compile(source, f"{notebook.name}[{index}]", "exec"), namespace)
    return namespace

### Initial random forest and source-schema boundary

The exploratory request changed from predictors across the whole record to columns strictly after `countries`. The 300-tree forest fits encoders and text vocabularies on training rows, groups duplicate predictors, and compares against mean and median baselines. The archived notebook leaves the optional facility-group column unset. Change that setting for an additional facility-ID sensitivity analysis. The leakage report flags descriptions and numeric fields that can disclose the target. A strong score here can measure recovery of a number already stated. [Random forest design and tests](https://github.com/augustchu/datathon-F26/tree/af8965890e4de2e5a5f2a24177425e5fdb77352d/ML/capacity_EDA)

In [ ]:
if have_ml_project:
    from capacity_rf import run_capacity_rf

    early_rf = run_capacity_rf(facility_ml, group_column=None, n_jobs=2)
    display(early_rf["metrics"])
    display(early_rf["target_audit"]["status"].value_counts().rename("rows").to_frame())
    display(early_rf["leakage_flags"])
    display(early_rf["column_selection"].groupby("role").size().rename("columns").to_frame())
    ml_bar(early_rf["importance"].head(12), "column", "mae_increase_mean",
           "Random forest permutation importance", "Increase in MAE when shuffled (beds)")
    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    p = early_rf["predictions"]
    axes[0].scatter(p["actual_capacity"], p["predicted_capacity"], s=10, alpha=.4)
    axes[0].set(xscale="symlog", yscale="symlog", xlabel="Reported beds", ylabel="Predicted beds")
    axes[1].hist(p["residual"], bins=45, color="#417a83")
    axes[1].set(xlabel="Reported minus predicted beds", ylabel="Facilities")
    figure.tight_layout()
    plt.show()
    plt.close(figure)
else:
    print("Stage the facility file to rerun the original forest.")

### Legacy neural models and NFHS-5 enrichment

The original PyTorch model uses a 128–64 MLP, dropout, Huber loss on `log1p(beds)`, and an 80/10/10 row split. It compares with histogram gradient boosting on the same inputs. The facility-only and NFHS-enriched paths run independently below. The input cells adapt the saved notebooks to file-backed DataFrames. The historical notebooks wrote a workspace table and logged tracking artifacts. These runs keep the enrichment in memory and use the installed compute's CPU or CUDA. [Facility and enriched training notebooks](https://github.com/augustchu/datathon-F26/tree/af8965890e4de2e5a5f2a24177425e5fdb77352d/ML/notebooks)

The standalone enrichment screened 18 candidate NFHS-5 indicators for at least 80% coverage. Greedy pruning removed indicators with absolute Spearman correlation above 0.7 to a selected representative. The legacy join picked the most frequent district/state pair per PIN and left-joined 15 indicators, retaining unmatched facilities. The later audited join holds ambiguous PIN matches and considers the facility's reported state. These are district attributes and their missingness can vary geographically. Coverage and added predictive value are separate questions.

In [ ]:
if have_ml_project:
    class NotebookInputTable:
        def __init__(self, frame):
            self.frame = frame

        def select(self, *columns):
            return NotebookInputTable(self.frame.loc[:, list(columns)])

        def toPandas(self):
            return self.frame.copy()

    class NotebookInputCatalog:
        def __init__(self, tables):
            self.tables = tables
            self.catalog = self

        def table(self, name):
            return NotebookInputTable(self.tables[name.rsplit(".", 1)[-1]])

        def tableExists(self, name):
            return name.rsplit(".", 1)[-1] in self.tables

    ml_catalog_sources = {
        "facilities": facility_ml,
        "india_post_pincode_directory": pin_ml.assign(pincode=pd.to_numeric(pin_ml["pincode"], errors="coerce")),
        "nfhs_5_district_health_indicators": nfhs_ml,
    }
    ml_legacy_tables = NotebookInputCatalog(ml_catalog_sources)

    import torch
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    torch.set_num_threads(2)

    # The source notebooks use the same sys.path import; preload their trainer
    # since a Databricks Workspace notebook is not necessarily in a Git folder.
    import gpu_capacity_models

In [ ]:
if have_ml_project:
    # Run the original feature selection and legacy join, omitting the catalog table write.
    nfhs_legacy = original_cells(ml_path / "notebooks/NFHS-5 Socioeconomic Enrichment.ipynb",
                                 (1, 2, 3), {"spark": ml_legacy_tables, "display": display})
    nfhs_enriched = nfhs_legacy["enriched"]
    selected_nfhs = nfhs_legacy["representatives"]
    display(pd.DataFrame({"indicator": selected_nfhs,
                          "district_coverage": nfhs_legacy["nfhs_num"][selected_nfhs].notna().mean().to_numpy()}))
    quiet_corr = nfhs_legacy["corr"].loc[selected_nfhs, selected_nfhs]
    figure, axis = plt.subplots(figsize=(7, 6))
    image = axis.imshow(quiet_corr, vmin=0, vmax=1, cmap="viridis")
    axis.set(title="Absolute Spearman correlation after pruning",
             xticks=range(len(selected_nfhs)), yticks=range(len(selected_nfhs)))
    axis.set_xticklabels(range(1, len(selected_nfhs) + 1))
    axis.set_yticklabels(range(1, len(selected_nfhs) + 1))
    figure.colorbar(image, ax=axis)
    figure.tight_layout()
    plt.show()
    plt.close(figure)
else:
    print("Stage the facility, PIN and NFHS inputs to rerun enrichment.")

In [ ]:
def legacy_gpu_scope():
    return {"spark": ml_legacy_tables, "display": display, "plt": plt,
            "np": np, "pd": pd, "torch": torch}

In [ ]:
if have_ml_project:
    # Facility-only GPU notebook, run its original data, EDA, training and scoring logic.
    base_legacy = original_cells(ml_path / "notebooks/Facility Capacity GPU Training.ipynb",
                                  range(2, 10), legacy_gpu_scope())
else:
    print("Stage the facility file to rerun the original MLP and histogram booster.")

In [ ]:
if have_ml_project:
    # Enriched GPU notebook, using the in-memory legacy NFHS result and skipping catalog writes.
    enriched_scope = legacy_gpu_scope()
    enriched_scope.update({key: nfhs_legacy[key] for key in ("enriched", "to_na")})
    enriched_legacy = original_cells(ml_path / "notebooks/Enriched Capacity GPU Training.ipynb",
                                      (17, 18, 19, 20, 21, 22, 23, 24), enriched_scope)

    legacy_report = pd.DataFrame([
        {"input": input_set, "estimator": model, **scope[key]}
        for input_set, scope in (("Facilities", base_legacy), ("Facilities + NFHS", enriched_legacy))
        for model, key in (("Histogram booster", "baseline_metrics"), ("MLP", "mlp_metrics"))
    ])
    legacy_report["run"] = legacy_report["input"] + " · " + legacy_report["estimator"]
    display(legacy_report.drop(columns="run"))
    ml_bar(legacy_report, "run", "mae", "Legacy random-row holdout", "MAE (beds)")
else:
    print("The enrichment replay uses the three organizer files.")

### Interpretable baselines

The interpretable notebook tried log-ridge and a depth-four decision tree on the same historical 80/10/10 scheme. It reports ridge coefficients, a tree diagram and raw-feature permutation importance. Coefficients and splits describe associations in the model. The pinned file has two versions of this analysis, both carrying the same estimator families, so the cell runs its maintained first section with the in-memory NFHS join. [Interpretable Models notebook](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/ML/notebooks/Interpretable%20Models.ipynb)

In [ ]:
if have_ml_project:
    from datetime import datetime, timezone

    interpretable = original_cells(
        ml_path / "notebooks/Interpretable Models.ipynb", (3, 4, 5, 6, 7, 8),
        {"spark": NotebookInputCatalog({**ml_catalog_sources, "facilities_enriched": nfhs_enriched}),
         "display": display, "datetime": datetime, "timezone": timezone},
    )
    display(pd.DataFrame([{"model": name, **interpretable[key]} for name, key in
                          (("Log ridge", "ridge_metrics"), ("Shallow tree", "tree_metrics"))]))
else:
    print("The interpretable graphics use the organizer files.")

### MLflow, W&B and the Databricks backup

The November 21 workspace backup compared 383 objects against GitHub at `9168412`. It found 381 matches and preserved the two Jupyter and DBC notebook exports with different workspace settings. The saved Jupyter copies have no embedded outputs. The inventory found no accessible MLflow logged models, no user-registered Unity Catalog model, and no `workspace.default.facilities_enriched` table. The GitHub tree separately includes model weights and preprocessors under `ML/notebooks/outputs_models/models`. The Git files and the workspace model registry are distinct inventories.

Later notebook revisions logged JSON metric tables, fitted preprocessing, exact evaluated weights and training curves through W&B in offline mode by default. The artifact convention records loss on the log scale separately from evaluation error in beds. The notebook cells here rerun fits without logging them to a remote account. The screenshot near the blend experiment preserves the historical W&B artifact view. [Tracking guide](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/ML/README.md)

In [ ]:
backup_value = globals().get("BACKUP_ROOT")
if backup_value and Path(backup_value).is_dir():
    backup = Path(backup_value)
    for exported in sorted((backup / "workspace/objects").rglob("*.ipynb")):
        saved = json.loads(exported.read_text(encoding="utf-8"))
        print(exported.name, "cells:", len(saved["cells"]),
              "saved outputs:", sum(len(cell.get("outputs", [])) for cell in saved["cells"]))
    model_inventory = backup / "models/summary.json"
    if model_inventory.is_file():
        inventory = load_json(model_inventory)
        print({key: value for key, value in inventory.items() if isinstance(value, int)})
else:
    print("Optional workspace backup not staged. These replays use the public GitHub sources.")

### Shared facility-held model comparison

Reported capacities were withheld at the facility level. The historical split had 1,498 training facilities, 489 for calibration and 522 for test. The updated saved GitHub models were scored using their own preprocessors. The original random-row training IDs are unavailable. The preprocessing row count implies possible overlap with the shared test. Treat the saved-model scores as descriptive. Comparable refits put preprocessing and weights inside training and used calibration for early stopping. On this split the facilities-only MLP averaged 121.9 beds MAE across three seeds and beat the enriched MLP each time. Its large-hospital estimates were still several hundred beds low. The result selected a candidate for validation.

The executable comparison below builds independent refits from the shared audit checkpoint and can score the pinned saved weights separately. The chart includes independent refits only. The readme records the original and updated GitHub artifact scores. [Comparison design, scores and commit links](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/ML/MODEL_EVALUATION.md#shared-test)

In [ ]:
def ml_snapshot():
    if not have_ml_project:
        return None
    for value in (globals().get("AUDIT_SNAPSHOT"), globals().get("SNAPSHOT"),
                  globals().get("AUDIT_STORE"), globals().get("store")):
        path = getattr(value, "path", value)
        if path and (Path(path) / "capacity_target_audit.jsonl").is_file():
            return Path(path)
    return None

def ml_checkpoint(name):
    snapshot = ml_snapshot()
    if snapshot is None:
        raise FileNotFoundError("Run the facility and count audit notebooks before this experiment.")
    with (snapshot / f"{name}.jsonl").open(encoding="utf-8") as stream:
        columns = json.loads(next(stream))["columns"]
        return pd.DataFrame((json.loads(line) for line in stream if line.strip()),
                            columns=columns, dtype=object)

ml_shared_archived = pd.DataFrame([
    ("Training median", 150.6, 853.3), ("Log ridge", 141.9, 834.0),
    ("Log random forest", 141.4, 817.5),
    ("Facility MLP · seed 42", 125.06, 608.74), ("Facility MLP · seed 104", 120.27, 575.74),
    ("Facility MLP · seed 2026", 120.33, 602.43),
    ("NFHS MLP · seed 42", 146.24, 678.83), ("NFHS MLP · seed 104", 128.26, 673.61),
    ("NFHS MLP · seed 2026", 128.63, 660.61),
], columns=["model", "mae", "mae_500_plus"])

In [ ]:
import joblib.numpy_pickle as jp
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
NULL_STRINGS = {'null', 'none', 'nan', 'n/a', 'na', '', '-', 'unknown'}
TORCH_SEEDS = (42, 104, 2026)
ALLOWED_GLOBALS = {
    ('joblib.numpy_pickle', 'NumpyArrayWrapper'),
    ('sklearn.compose._column_transformer', 'ColumnTransformer'),
    ('sklearn.compose._column_transformer', '_RemainderColsList'),
    ('sklearn.pipeline', 'Pipeline'),
    ('sklearn.impute._base', 'SimpleImputer'),
    ('sklearn.preprocessing._data', 'StandardScaler'),
    ('sklearn.preprocessing._encoders', 'OneHotEncoder'),
    ('numpy', 'ndarray'),
    ('numpy', 'dtype'),
    ('numpy', 'float64'),
    ('numpy', 'float32'),
    ('numpy', 'int64'),
    ('numpy._core.multiarray', '_reconstruct'),
    ('numpy._core.multiarray', 'scalar'),
    ('numpy.core.multiarray', '_reconstruct'),
    ('numpy.core.multiarray', 'scalar'),
    ('pandas._libs.missing', 'NA'),
    ('builtins', 'slice'),
}


class LocalModelUnpickler(jp.NumpyUnpickler):
    def find_class(self, module, name):
        if (module, name) not in ALLOWED_GLOBALS:
            raise ValueError(f'Unexpected serialized global {module}.{name}')
        return super().find_class(module, name)


def read_joblib(path):
    with path.open('rb') as source:
        return LocalModelUnpickler(str(path), source, ensure_native_byte_order=True).load()


def clean(series):
    series = series.astype('string').str.strip().str.lower()
    return series.mask(series.isin(NULL_STRINGS))


def normalize_label(series):
    return (series.astype('string').str.strip().str.lower()
            .str.replace(r'\s+', ' ', regex=True).str.replace(r'[^a-z ]', '', regex=True).str.strip())


def nfhs_join(facilities, indicators):
    pins = pin_ml.copy()
    pins = pins.assign(
        pincode=pd.to_numeric(pins['pincode'], errors='coerce'),
        d=normalize_label(pins['district']), st=normalize_label(pins['statename']),
    )
    counts = pins.groupby(['pincode', 'd', 'st']).size().rename('n').reset_index()
    pin_lookup = counts.sort_values('n', ascending=False).drop_duplicates('pincode').set_index('pincode')
    nfhs = nfhs_ml.copy()
    nfhs = nfhs.assign(d=normalize_label(nfhs['district_name']), st=normalize_label(nfhs['state_ut']))
    for feature in indicators:
        text = nfhs[feature].astype('string')
        nfhs[feature] = pd.to_numeric(text.where(text.str.fullmatch(r'\s*-?\d+(\.\d+)?\s*')), errors='coerce')
    zips = pd.to_numeric(clean(facilities['address_zipOrPostcode']), errors='coerce')
    enriched = facilities.assign(d=zips.map(pin_lookup['d']), st=zips.map(pin_lookup['st'])).merge(
        nfhs[['d', 'st', *indicators]].assign(nfhs_matched=True),
        how='left', on=['d', 'st'], validate='m:1', sort=False,
    )
    assert len(enriched) == len(facilities)
    return enriched


def make_features(frame, metadata):
    inputs = pd.DataFrame(index=frame.index)
    for col in metadata['numeric_features']:
        if col.endswith('_len') and col[:-4] in frame:
            inputs[col] = clean(frame[col[:-4]]).str.len().fillna(0).astype(float)
        elif col.endswith('_present') and col[:-8] in frame:
            inputs[col] = clean(frame[col[:-8]]).notna().astype(float)
        else:
            inputs[col] = pd.to_numeric(clean(frame[col]), errors='coerce').astype(float)
    for col in metadata['categorical_features']:
        values = clean(frame[col])
        inputs[col] = values.astype(object).where(values.notna(), np.nan)
    return inputs


def net(n_features):
    return nn.Sequential(
        nn.Linear(n_features, 128), nn.ReLU(), nn.Dropout(.2),
        nn.Linear(128, 64), nn.ReLU(), nn.Dropout(.2),
        nn.Linear(64, 1),
    )


def matrix(preprocessor, features):
    values = np.asarray(preprocessor.transform(features), dtype=np.float32)
    assert np.isfinite(values).all()
    return torch.from_numpy(values)


def to_count(model, values):
    model.eval()
    with torch.inference_mode():
        point = np.expm1(model(values).squeeze(1).numpy()).astype(float)
    assert np.isfinite(point).all()
    return np.maximum(0, np.rint(point)), float(np.min(point))


def refit(preprocessor, features_train, features_validation, y_train, y_validation, seed):
    torch.manual_seed(seed)
    processor = clone(preprocessor).fit(features_train)
    train = matrix(processor, features_train)
    validation = matrix(processor, features_validation)
    train_y = torch.tensor(np.log1p(y_train), dtype=torch.float32)[:, None]
    validation_y = torch.tensor(np.log1p(y_validation), dtype=torch.float32)[:, None]
    model = net(train.shape[1])
    loader = DataLoader(TensorDataset(train, train_y), batch_size=256, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=.5, patience=10)
    criterion = nn.HuberLoss(delta=1)
    best, bad_epochs = float('inf'), 0
    weights = None
    for epoch in range(300):
        model.train()
        for x, y in loader:
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.inference_mode():
            loss = float(criterion(model(validation), validation_y))
        scheduler.step(loss)
        if loss < best - 1e-4:
            best, bad_epochs = loss, 0
            weights = {name: value.clone() for name, value in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= 30:
                break
    model.load_state_dict(weights)
    return processor, model, epoch + 1

In [ ]:
from sklearn.base import clone
from facility_tool.count_models import _audit_rows, _fit, _point, _select_features, _transformer

def replay_shared_mlp():
    previous_threads = torch.get_num_threads()
    torch.set_num_threads(1)
    audited_rows, candidates, _ = _audit_rows(
        "capacity", ml_checkpoint("capacity_target_audit"), ml_checkpoint("number_doctors_target_audit"),
        ml_checkpoint("raw_facilities").columns)
    groups = {name: audited_rows.loc[audited_rows["split"].eq(name) & audited_rows["value"].notna()]
              for name in ("train", "calibration", "test")}
    train, validation, test = groups.values()
    if any(len(split) < 2 for split in groups.values()):
        raise ValueError("This comparison needs capacity labels in train, calibration and test.")
    holdout = pd.concat(groups.values(), ignore_index=True)[["facility_id", "split", "value"]]
    linked = ml_checkpoint("linked_fdr")
    linked["_numeric_beds"] = pd.to_numeric(clean(linked["capacity"]), errors="coerce")
    recorded = linked.merge(holdout, on="facility_id", validate="m:1")
    inputs = recorded.loc[recorded["_numeric_beds"].eq(recorded["value"])].drop_duplicates("facility_id")
    inputs = inputs.set_index("facility_id").loc[holdout["facility_id"]].reset_index()
    models_root = ml_path / "notebooks/outputs_models/models"
    metadata = {variant: read_joblib(models_root / variant / "model_meta.joblib")
                for variant in ("base", "enriched")}
    indicators = [field for field in metadata["enriched"]["numeric_features"]
                  if field not in metadata["base"]["numeric_features"]]
    inputs = nfhs_join(inputs, indicators)
    fitted_features, *_ = _select_features("capacity", train, candidates, 42)
    transform = _transformer(fitted_features).fit(train)
    actual = test["value"].to_numpy(dtype=float)
    predicted = {"Training median": np.full_like(actual, np.rint(train["value"].median()))}
    for kind, model in _fit(transform, train, 42).items():
        predicted[{"regression": "Log ridge", "ml": "Log random forest"}[kind]] = _point(model, transform.transform(test))
    artifact = {}
    for variant, meta in metadata.items():
        templates = read_joblib(models_root / variant / "preprocessor.joblib")
        features = make_features(inputs, meta)
        x_train, x_cal, x_test = (features.loc[inputs["split"].eq(split)] for split in groups)
        saved_model = net(meta["n_features"])
        saved_model.load_state_dict(torch.load(models_root / variant / "mlp_state_dict.pt",
                                              map_location="cpu", weights_only=True))
        artifact[variant], _ = to_count(saved_model, matrix(templates, x_test))
        for seed in TORCH_SEEDS:
            prep, model, _ = refit(templates, x_train, x_cal, train["value"].to_numpy(dtype=float),
                                   validation["value"].to_numpy(dtype=float), seed)
            point, _ = to_count(model, matrix(prep, x_test))
            label = "Facility MLP" if variant == "base" else "NFHS MLP"
            predicted[f"{label} · seed {seed}"] = point
    torch.set_num_threads(previous_threads)
    print("Replayed facility split counts", [len(split) for split in groups.values()],
          "and NFHS matches among test", int(inputs.loc[inputs["split"].eq("test"), "nfhs_matched"].eq(True).sum()))
    print("Pinned saved-model MAEs, possible overlap",
          {name: mean_absolute_error(actual, point) for name, point in artifact.items()})
    return pd.DataFrame([
        {"model": name, "mae": mean_absolute_error(actual, point),
         "mae_500_plus": np.mean(np.abs((actual - point)[actual >= 500])) if (actual >= 500).any() else np.nan}
        for name, point in predicted.items()])

shared_replay = ml_snapshot()
if shared_replay:
    shared_report = replay_shared_mlp()
    shared_label = "Workspace facility-held refit"
else:
    shared_report = ml_shared_archived
    shared_label = "Archived GitHub results · 522 facilities"
display(shared_report.round(2))
ml_bar(shared_report, "model", "mae", shared_label, "MAE (beds)")

### Population-density controls and the extreme prediction

The shared trainer links repeated facility IDs and clusters before assigning groups. Run the three feature variants on the same split and compare both the split and holdout fingerprints. The group split uses 60% for training, 20% validation and 20% test. Its early facilities-only control predicted 1,469,024 beds for a 109-bed facility. An out-of-range social count and rare procedure flag became +42.12 and −27.46 after standard scaling. The training-fitted uniform quantile transform bounded continuous inputs to [0, 1], left indicator values binary and reduced MAE from 3,042.13 to 126.48 on the same 504 test rows. The retest followed inspection of that holdout.

The teammate's density file was unavailable for the documented real-data comparison. The tracked district population CSV contains people, with no area denominator. Supply district or PIN density in `population_density` or `population_density_per_km2`. The file can also provide `population` and `land_area_km2`. Include `state` and `district` for district data or `pincode` for PIN data. Without it, the two enrichment cells demonstrate the join and comparability logic with deterministic synthetic values and label the scores accordingly. [Density data handoff](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/ML/models%20with%20population%20density/CLAUDE_HANDOFF.md)

In [ ]:
if have_ml_project:
    from density_capacity import train_capacity
    from density_data import join_nfhs, join_population_density, prepare_district_keys

    density_control = train_capacity(facility_ml, include_nfhs=False, include_density=False)
    district_keys_ml = prepare_district_keys(facility_ml, pin_ml)
    density_value = SRC.get("density")
    density_input_label = "Workspace district density"
    density_cfg = {"density_column": "population_density_per_km2"}
    if density_value:
        density_source = _read_export(Path(density_value))
        if "population_density" in density_source:
            density_cfg["density_column"] = "population_density"
        elif "population_density_per_km2" not in density_source and {"population", "land_area_km2"} <= set(density_source):
            density_cfg.update(density_column=None, population_column="population",
                               area_column="land_area_km2", area_unit="km2")
        if "pincode" in density_source.columns and not {"state", "district"} <= set(density_source.columns):
            density_cfg.update(key="pin", pin_column="pincode")
            density_input_label = "Workspace PIN density"
        else:
            density_cfg.update(key="district", state_column="state", district_column="district",
                               keys=district_keys_ml)
    else:
        density_input_label = "Synthetic join demonstration"
        density_source = district_keys_ml[["state_key", "district_key"]].dropna().drop_duplicates()
        density_source = density_source.loc[density_source["state_key"].ne("") & density_source["district_key"].ne("")]
        density_source = density_source.sort_values(["state_key", "district_key"]).reset_index(drop=True)
        density_source["population_density_per_km2"] = 20 + np.arange(len(density_source)) % 700
        density_cfg.update(key="district", state_column="state_key", district_column="district_key",
                           keys=district_keys_ml)
    with_density, density_receipt = join_population_density(facility_ml, density_source, **density_cfg)
    with_density_and_nfhs, density_nfhs_receipt = join_nfhs(with_density, nfhs_ml, keys=district_keys_ml)

    density_only = train_capacity(with_density, include_nfhs=False)
    density_and_nfhs = train_capacity(with_density_and_nfhs, include_nfhs=True)
    density_models = {"Facilities": density_control, "+ Density": density_only, "+ Density + NFHS": density_and_nfhs}
    assert len({(fit["split_fingerprint"], fit["holdout_fingerprint"]) for fit in density_models.values()}) == 1
    print(density_input_label, "split and holdout fingerprints",
          density_control["split_fingerprint"], density_control["holdout_fingerprint"])
    density_comparison = pd.concat([fit["test_cohorts"].assign(arm=arm) for arm, fit in density_models.items()])
    display(density_comparison.round(2))
    ml_bar(density_comparison.loc[density_comparison["cohort"].eq("all")], "arm", "mae",
           f"Population-density experiment · {density_input_label}", "MAE (beds)")
    archived_outlier = pd.DataFrame({"stage": ["Observed", "Legacy standard scaling", "Bounded preprocessing"],
                                     "beds": [109, 1469024, 11]})
    figure, axis = plt.subplots(figsize=(7, 3))
    axis.barh(archived_outlier["stage"], archived_outlier["beds"], color="#417a83")
    axis.set(xscale="log", xlabel="Beds (log scale)", title="Archived outlier diagnosis · 504 test rows")
    axis.invert_yaxis()
    figure.tight_layout()
    plt.show()
    plt.close(figure)
else:
    print("The density replays use the organizer files and optional density input.")

In [ ]:
if have_ml_project:
    import inspect
    import density_capacity
    from sklearn.preprocessing import StandardScaler as LegacyDensityScaler

    density_original = inspect.getsource(density_capacity.train_capacity)
    offset_from = density_original.index("    indicators = ")
    offset_to = density_original.index("    torch.manual_seed(seed)")
    legacy_density_prep = '    processor = ColumnTransformer([\n        ("numeric", Pipeline([("impute", SimpleImputer(strategy="median", keep_empty_features=True)),\n                              ("scale", StandardScaler())]), numeric),\n        ("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),\n                                  ("encode", OneHotEncoder(handle_unknown="ignore", min_frequency=5,\n                                                           sparse_output=False))]), list(CATEGORICAL)),\n    ]).fit(features.loc[train])\n\n'
    legacy_density_ns = dict(vars(density_capacity), StandardScaler=LegacyDensityScaler)
    exec(compile(density_original[:offset_from] + legacy_density_prep + density_original[offset_to:],
                 "af6238e legacy numeric preprocessing", "exec"), legacy_density_ns)
    density_legacy = legacy_density_ns["train_capacity"](facility_ml, include_nfhs=False, include_density=False)
    assert density_legacy["holdout_fingerprint"] == density_control["holdout_fingerprint"]
    density_preprocessing_compare = pd.DataFrame([
        {"method": "Median baseline", **density_control["baseline_metrics"]},
        {"method": "Standard scaling", **density_legacy["test_metrics"]},
        {"method": "Bounded preprocessing", **density_control["test_metrics"]},
    ])
    display(density_preprocessing_compare.round(2))
    figure, axis = plt.subplots(figsize=(7, 3))
    axis.barh(density_preprocessing_compare["method"], density_preprocessing_compare["mae"], color="#417a83")
    axis.set(xscale="log", xlabel="Test MAE (beds, log scale)", title="Same holdout and learner across preprocessing variants")
    axis.invert_yaxis()
    figure.tight_layout()
    plt.show()
    plt.close(figure)
else:
    display(pd.DataFrame([
        ("Training median", 151.11, 326.38), ("Legacy standard scaling", 3042.13, 65431.29),
        ("Bounded input transformation", 126.48, 284.89),
    ], columns=["archived arm", "test_mae", "test_rmse"]))
    print("Archived density-control cohort had 504 test rows.")

### XGBoost forests on audited reported capacity

The forest experiment reused the count splits after branch and location exclusions. It preserved predictors with missing values, used a separate categorical `missing` level, and compared a single 192-tree forest with 24 rounds of eight trees. Calibration chose the compact boosted forest. The documented split has 1,311 training, 441 calibration and 467 test facilities. The location gate excluded 895 source rows. The 123.00-bed MAE beat the 156.29-bed median baseline, with mean underprediction of 71.6 beds and a largest miss of 3,459. [Architecture and audit](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/ML/MODEL_EVALUATION.md#xgboost-forests-with-explicit-missing-predictors)

In [ ]:
from xgboost_capacity import train_xgboost_capacity
from density_data import join_nfhs, prepare_district_keys
xgb_original_replay = None
if ml_snapshot():
    xgb_args = [ml_checkpoint(name) for name in ("linked_fdr", "capacity_target_audit",
                "number_doctors_target_audit", "location_records", "location_entity_review")]
    xgb_core = train_xgboost_capacity(*xgb_args)
    xgb_linked = xgb_args[0]
    xgb_district_keys = prepare_district_keys(xgb_linked, pin_ml)
    xgb_enriched, _ = join_nfhs(xgb_linked, nfhs_ml, keys=xgb_district_keys)
    xgb_original_replay = train_xgboost_capacity(xgb_enriched, *xgb_args[1:])
    display(xgb_core["candidate_metrics"].assign(input_data="Facility data"))
    display(xgb_original_replay["candidate_metrics"].assign(input_data="Facility and NFHS data"))
    display(xgb_original_replay["exclusions"])
    print("Holdout fingerprint", xgb_core["holdout_fingerprint"], "facilities:", xgb_core["facility_counts"])
    assert xgb_original_replay["holdout_fingerprint"] == xgb_core["holdout_fingerprint"]
    forest_report = xgb_core["test_metrics"]
    histories = [pd.DataFrame(points).assign(model=name)
                 for name, points in xgb_core["training_history"].items()]
    figure, axis = plt.subplots(figsize=(7, 4))
    for history in histories:
        axis.plot(history["round"], history["validation_rmse_log1p"], label=history["model"].iloc[0])
    axis.set(xlabel="Boosting round", ylabel="RMSE on log1p scale", title="Calibration learning curves")
    axis.legend()
    figure.tight_layout()
    plt.show()
    plt.close(figure)
else:
    forest_report = pd.DataFrame({"model": ["Training median", "Standalone forest", "Boosted forest"],
                                  "mae": [156.29, 128.16, 123.00]})
    print("Archived GitHub scores · 467 facilities")
display(forest_report)
ml_bar(forest_report, "model", "mae", "Audited forest comparison", "Test MAE (beds)")

### Missingness, full-width leakage and label recovery tests

The first ablation toggled the numeric missingness indicators while leaving native `NaN`, categorical missing levels, rows and settings fixed. The test difference was −0.15 beds, with a paired interval from −1.30 to +1.01, and the flags helped three of ten additional seeds. The broad exploratory forest included all 50 other source fields and got 111.78 beds MAE versus 123.14 for core features without flags. It also had an important leakage risk. Text contained capacity-like wording in 360 test records and a number matching the target in 332 of them. Some content identifiers crossed the split. Masking recognized capacity phrases and removing the phone fields preserved much of the apparent gain, with 112.25 beds MAE. The report treats that result as exploratory evidence.

The first description pass kept exact whole-branch counts and rejected approximate, historical, external and unit-only statements. Twelve previously unreported branches had recoverable counts. Nine joined the training set and the calibration and test labels stayed fixed. The 183 overlapping structured and description fields agreed on 180 values. That is agreement within the same source. Across 13 configurations and additional MLP seeds, retraining gave mixed or small improvements. The tests also found no consistent benefit from adding a phone-listing flag. Listing status was retained for audit. [Experimental methods and results](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/ML/MODEL_EVALUATION.md#full-source-width-comparison)

In [ ]:
from hashlib import sha256
from pathlib import Path
import re
from time import perf_counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

from facility_tool.workspace import Store
from xgboost_capacity import (
    CORE_NUMERIC, DATE_FIELDS, FORESTS, METADATA_NUMERIC, SPLITS,
    _clean, _eligible_rows, _numeric, _scores, build_capacity_features,
)

NUMERIC = {*CORE_NUMERIC, *METADATA_NUMERIC, 'area', 'yearEstablished'}
CATEGORICAL = {
    'organization_type', 'content_table_id', 'acceptsVolunteers', 'address_city', 'address_stateOrRegion',
    'address_country', 'address_countryCode', 'countries', 'facilityTypeId', 'operatorTypeId',
    'affiliated_staff_presence', 'custom_logo_presence', 'source',
}
RICH_TEXT = {'description', 'specialties', 'procedure', 'equipment', 'capability', 'affiliationTypeIds'}
PHONES = {'phone_numbers', 'officialPhone'}
COUNT = r'[0-9][0-9,]*(?:\.[0-9]+)?'
BED_PHRASE = re.compile(
    rf'(?<![\w.])(?P<before>{COUNT})\s*[-–]?\s*(?:[a-z-]+\s+){{0,1}}bed(?:s|ded)?\b'
    rf'|\b(?:bed(?:s|\s+(?:capacity|count|strength))?|capacity)\s*(?:(?:is|of|about|over)\s+|[:=]\s*)?(?P<after>{COUNT})\b',
    re.I,
)
HOLDOUT = 'b14d649516340f1f2d872dd4582f6026cb805a35311cad3f4a8966240ae2c401'
MODEL_SETTINGS = dict(objective='reg:squarederror', tree_method='hist', max_depth=5,
                      min_child_weight=5, subsample=.8, colsample_bynode=.8, reg_lambda=2, n_jobs=2)


def audited_inputs():
    linked = ml_checkpoint("linked_fdr")
    cols = list(linked.columns.drop("facility_id"))
    rows, labels, excluded = _eligible_rows(
        linked, ml_checkpoint("capacity_target_audit"), ml_checkpoint("number_doctors_target_audit"),
        ml_checkpoint("location_records"), ml_checkpoint("location_entity_review"),
    )
    reported = labels["status"].eq("reported")
    train, calibration, test = [reported & labels["split"].eq(name) for name in SPLITS]
    target = _numeric(labels["value"])
    digest = sha256("\n".join(f"{key}:{value:.0f}" for key, value in target.loc[test].items()).encode()).hexdigest()
    print("Workspace ablation cohort", [int(mask.sum()) for mask in (train, calibration, test)], digest)
    if digest != HOLDOUT:
        print("Workspace checkpoint differs from the archived 467-facility holdout. Use separate comparisons.")
    features = rows[[field for field in cols if field != "capacity"]].copy()
    features["coordinates"] = features["coordinates"].where(features["latitude"].notna() & features["longitude"].notna())
    return features, labels, target, train, calibration, test


def bed_matches(value):
    return {float((m['before'] or m['after']).replace(',', '')) for m in BED_PHRASE.finditer(value)}


def bed_text_check(rows, target, masks):
    hits = pd.Series([set() for _ in range(len(rows))], index=rows.index)
    for field in RICH_TEXT | {'name'}:
        matches = _clean(rows[field]).fillna('').map(bed_matches)
        hits = pd.Series([one | other for one, other in zip(hits, matches)], index=hits.index)
    any_hint = hits.map(bool)
    exact = pd.Series([beds in extracted for beds, extracted in zip(target, hits)], index=hits.index)
    for name, mask in masks.items():
        print('Text audit', name, int(mask.sum()), int(any_hint.loc[mask].sum()), int(exact.loc[mask].sum()), flush=True)
    return any_hint


def numeric_values(values):
    clean = _clean(values)
    grouped = clean.str.fullmatch(r'[+-]?(?:\d{1,3}(?:,\d{3})+|\d{1,2}(?:,\d{2})+,\d{3})(?:\.\d+)?').fillna(False)
    return _numeric(clean.mask(grouped, clean.str.replace(',', '', regex=False)))


def block(values, field, train):
    if field in NUMERIC:
        numeric = numeric_values(values)
        return numeric.to_numpy(dtype=np.float32)[:, None], numeric.isna().to_numpy()
    if field in DATE_FIELDS:
        dates = pd.to_datetime(values.str[:10], format='%Y-%m-%d', errors='coerce')
        days = (dates - pd.Timestamp('1970-01-01')).dt.days
        return days.to_numpy(dtype=np.float32)[:, None], days.isna().to_numpy()
    docs = values.fillna('')
    missing = values.isna().to_numpy()
    if field in CATEGORICAL:
        levels = docs.loc[train].value_counts()
        common = set(levels.index[(levels >= 5) & (levels.index != '')])
        categories = np.asarray(sorted(common | {'[missing]', '[other]'}), dtype=object)
        codes = values.where(values.isin(common), '[other]').mask(values.isna(), '[missing]').to_numpy()[:, None]
        encoder = OneHotEncoder(categories=[categories], handle_unknown='error', sparse_output=False, dtype=np.float32)
        return encoder.fit(codes[train.to_numpy()]).transform(codes), missing
    # Train-only frequency and text vocabulary keep identifiers nominal and support novel values.
    occurrence = values.loc[train].value_counts()
    exact_frequency = values.map(occurrence).fillna(0).where(values.notna())
    length = values.str.len().to_numpy(dtype=np.float32, na_value=np.nan)
    summary = np.column_stack((length, exact_frequency.to_numpy(dtype=np.float32)))
    vectorizer = TfidfVectorizer(
        token_pattern=r'(?u)\b\w{2,}\b', min_df=4, max_features=96 if field in RICH_TEXT else 24,
        sublinear_tf=True, dtype=np.float32,
    )
    if docs.loc[train].str.contains(r'\b\w{2,}\b', regex=True).sum() < 4:
        return summary, missing
    try:
        vectorizer.fit(docs.loc[train])
    except ValueError as error:
        if 'no terms remain' not in str(error).lower() and 'empty vocabulary' not in str(error).lower():
            raise
        return summary, missing
    return np.column_stack((summary, vectorizer.transform(docs).toarray())), missing


def encode(rows, train, redact=False):
    blocks = {}
    for field in rows.columns:
        values = _clean(rows[field])
        if redact and field not in NUMERIC and field not in DATE_FIELDS:
            values = values.str.replace(BED_PHRASE, ' [bed_count] ', regex=True)
        columns, missing = block(values, field, train)
        # Categorical fields already have an explicit missing level in both arms.
        flags = missing[:, None].astype(np.float32) if field not in CATEGORICAL else None
        blocks[field] = columns, flags
    return blocks


def assemble(blocks, include_flags, skip=()):
    selected = [(key, block, flag) for key, (block, flag) in blocks.items() if key not in skip]
    columns = [block for _, block, _ in selected]
    if include_flags:
        columns += [flag for _, _, flag in selected if flag is not None]
    matrix = np.concatenate(columns, axis=1, dtype=np.float32)
    assert not np.isinf(matrix).any()
    return matrix


def core_matrix(rows, train):
    frame = rows.copy()
    features, core, _, category_fields = build_capacity_features(frame)
    numeric = [key for key in core if key not in category_fields and not key.endswith('_missing')]
    categories = [key for key in core if key in category_fields]
    levels = [sorted({'missing', *features.loc[train, key].unique()}) for key in categories]
    encoder = OneHotEncoder(categories=levels, handle_unknown='ignore', sparse_output=False)
    category_values = encoder.fit(features.loc[train, categories]).transform(features[categories])
    return np.concatenate((features[numeric].to_numpy(), category_values), axis=1)


def fit(matrix, variant, split, target, seed=42):
    train, calibration, test = split
    model = XGBRegressor(**FORESTS[variant], **MODEL_SETTINGS, random_state=seed)
    model.fit(matrix[train], np.log1p(target.loc[train].to_numpy()))
    prediction = [np.maximum(0, np.rint(np.expm1(model.predict(matrix[mask]).astype(float)))) for mask in (calibration, test)]
    assert all(np.isfinite(p).all() for p in prediction)
    return prediction


def compare(actual, current, base):
    delta = np.abs(current - actual) - np.abs(base - actual)
    rng = np.random.default_rng(20261121)
    sample_ix = rng.integers(0, len(delta), size=(20000, len(delta)))
    bounds = np.quantile(delta[sample_ix].mean(axis=1), [.025, .975])
    return [delta.mean(), *bounds]


def engineered_core(rows, train, flags):
    data, fields, _, categorical = build_capacity_features(rows)
    numeric = [key for key in fields if key not in categorical and (flags or not key.endswith('_missing'))]
    categories = [key for key in fields if key in categorical]
    levels = [sorted({'missing', *data.loc[train, key].unique()}) for key in categories]
    encoder = OneHotEncoder(categories=levels, handle_unknown='ignore', sparse_output=False)
    encoded = encoder.fit(data.loc[train, categories]).transform(data[categories])
    return np.concatenate((data[numeric].to_numpy(), encoded), axis=1)


def scores(actual, predicted):
    diff = predicted - actual
    ae = np.abs(diff)
    return {'mae': float(ae.mean()), 'rmse': float(np.sqrt(np.mean(diff**2))), 'p90': float(np.quantile(ae, .9))}

In [ ]:
archived_width = pd.DataFrame([('Core · no flags', 117.52, 123.14, 287.01), ('Core · flags', 118.12, 123.0, 286.38), ('50 · standalone · no flags', 108.32, 115.99, 271.6), ('50 · standalone · flags', 108.03, 116.34, 271.62), ('50 · boosted · no flags', 105.55, 111.78, 268.27), ('50 · boosted · flags', 105.7, 112.46, 268.65), ('50 masked · no flags', 104.54, 112.12, 267.73), ('50 masked · flags', 105.11, 112.11, 266.0), ('48 without phone fields', 105.88, 112.53, 270.05), ('48 masked without phone fields', 104.94, 112.25, 267.85)], columns=["label", "validation_mae", "test_mae", "test_rmse"])

In [ ]:
deep_capacity = bool(globals().get("FULL_REPLAY", False)) and ml_snapshot() is not None
if deep_capacity:
    width_rows, width_labels, width_target, width_train, width_cal, width_test = audited_inputs()
    ablation_split = (width_train, width_cal, width_test)
    ablation_outcomes = width_target.loc[width_test].to_numpy()
    ablation_calibration = width_target.loc[width_cal].to_numpy()
    text_mentions = bed_text_check(width_rows, width_target, {"test": width_test,
                                 "missing capacity": width_labels["status"].eq("missing")})
    # Legacy raw source phone fields are used only inside the historical ablation.
    # No facilities, phone values, learned tokens or predictions are displayed here.
    enc_full = encode(width_rows, width_train)
    enc_masked = encode(width_rows, width_train, redact=True)
    standalone, boosted = "xgboost_random_forest", "xgboost_boosted_forest"
    width_matrices = {
        "Core · no flags": (engineered_core(width_rows, width_train, False), boosted),
        "Core · flags": (engineered_core(width_rows, width_train, True), boosted),
        "50 · standalone · no flags": (assemble(enc_full, False), standalone),
        "50 · standalone · flags": (assemble(enc_full, True), standalone),
        "50 · boosted · no flags": (assemble(enc_full, False), boosted),
        "50 · boosted · flags": (assemble(enc_full, True), boosted),
        "50 masked · no flags": (assemble(enc_masked, False), boosted),
        "50 masked · flags": (assemble(enc_masked, True), boosted),
        "48 without phone fields": (assemble(enc_full, False, skip=PHONES), boosted),
        "48 masked without phone fields": (assemble(enc_masked, False, skip=PHONES), boosted),
    }
    width_scores, width_predictions = [], {}
    for label, (matrix, variant) in width_matrices.items():
        p_cal, p_test = fit(matrix, variant, ablation_split, width_target)
        width_predictions[label] = p_test
        no_hint = ~text_mentions.loc[width_test].to_numpy()
        width_scores.append({"label": label, "validation_mae": mean_absolute_error(ablation_calibration, p_cal),
                             "test_mae": mean_absolute_error(ablation_outcomes, p_test),
                             "test_rmse": np.sqrt(mean_squared_error(ablation_outcomes, p_test)),
                             "no_mention_mae": (mean_absolute_error(ablation_outcomes[no_hint], p_test[no_hint])
                                                if no_hint.any() else np.nan)})
    print("Core flags minus no flags [MAE change, paired CI]",
          compare(ablation_outcomes, width_predictions["Core · flags"], width_predictions["Core · no flags"]))
    sensitivity = []
    for seed in range(43, 53):
        for no_flag, with_flag in [("Core · no flags", "Core · flags"),
                                   ("50 · boosted · no flags", "50 · boosted · flags")]:
            predictions = []
            for arm in (no_flag, with_flag):
                matrix, variant = width_matrices[arm]
                _, point = fit(matrix, variant, ablation_split, width_target, seed=seed)
                predictions.append(mean_absolute_error(ablation_outcomes, point))
            sensitivity.append({"inputs": no_flag.split(" · ")[0], "seed": seed, "change": predictions[1] - predictions[0]})
    sensitivity = pd.DataFrame(sensitivity)
    display(sensitivity.groupby("inputs")["change"].agg(["mean", "min", "max"]))
    width_report = pd.DataFrame(width_scores)
    width_label = "Workspace checkpoint replay"
else:
    width_report = archived_width
    width_label = "Archived GitHub results · 467 facilities"
display(width_report.round(2))
ml_bar(width_report, "label", "test_mae", width_label, "Test MAE (beds)")

In [ ]:
from collections import Counter
import json
from pathlib import Path
import re
from urllib.parse import urlsplit

import numpy as np
import pandas as pd

from facility_tool.workspace import Store
from xgboost_capacity import _eligible_rows


NUMBER = r"(?:\d{1,3}(?:,\d{2,3})+|\d+)"
MENTION = re.compile(
    rf"(?<![\w.,])(?P<leading>{NUMBER})\s*[-–—]?\s*(?:(?:inpatient|hospital)\s+)?bed(?:s|ded)?\b"
    rf"|\b(?:beds?\s*(?:capacity|count|strength)?|capacity)\s*(?:(?:is|of)\s+)?[:=]?\s*(?P<trailing>{NUMBER})(?!\w|[.,]\d)",
    re.I,
)
SENTENCE_BOUNDARY = re.compile(r"(?<=[.!?;])\s+|\r?\n")
APPROXIMATED = re.compile(
    r"(?:more than|less than|greater than|just under|just over|over|under|over a total of|"
    r"at least|at most|up to|around|about|approx(?:imately)?\.?|nearly|at capacity of)\s*$|"
    r"\d\s*(?:[-–—]|to|or)\s*$|[+<>~≤≥]\s*$|\bbetween\b[^.;]{0,25}$", re.I,
)
HISTORICAL_OR_PLANNED = re.compile(
    r"\b(?:began|started (?:out|with|as)|originally|former(?:ly)?|previously|once had|initially|"
    r"first opened|was (?:established|founded|opened) (?:as|with)|formerly|"
    r"grew from|increased from|expanded from|rose from|used to|discontinued|"
    r"will\s+(?:be|add|have|offer|house|provide)|plans? to|expected to|projected|"
    r"planned (?:capacity|bed)|proposed|under construction|being built|temporary|field hospital)\b", re.I,
)
DISCONTINUED_INPATIENT = re.compile(
    r"\b(?:indoor|inpatient|hospitalization|overnight)\b[^.!?;]{0,70}\b(?:discontinued|closed|ceased|stopped)\b", re.I,
)
AGGREGATE = re.compile(
    r"\b(?:across\s+(?:all|its|the|\d)|\d+\s+(?:hospitals|branches|locations)|"
    r"network|group\s+of\s+hospitals|chains?\s+of\s+hospitals|combined|aggregate|between\s+\w+\s+and)\b", re.I,
)
SUBUNIT = re.compile(
    r"\b(?:icu|nicu|picu|sicu|ccu|itu|intensive care|critical care|emergency|trauma|dialysis|"
    r"neonatal|paediatric|pediatric|onco\w*|obstetrics|gynecology|maternity|cardiac|burn|"
    r"hdu|day care|ward|room|floor|unit|department|wing|isolation)\b", re.I,
)
EXTERNAL = re.compile(
    r"\b(?:associated hospital|nearby hospital|adjacent hospital|referral hospital has|"
    r"affiliated hospital|parent hospital|at another facility|at hotel|tie[- ]?up|"
    r"beds?\s+(?:at|of)\s+(?!the\s+(?:site|facility|hospital)\b)[A-Za-z][\w'-]*)\b", re.I,
)
WHOLE_DESCRIPTOR = re.compile(
    r"^\s+(?:(?:multi[- ]?special(?:ity|ty)|super[- ]?special(?:ity|ty)|regional|referral|tertiary|"
    r"secondary|charitable|multidisciplinary|special(?:ity|ty)|general|modern|private|government|"
    r"teaching|speciali[sz]ed|paediatric|pediatric|children's|heart|eye|comprehensive|rural|district|"
    r"portable|multilevel|state-of-the-art)\s+){0,6}"
    r"(?:hospital|nursing home|medical cent(?:re|er)|regional referral cent(?:re|er)|facility)\b", re.I,
)
WHOLE_SUBJECT = re.compile(
    r"\b(?:hospital|facility|nursing home|medical cent(?:re|er)|it|we)\s+"
    r"(?:now\s+|currently\s+)?(?:has|have|houses|accommodates|maintains|has a capacity of|"
    r"includes|boasts|comprises|consists of|provides|provided|offers|features|has been equipped with|"
    r"have also provided|is equipped with)\s+(?:a total of\s+|a capacity of\s+)?$"
    r"|\b(?:now\s+boasting|bed capacity\s+(?:of|is)|a\s+total\s+of)\s*$", re.I,
)
WHOLE_FROM_RIGHT = re.compile(r"^[\s,]+(?:the\s+)?(?:hospital|facility|nursing home)\s+(?:is|offers|has|provides)\b", re.I)
UNIT_AFTER = re.compile(
    r"^[\s-]+(?:(?:for\s+the|surgical|medical|paediatric|pediatric|neonatal|level\s+\w+)\s+){0,2}"
    r"(?:icu|nicu|picu|sicu|ccu|intensive care|critical care|trauma|dialysis|wards?|rooms?|units?)\b", re.I,
)
WHOLE_CAPACITY = re.compile(r"\b(?:total\s+(?:of\s+)?beds?|bed\s+(?:capacity|count|strength)|total\s+(?:inpatient\s+)?capacity)\b", re.I)
OLD_DATE = re.compile(r"\b(?:in|by|until|as of)\s+(?:18|19)\d{2}\b", re.I)
PAST_ONLY = re.compile(r"\b(?:it|hospital|facility)\s+(?:had|was)\b(?:(?!\b(?:now|currently|has|houses)\b).)*$", re.I)
RANGE_AFTER = re.compile(r"^\s*(?:[+<>~≤≥]|(?:[-–—]|to|or)\s*\d)", re.I)
URL = re.compile(r"^https?://[^\s]+$", re.I)
NULL_WORDS = {"", "null", "nan", "none", "not available", "not stated", "unknown", "n/a", "na"}


def text(value):
    return str(value).strip() if pd.api.types.is_scalar(value) and not pd.isna(value) else ""


def source_link(record):
    try:
        urls = json.loads(text(record["source_urls"]))
    except (TypeError, ValueError):
        urls = []
    source_urls = [url for url in urls if isinstance(url, str) and URL.fullmatch(url)] if isinstance(urls, list) else []
    official = text(record["officialWebsite"])
    if URL.fullmatch(official):
        def host(url):
            return urlsplit(url).netloc.lower().removeprefix("www.")

        for url in source_urls:
            if host(url) == host(official):
                return url
    return next(iter(source_urls), official if URL.fullmatch(official) else None)


def extract_description(description):
    """Return validated candidate excerpts and counts noticed with uncertain scope."""
    verified = []
    saw_count = False
    if DISCONTINUED_INPATIENT.search(description):
        return verified, bool(MENTION.search(description))
    for sentence in SENTENCE_BOUNDARY.split(description):
        for hit in MENTION.finditer(sentence):
            saw_count = True
            left = sentence[:hit.start()]
            right = sentence[hit.end():]
            clause_left = re.split(r"[,;:]|\s+and\s+", left)[-1]
            clause_right = re.split(r"[,;:]|\s+and\s+", right)[0]
            count = int((hit["leading"] or hit["trailing"]).replace(",", ""))
            if (not 1 <= count <= 10000 or APPROXIMATED.search(left) or RANGE_AFTER.search(right)
                    or HISTORICAL_OR_PLANNED.search(sentence) or AGGREGATE.search(sentence)
                    or EXTERNAL.search(sentence) or OLD_DATE.search(clause_left + hit[0] + clause_right)
                    or PAST_ONLY.search(left) or UNIT_AFTER.search(right)):
                continue

            bounded_total = bool(WHOLE_DESCRIPTOR.match(right))
            explicit_total = bool(WHOLE_CAPACITY.search(hit[0]) or WHOLE_SUBJECT.search(left)
                                  or WHOLE_FROM_RIGHT.match(right))
            # A facility descriptor or direct subject supplies scope. Unit descriptions and external
            # references near an otherwise terse bed count cannot stand in for a branch-wide total.
            if explicit_total and SUBUNIT.search(clause_left[max(0, len(clause_left) - 65):]):
                explicit_total = False
            if bounded_total or explicit_total:
                evidence = sentence.strip()
                if len(evidence) > 420:
                    evidence = sentence[max(0, hit.start() - 120): min(len(sentence), hit.end() + 150)].strip()
                verified.append((count, evidence))
    return verified, saw_count


from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from density_capacity import CATEGORICAL, TEXT, _features
from density_data import NFHS_FEATURES, join_nfhs, prepare_district_keys
from facility_tool.count_models import CATEGORY_NAMES, _fit, _point, _select_features, _transformer
from facility_tool.workspace import Store
from xgboost_capacity import _clean, _eligible_rows, _numeric, _scores

SEED = 42
BED_NUMBER = r'\b\d+(?:[,.]\d+)*\b'
BED_WORD = r'\bbeds?(?:ded|ding)?\b'
BED_CUE = re.compile(rf'{BED_NUMBER}.{{0,26}}{BED_WORD}|{BED_WORD}.{{0,26}}{BED_NUMBER}', re.IGNORECASE)


def classical(rows, target, new_target, train, augmented, test):
    frame = pd.DataFrame(index=rows.index)
    for name, mapping in CATEGORY_NAMES.items():
        frame[name] = _clean(rows[name]).map(mapping).fillna('(unknown)')
    doctors = _numeric(rows['numberDoctors'])
    frame['numberDoctors'] = np.log1p(doctors)
    frame['other_count_reported'] = doctors.notna()
    frame['value'] = target
    features, _, _, _ = _select_features('capacity', frame.loc[train].reset_index(),
                                         [*CATEGORY_NAMES, 'numberDoctors'], SEED)
    assert features
    print('Legacy candidate features selected on original training labels:', features, flush=True)
    runs = []
    for mask, outcome in ((train, target), (augmented, new_target)):
        inputs = frame.loc[mask].assign(value=outcome.loc[mask])
        transformer = _transformer(features).fit(inputs)
        models = _fit(transformer, inputs, SEED)
        x_test = transformer.transform(frame.loc[test])
        runs.append({name: _point(model, x_test) for name, model in models.items()})
    return runs


def mlp_features(rows, include_nfhs):
    frame = rows.drop(columns=['description']).copy()
    frame['capacity'] = np.nan
    for col in TEXT:
        frame[col] = frame[col].astype('string').str.replace(BED_CUE, '', regex=True)
    return _features(frame, include_density=False, include_nfhs=include_nfhs)


def mlp(features, numeric, target, train, cal, test, seed=42):
    indicators = [field for field in numeric if field.endswith('_present')]
    continuous = [field for field in numeric if field not in indicators]
    processor = ColumnTransformer([
        ('numeric', Pipeline([
            ('impute', SimpleImputer(strategy='median', keep_empty_features=True)),
            ('quantile', QuantileTransformer(
                n_quantiles=min(1000, int(train.sum())), output_distribution='uniform', subsample=None,
            )),
        ]), continuous),
        ('indicators', 'passthrough', indicators),
        ('categorical', Pipeline([
            ('impute', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
            ('encode', OneHotEncoder(handle_unknown='ignore', min_frequency=5, sparse_output=False)),
        ]), list(CATEGORICAL)),
    ]).fit(features.loc[train])
    torch.manual_seed(seed)

    def tensor(mask):
        return torch.as_tensor(processor.transform(features.loc[mask]), dtype=torch.float32)

    def log_target(mask):
        return torch.as_tensor(np.log1p(target.loc[mask].to_numpy()), dtype=torch.float32)[:, None]

    x_train, x_val = tensor(train), tensor(cal)
    y_val = log_target(cal)
    loader = DataLoader(
        TensorDataset(x_train, log_target(train)), batch_size=256, shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )
    model = nn.Sequential(
        nn.Linear(x_train.shape[1], 128), nn.ReLU(), nn.Dropout(.2),
        nn.Linear(128, 64), nn.ReLU(), nn.Dropout(.2), nn.Linear(64, 1),
    )
    optim = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, factor=.5, patience=10)
    loss = nn.HuberLoss(delta=1)
    best_loss, bad_epochs, best_weights = float('inf'), 0, None
    for _ in range(300):
        model.train()
        for x_batch, y_batch in loader:
            optim.zero_grad()
            loss(model(x_batch), y_batch).backward()
            optim.step()
        model.eval()
        with torch.no_grad():
            observed = float(loss(model(x_val), y_val))
        scheduler.step(observed)
        if observed < best_loss - 1e-4:
            best_loss, bad_epochs = observed, 0
            best_weights = {key: value.detach().clone() for key, value in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= 30:
                break
    assert best_weights is not None
    model.load_state_dict(best_weights)
    model.eval()
    with torch.inference_mode():
        predictions = np.maximum(0, np.rint(np.expm1(model(tensor(test)).flatten().numpy())))
    assert np.isfinite(predictions).all()
    return predictions


def result(name, actual, baseline, augmented, n_added):
    original = _scores(name, actual, baseline)
    extra = _scores(name, actual, augmented)
    row = {'model': name, 'training_added': int(n_added)}
    for field in ('mae', 'rmse', 'p90_ae'):
        suffix = 'p90' if field == 'p90_ae' else field
        row[f'baseline_{suffix}'] = original[field]
        row[f'augmented_{suffix}'] = extra[field]
    print(row, flush=True)
    return row



from torch.utils.data import DataLoader, TensorDataset

In [ ]:
archived_description_labels = pd.DataFrame([{'model': 'Log-ridge regression',
  'training_added': 9,
  'baseline_mae': 140.036,
  'augmented_mae': 140.049,
  'baseline_rmse': 327.989,
  'augmented_rmse': 327.971,
  'baseline_p90': 345.4,
  'augmented_p90': 345.4},
 {'model': 'Scikit-learn log-random forest',
  'training_added': 9,
  'baseline_mae': 142.317,
  'augmented_mae': 142.557,
  'baseline_rmse': 332.05,
  'augmented_rmse': 332.33,
  'baseline_p90': 358.2,
  'augmented_p90': 360.4},
 {'model': 'Facilities-only MLP',
  'training_added': 9,
  'baseline_mae': 131.345,
  'augmented_mae': 131.193,
  'baseline_rmse': 302.892,
  'augmented_rmse': 304.293,
  'baseline_p90': 342.6,
  'augmented_p90': 339.8},
 {'model': 'NFHS MLP',
  'training_added': 9,
  'baseline_mae': 133.704,
  'augmented_mae': 128.567,
  'baseline_rmse': 307.822,
  'augmented_rmse': 298.82,
  'baseline_p90': 350.2,
  'augmented_p90': 323.8},
 {'model': 'Original post-countries random forest',
  'training_added': 9,
  'mae_delta_ci_low': -1.85,
  'mae_delta_ci_high': 1.743,
  'baseline_mae': 124.366,
  'baseline_rmse': 249.062,
  'baseline_p90': 342.0,
  'augmented_mae': 124.315,
  'augmented_rmse': 249.929,
  'augmented_p90': 346.8},
 {'model': 'XGBoost random forest (core, no flags)',
  'training_added': 9,
  'columns': 15,
  'mae_delta_ci_low': -1.64,
  'mae_delta_ci_high': 0.407,
  'baseline_mae': 128.632,
  'baseline_rmse': 295.229,
  'baseline_p90': 348.0,
  'augmented_mae': 128.041,
  'augmented_rmse': 294.178,
  'augmented_p90': 349.2},
 {'model': 'XGBoost boosted forest (core, no flags)',
  'training_added': 9,
  'columns': 15,
  'mae_delta_ci_low': -2.188,
  'mae_delta_ci_high': 0.786,
  'baseline_mae': 123.143,
  'baseline_rmse': 287.012,
  'baseline_p90': 328.2,
  'augmented_mae': 122.48,
  'augmented_rmse': 285.474,
  'augmented_p90': 336.0},
 {'model': 'XGBoost random forest (core, flags)',
  'training_added': 9,
  'columns': 23,
  'mae_delta_ci_low': -1.559,
  'mae_delta_ci_high': 0.233,
  'baseline_mae': 128.165,
  'baseline_rmse': 295.166,
  'baseline_p90': 353.4,
  'augmented_mae': 127.514,
  'augmented_rmse': 292.867,
  'augmented_p90': 344.4},
 {'model': 'XGBoost boosted forest (core, flags)',
  'training_added': 9,
  'columns': 23,
  'mae_delta_ci_low': -0.58,
  'mae_delta_ci_high': 1.589,
  'baseline_mae': 122.998,
  'baseline_rmse': 286.377,
  'baseline_p90': 343.0,
  'augmented_mae': 123.49,
  'augmented_rmse': 285.84,
  'augmented_p90': 332.4},
 {'model': 'XGBoost random forest (47 fields, no flags)',
  'training_added': 9,
  'columns': 871,
  'mae_delta_ci_low': -0.218,
  'mae_delta_ci_high': 2.09,
  'baseline_mae': 116.036,
  'baseline_rmse': 270.582,
  'baseline_p90': 308.6,
  'augmented_mae': 116.983,
  'augmented_rmse': 273.63,
  'augmented_p90': 321.8},
 {'model': 'XGBoost boosted forest (47 fields, no flags)',
  'training_added': 9,
  'columns': 871,
  'mae_delta_ci_low': -2.011,
  'mae_delta_ci_high': 1.086,
  'baseline_mae': 112.801,
  'baseline_rmse': 267.571,
  'baseline_p90': 277.0,
  'augmented_mae': 112.347,
  'augmented_rmse': 264.792,
  'augmented_p90': 273.0},
 {'model': 'XGBoost random forest (47 fields, flags)',
  'training_added': 9,
  'columns': 905,
  'mae_delta_ci_low': -1.045,
  'mae_delta_ci_high': 1.495,
  'baseline_mae': 115.535,
  'baseline_rmse': 270.472,
  'baseline_p90': 312.6,
  'augmented_mae': 115.762,
  'augmented_rmse': 270.325,
  'augmented_p90': 317.4},
 {'model': 'XGBoost boosted forest (47 fields, flags)',
  'training_added': 9,
  'columns': 905,
  'mae_delta_ci_low': -0.388,
  'mae_delta_ci_high': 3.032,
  'baseline_mae': 111.355,
  'baseline_rmse': 266.125,
  'baseline_p90': 268.6,
  'augmented_mae': 112.675,
  'augmented_rmse': 266.91,
  'augmented_p90': 272.4}])
archived_phone_ablation = pd.DataFrame([{'cohort': 'with recovered labels',
  'seed': 42,
  'variant': 'without status',
  'calibration_mae': 106.986,
  'mae': 112.347,
  'rmse': 264.792,
  'p90': 273.0,
  'listed_mae': 111.0,
  'not_listed_mae': 163.417},
 {'cohort': 'with recovered labels',
  'seed': 42,
  'variant': 'with status',
  'calibration_mae': 106.163,
  'mae': 112.338,
  'rmse': 267.342,
  'p90': 262.8,
  'listed_mae': 110.732,
  'not_listed_mae': 173.25,
  'test_paired_delta': -0.009,
  'test_paired_95_interval': [-1.5802997858672376, 1.642451820128475]},
 {'cohort': 'with recovered labels',
  'seed': 104,
  'variant': 'without status',
  'calibration_mae': 104.798,
  'mae': 111.398,
  'rmse': 264.099,
  'p90': 281.8,
  'listed_mae': 109.936,
  'not_listed_mae': 166.833},
 {'cohort': 'with recovered labels',
  'seed': 104,
  'variant': 'with status',
  'calibration_mae': 105.286,
  'mae': 112.176,
  'rmse': 269.993,
  'p90': 269.4,
  'listed_mae': 110.855,
  'not_listed_mae': 162.25},
 {'cohort': 'with recovered labels',
  'seed': 2026,
  'variant': 'without status',
  'calibration_mae': 105.621,
  'mae': 110.901,
  'rmse': 264.576,
  'p90': 271.0,
  'listed_mae': 109.501,
  'not_listed_mae': 164.0},
 {'cohort': 'with recovered labels',
  'seed': 2026,
  'variant': 'with status',
  'calibration_mae': 104.819,
  'mae': 111.334,
  'rmse': 264.441,
  'p90': 273.6,
  'listed_mae': 109.985,
  'not_listed_mae': 162.5},
 {'cohort': 'reported labels only',
  'seed': 42,
  'variant': 'without status',
  'calibration_mae': 105.376,
  'mae': 112.801,
  'rmse': 267.571,
  'p90': 277.0,
  'listed_mae': 111.484,
  'not_listed_mae': 162.75},
 {'cohort': 'reported labels only',
  'seed': 42,
  'variant': 'with status',
  'calibration_mae': 104.893,
  'mae': 111.867,
  'rmse': 267.182,
  'p90': 264.6,
  'listed_mae': 110.503,
  'not_listed_mae': 163.583,
  'test_paired_delta': -0.934,
  'test_paired_95_interval': [-2.5524625267665955, 0.6531584582441067]}])

In [ ]:
from facility_tool.data import _array
from facility_tool.profiles import items
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from capacity_rf import FacilityEncoder, cell_text

if deep_capacity:
    linked_for_recovery = ml_checkpoint("linked_fdr")
    extracted_beds = {}
    for facility_id, group in linked_for_recovery.groupby("facility_id"):
        if facility_id not in width_rows.index:
            continue
        exact_counts = {count for description in group["description"] if text(description).casefold() not in NULL_WORDS
                        for count, _ in extract_description(text(description))[0]}
        if len(exact_counts) == 1:
            extracted_beds[facility_id] = next(iter(exact_counts))
    extraction = pd.Series(extracted_beds, dtype=float).reindex(width_rows.index)
    paired = extraction.notna() & width_target.notna()
    print("Co-observed facilities", int(paired.sum()), "matching counts",
          int(extraction.loc[paired].eq(width_target.loc[paired]).sum()))
    new_cases = width_labels["status"].eq("missing") & width_labels["split"].eq("train") & extraction.notna()
    augmented_target = width_target.where(~new_cases, extraction)
    augmented_train = width_train | new_cases
    print("Candidate extra training labels with an unchanged holdout", int(new_cases.sum()))
    assert not (new_cases & (width_cal | width_test)).any()
    desc_rows = []
    no_description = width_rows.drop(columns=["description", *PHONES])
    encoded = encode(no_description, width_train, redact=True)
    raw, expanded = classical(width_rows, width_target, augmented_target, width_train, augmented_train, width_test)
    for key, label in (("regression", "Log-ridge regression"), ("ml", "Scikit-learn log-random forest")):
        desc_rows.append(result(label, ablation_outcomes, raw[key], expanded[key], new_cases.sum()))

    nfhs_for_ablation, _ = join_nfhs(width_rows, nfhs_ml, keys=prepare_district_keys(width_rows, pin_ml))
    for use_nfhs, features_frame in ((False, width_rows), (True, nfhs_for_ablation)):
        predictors, numeric = mlp_features(features_frame, use_nfhs)
        for seed in ((42, 104, 2026) if use_nfhs else (42,)):
            before = mlp(predictors, numeric, width_target, width_train, width_cal, width_test, seed=seed)
            after = mlp(predictors, numeric, augmented_target, augmented_train, width_cal, width_test, seed=seed)
            label = ("NFHS MLP" if use_nfhs else "Facility MLP") + f" · {seed}"
            desc_rows.append(result(label, ablation_outcomes, before, after, new_cases.sum()))

    post_boundary = width_rows.iloc[:, width_rows.columns.get_loc("countries") + 1:].drop(columns="description").copy()
    for field in post_boundary.select_dtypes(include=["object", "string"]):
        post_boundary[field] = post_boundary[field].map(cell_text).str.replace(BED_PHRASE, " [bed_count] ", regex=True)
    post_results = []
    for mask, target in ((width_train, width_target), (augmented_train, augmented_target)):
        forest = make_pipeline(FacilityEncoder(128, 40), RandomForestRegressor(
            n_estimators=300, max_features=.7, min_samples_leaf=2, n_jobs=2, random_state=42))
        forest.fit(post_boundary.loc[mask], target.loc[mask])
        post_results.append(np.maximum(0, np.rint(forest.predict(post_boundary.loc[width_test]))))
    desc_rows.append(result("Post-countries random forest", ablation_outcomes, *post_results, new_cases.sum()))

    for flags in (False, True):
        for bundle, matrix in (("Core", engineered_core(width_rows, width_train, flags)),
                               ("47 masked fields", assemble(encoded, flags))):
            for name, variant in (("Standalone", "xgboost_random_forest"),
                                  ("Boosted", "xgboost_boosted_forest")):
                _, before = fit(matrix, variant, ablation_split, width_target)
                _, after = fit(matrix, variant, (augmented_train, width_cal, width_test), augmented_target)
                label = f"{name} · {bundle} · {'flags' if flags else 'no flags'}"
                desc_rows.append(result(label, ablation_outcomes, before, after, new_cases.sum()))
    description_labels_report = pd.DataFrame(desc_rows)
    desc_label = "Workspace recovery candidate replay"
else:
    description_labels_report = archived_description_labels
    desc_label = "Archived reported-only versus augmented training"
print(desc_label)
display(description_labels_report.round(2))
figure, axis = plt.subplots(figsize=(10, 7))
description_labels_report.set_index("model")[["baseline_mae", "augmented_mae"]].plot.barh(ax=axis)
axis.invert_yaxis()
axis.set(title=desc_label, xlabel="Test MAE (beds)", ylabel="")
figure.tight_layout()
plt.show()
plt.close(figure)

In [ ]:
if deep_capacity:
    listed = np.array([bool(items(_array(left) + _array(right)))
                       for left, right in width_rows[["phone_numbers", "officialPhone"]].itertuples(index=False, name=None)])
    matrix = assemble(encoded, False)
    phone_matrix = np.column_stack([matrix, listed.astype(np.float32)])
    print("Phone listed for", int(listed[width_test].sum()), "of", int(width_test.sum()), "test facilities.")
    phone_rows = []
    for label, mask, target, seeds in [
        ("with recovered labels", augmented_train, augmented_target, (42, 104, 2026)),
        ("reported labels only", width_train, width_target, (42,)),
    ]:
        for seed in seeds:
            point = {}
            for variant, values in (("without status", matrix), ("with status", phone_matrix)):
                p_cal, p_test = fit(values, "xgboost_boosted_forest", (mask, width_cal, width_test), target, seed=seed)
                point[variant] = p_test
                phone_rows.append({"cohort": label, "seed": seed, "variant": variant,
                                   "calibration_mae": mean_absolute_error(ablation_calibration, p_cal),
                                   **scores(ablation_outcomes, p_test)})
            print(label, seed, "paired MAE change [point, lower, upper]",
                  compare(ablation_outcomes, point["with status"], point["without status"]))
    phone_report = pd.DataFrame(phone_rows)
else:
    print("Archived phone-listing ablation · 455 listed and 12 unlisted test facilities")
    phone_report = archived_phone_ablation
display(phone_report.round(2))
phone_gap = phone_report.pivot(index=["cohort", "seed"], columns="variant", values="mae").reset_index()
phone_gap["label"] = phone_gap["cohort"] + " · seed " + phone_gap["seed"].astype(str)
phone_gap["delta"] = phone_gap["with status"] - phone_gap["without status"]
ml_bar(phone_gap, "label", "delta", "Phone-listing indicator effect", "MAE change with indicator (beds)")

### From source choices to a release model

### First comparison of two supplementation iterations

The initial capacity experiment froze two independently filled source datasets. It held the ten external desk-review branches out of model fitting, collapsed linked duplicate rows in model space, and excluded the implausible 200,000-bed target for Harsh Hospital. The six XGBoost arms varied the supplement and the feature tier. Each fit searched raw absolute-error vs log-square-error training and depths three and five. A separate 213 branches set interval widths. On the 456 reported test facilities, iteration 01 with both district blocks led by test MAE, 136.49 beds against the median's 160.78. Iteration 01 facility-only led validation and had better RMSE. The first provisional release used the test minimum, so 136.49 is a selection result on a reused holdout. The winner also changed after excluding the independent review cases. It underpredicted the 201+ group by about 449 beds. [Six-arm results and cohort design](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/max-supplemented-data/experiments/capacity/RESULTS.md)

In [ ]:
supplement_path = RUN / "supplementation"
first_two = [supplement_path / f"iteration-{i:02d}" for i in (1, 2)]
capacity_full = bool(globals().get("FULL_REPLAY", False))

# The published population-only CSV has no keys. Verify it against its own
# keyed upstream output and use state and district explicitly for the replay.
import csv

with (ROOT / "data/population/nfhs_population_only.csv").open(encoding="utf-8-sig", newline="") as source:
    anonymous_population = list(csv.DictReader(source))
with (ROOT / "load-balancing-pipeline/outputs/nfhs_district_population_2020.csv").open(
        encoding="utf-8-sig", newline="") as source:
    keyed_population = list(csv.DictReader(source))
assert len(anonymous_population) == len(keyed_population) == 706
paired_population = [(value["population"], key["pop_2020"])
                     for value, key in zip(anonymous_population, keyed_population, strict=True)]
for historical, upstream in paired_population:
    assert bool(historical) == bool(upstream)
    if historical:
        assert abs(float(historical) - float(upstream)) <= 1e-6
population_for_models = ml_run / "nfhs_population_keyed.csv"
with population_for_models.open("w", encoding="utf-8", newline="") as target:
    writer = csv.writer(target)
    writer.writerow(("state", "district", "population"))
    writer.writerows((key["state_ut"], key["district_name"], historical["population"])
                     for historical, key in zip(anonymous_population, keyed_population, strict=True))
print("Keyed district population estimates available:",
      sum(bool(historical) for historical, _ in paired_population), "of", len(keyed_population))

def capacity_subprocess(args):
    python = globals().get("CPU_PYTHON", sys.executable)
    return run_cmd([python, *args[1:]], env={**os.environ, "KMP_DUPLICATE_LIB_OK": "FALSE"})

def assemble_capacity(iterations, output):
    args = [sys.executable, capacity_path / "prepare_data.py", "--snapshot", ml_snapshot(),
            "--population", population_for_models, "--population-source-revision",
            "d6bb601d61882d8f42e200793cd11d629a9f7fe0", "--output-dir", output]
    for i, iteration in enumerate(iterations, 1):
        args.extend((f"--iteration-{i:02d}", iteration))
    return run_cmd(args)

def fit_capacity_arms(inputs, training, *, four_rules=False):
    if not (training / "metrics.json").is_file():
        args = [sys.executable, capacity_path / "train_evaluate.py", "--input-dir", inputs, "--output-dir", training]
        if four_rules:
            args.append("--rule-cleaning")
        run_cmd(args)
    return load_json(training / "metrics.json")

def capacity_arm_table(metrics):
    return pd.DataFrame([{"run": arm["run_id"], "feature_tier": arm["feature_set"],
                          "validation_mae": arm["validation"]["mae"], "test_mae": arm["test"]["mae"],
                          "test_rmse": arm["test"]["rmse"], "test_bias": arm["test"]["mean_bias"],
                          "coverage": arm["interval"]["coverage"]} for arm in metrics["runs"]])

two_selector_run = ml_run / "two-selector"
if capacity_full and ml_snapshot() and all((path / "dataset/facilities.jsonl").is_file() for path in first_two):
    if not (two_selector_run / "inputs/input_audit.json").is_file():
        assemble_capacity(first_two, two_selector_run / "inputs")
    two_metrics = fit_capacity_arms(two_selector_run / "inputs", two_selector_run / "training")
    if not (two_selector_run / "datasets").exists():
        run_cmd([sys.executable, capacity_path / "export_arms.py", "--input-dir", two_selector_run / "inputs",
                 "--metrics", two_selector_run / "training/metrics.json", "--output-dir", two_selector_run / "datasets"])
    display(capacity_arm_table(two_metrics).round(2))
    print("Replayed cohort", two_metrics["holdout"], "selection", two_metrics["winner"])
    two_chart = two_selector_run / "capacity-test-results.png"
    run_cmd([sys.executable, capacity_path / "plot_results.py", "--metrics",
             two_selector_run / "training/metrics.json", "--output", two_chart])
    display(Image(filename=str(two_chart)))
else:
    print("Archived two-selector chart · 456 reported test facilities")
    ml_picture("../figures/capacity-test-results.png")

### Four deterministic rules and an original-data control

The expanded experiment rebuilt the same source snapshot with four policies. Publisher ranking used the review-based order FDR, IFFCO, SBI and recency for ties. The three date-based policies selected the newest eligible report or averaged within the newest year or trailing two years. Holds and direct decisions took precedence. The experiment tested facility-only, socioeconomic, population and both district feature blocks for each rule, with a seventeenth unsupplemented FDR control. The date-based rules produced the same selected inputs. An explicit hold removed the disputed Jorhat target, giving 455 reported test cases. Cleaned arms used 1,608 training cases and the raw control used 1,417.

The team selected on 213 validation cases, leaving another 213 to calibrate intervals. Publisher-ranked plus population led validation at 129.09 beds MAE and became the intermediate release. Newest plus NFHS led the historical test at 134.99, versus 136.44 for the release model. Their paired 95% interval included zero. The original-data control had 136.41 test MAE and the lowest RMSE of 313.21. This comparison established traceable cleaning and field holds while leaving the effect on prediction error uncertain. [Rules, scores and validation selection](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/max-supplemented-data/experiments/capacity/RULE_CLEANING_COMPARISON.md)

In [ ]:
supplied_rule_path = globals().get("RULE_ARCHIVE")
rule_archive = Path(supplied_rule_path) if supplied_rule_path else ml_run / "rule-cleaning"
rule_build = globals().get("CAPACITY_RULE_BUILD")
rule_policy = {1: "ranked", 2: "latest_year_mean", 3: "newest", 4: "recent_two_year_mean"}
experiment_path = capacity_path.parents[1]

def created_rule_outputs(path):
    return all((path / f"iteration-{i:02d}/dataset/facilities.jsonl").is_file() for i in rule_policy)

if (capacity_full and not supplied_rule_path and ml_snapshot() and rule_build
        and all((path / "dataset/facilities.jsonl").is_file() for path in first_two)):
    if not created_rule_outputs(rule_archive):
        extension = first_two[-1]
        required = [extension / "sample.json", experiment_path / "review", extension / "model"]
        for item in required:
            if not item.exists():
                raise FileNotFoundError(f"Iteration 02 requires {item}.")
        rule_archive.mkdir(parents=True, exist_ok=True)
        direct_reviews = [
            item for filename in ("scrape-audit-12.json", "rule-cleaning-source-review.json")
            for item in load_json(experiment_path / "field-reviews" / filename)
        ]
        rule_reviews = rule_archive / "combined-field-reviews.json"
        rule_reviews.write_text(json.dumps(direct_reviews, ensure_ascii=False), encoding="utf-8")
        for i, policy in rule_policy.items():
            iteration = rule_archive / f"iteration-{i:02d}"
            iteration.mkdir(exist_ok=True)
            if (iteration / "dataset/facilities.jsonl").is_file():
                continue
            args = [sys.executable, experiment_path / "src/build_dataset.py",
                    "--output-dir", iteration / "dataset", "--count-policy", policy,
                    "--sample", required[0], "--reviews", required[1], "--model-dir", required[2],
                    "--field-reviews", rule_reviews]
            for flag, path in rule_build.items():
                if path and flag.lstrip("-") not in {"output-dir", "count-policy", "sample", "reviews", "model-dir", "field-reviews"}:
                    args.extend(("--" + flag.lstrip("-").replace("_", "-"), path))
            run_cmd(args)
    if not (rule_archive / "inputs/input_audit.json").is_file():
        assemble_capacity([rule_archive / f"iteration-{i:02d}" for i in rule_policy], rule_archive / "inputs")
    fit_capacity_arms(rule_archive / "inputs", rule_archive / "training", four_rules=True)
    if not (rule_archive / "release/facilities_with_verification.csv").is_file():
        run_cmd([sys.executable, capacity_path / "publish_rule_comparison.py",
                 "--run", rule_archive, "--contact-evidence", ml_snapshot() / "contact_evidence.jsonl"])

archived_rule_metrics_path = rule_archive / "training/metrics.json"
if supplied_rule_path and not archived_rule_metrics_path.is_file():
    raise FileNotFoundError(f"An authorized rule archive needs {archived_rule_metrics_path}.")

In [ ]:
rule_history = pd.DataFrame([{'run': 'iteration_01__facility_only',
  'feature_tier': 'facility_only',
  'validation_mae': 129.76,
  'test_mae': 138.65,
  'test_rmse': 319.19,
  'test_bias': -90.94,
  'coverage': 0.8989},
 {'run': 'iteration_01__socioeconomic',
  'feature_tier': 'socioeconomic',
  'validation_mae': 132.91,
  'test_mae': 137.37,
  'test_rmse': 323.08,
  'test_bias': -95.2,
  'coverage': 0.8967},
 {'run': 'iteration_01__socioeconomic_population',
  'feature_tier': 'socioeconomic_population',
  'validation_mae': 131.64,
  'test_mae': 139.04,
  'test_rmse': 325.81,
  'test_bias': -95.67,
  'coverage': 0.8989},
 {'run': 'iteration_01__population',
  'feature_tier': 'population',
  'validation_mae': 129.09,
  'test_mae': 136.44,
  'test_rmse': 316.89,
  'test_bias': -91.41,
  'coverage': 0.9055},
 {'run': 'iteration_02__facility_only',
  'feature_tier': 'facility_only',
  'validation_mae': 129.58,
  'test_mae': 136.69,
  'test_rmse': 319.28,
  'test_bias': -93.48,
  'coverage': 0.8835},
 {'run': 'iteration_02__socioeconomic',
  'feature_tier': 'socioeconomic',
  'validation_mae': 131.41,
  'test_mae': 134.99,
  'test_rmse': 321.35,
  'test_bias': -95.23,
  'coverage': 0.8967},
 {'run': 'iteration_02__socioeconomic_population',
  'feature_tier': 'socioeconomic_population',
  'validation_mae': 132.3,
  'test_mae': 136.69,
  'test_rmse': 326.39,
  'test_bias': -98.69,
  'coverage': 0.8945},
 {'run': 'iteration_02__population',
  'feature_tier': 'population',
  'validation_mae': 130.38,
  'test_mae': 137.47,
  'test_rmse': 321.12,
  'test_bias': -93.95,
  'coverage': 0.8967},
 {'run': 'iteration_03__facility_only',
  'feature_tier': 'facility_only',
  'validation_mae': 129.58,
  'test_mae': 136.69,
  'test_rmse': 319.28,
  'test_bias': -93.48,
  'coverage': 0.8835},
 {'run': 'iteration_03__socioeconomic',
  'feature_tier': 'socioeconomic',
  'validation_mae': 131.41,
  'test_mae': 134.99,
  'test_rmse': 321.35,
  'test_bias': -95.23,
  'coverage': 0.8967},
 {'run': 'iteration_03__socioeconomic_population',
  'feature_tier': 'socioeconomic_population',
  'validation_mae': 132.3,
  'test_mae': 136.69,
  'test_rmse': 326.39,
  'test_bias': -98.69,
  'coverage': 0.8945},
 {'run': 'iteration_03__population',
  'feature_tier': 'population',
  'validation_mae': 130.38,
  'test_mae': 137.47,
  'test_rmse': 321.12,
  'test_bias': -93.95,
  'coverage': 0.8967},
 {'run': 'iteration_04__facility_only',
  'feature_tier': 'facility_only',
  'validation_mae': 129.58,
  'test_mae': 136.69,
  'test_rmse': 319.28,
  'test_bias': -93.48,
  'coverage': 0.8835},
 {'run': 'iteration_04__socioeconomic',
  'feature_tier': 'socioeconomic',
  'validation_mae': 131.41,
  'test_mae': 134.99,
  'test_rmse': 321.35,
  'test_bias': -95.23,
  'coverage': 0.8967},
 {'run': 'iteration_04__socioeconomic_population',
  'feature_tier': 'socioeconomic_population',
  'validation_mae': 132.3,
  'test_mae': 136.69,
  'test_rmse': 326.39,
  'test_bias': -98.69,
  'coverage': 0.8945},
 {'run': 'iteration_04__population',
  'feature_tier': 'population',
  'validation_mae': 130.38,
  'test_mae': 137.47,
  'test_rmse': 321.12,
  'test_bias': -93.95,
  'coverage': 0.8967},
 {'run': 'raw_fdr__facility_only',
  'feature_tier': 'facility_only',
  'validation_mae': 129.97,
  'test_mae': 136.41,
  'test_rmse': 313.21,
  'test_bias': -86.05,
  'coverage': 0.9055}])

In [ ]:
rule_metrics = load_json(archived_rule_metrics_path) if archived_rule_metrics_path.is_file() else None
rule_table = capacity_arm_table(rule_metrics) if rule_metrics else rule_history
rule_source = "Replayed or supplied run" if rule_metrics else "Archived November 21 run · 455 branches"
print(rule_source)
if rule_metrics:
    print("Selection", rule_metrics["winner"])
display(rule_table.round(2))
ml_bar(rule_table.sort_values("test_mae"), "run", "test_mae",
       "Four rule experiment and FDR control", "Test MAE (beds)")

### Feedforward blend of source-rule predictions

The blend treated the three identical date-based arms as one signal. It trained five district-grouped folds of each rule to generate 1,284 out-of-fold predictions and learned a fixed blend and a small softmax network on a district-grouped development split. The fixed blend selected on development scored 136.81 beds MAE. The feedforward blend scored 136.96. Publisher ranking at the same population tier scored 136.44. Paired intervals included zero for both blends. The team retained the publisher-ranked intermediate release. The captured W&B view preserves the original run and its artifacts. [Feedforward report, archived result and artifact links](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/max-supplemented-data/experiments/capacity/FEEDFORWARD_EVALUATION.md)

In [ ]:
if capacity_full and rule_metrics:
    feedforward_notebook = ml_path / "notebooks/Source Rule Feedforward Ensemble.ipynb"
    feedforward_runner = ml_run / "replay_feedforward.py"
    feedforward_document = load_json(feedforward_notebook)
    # A clean process keeps Torch import isolated from the rest of the model fits.
    feedforward_runner.write_text(
        "import os\nfrom pathlib import Path\n"
        + f"os.environ['DATATHON_RULE_ENSEMBLE_REPO'] = {str(ROOT)!r}\n"
        + f"os.environ['DATATHON_RULE_ENSEMBLE_RUN'] = {str(rule_archive)!r}\n"
        + "def display(frame): print(frame.to_string(index=False))\n"
        + "\n\n".join("".join(cell["source"]) for cell in feedforward_document["cells"]
                        if cell["cell_type"] == "code")
        + f"\npd.DataFrame(results).to_csv({str(ml_run / 'feedforward-test.csv')!r}, index=False)\n",
        encoding="utf-8")
    run_cmd([sys.executable, feedforward_runner], env={**os.environ, "KMP_DUPLICATE_LIB_OK": "FALSE"})
    replayed_blend = pd.read_csv(ml_run / "feedforward-test.csv")
    display(replayed_blend)
    ml_bar(replayed_blend, "method", "mae", "Replayed population-tier ensemble", "Test MAE (beds)")
else:
    ml_picture("assets/feedforward-decision/heldout.jpg")
print("Archived W&B artifact view from the original run")
ml_picture("assets/feedforward-decision/wandb-artifacts.jpg")

### Source rule × feature factorial and disputed-label sensitivity

The follow-up isolated the effects of source policy, district features and disputed training labels. It reran each arm on the same 455 reported test labels, adding a strict comparison that withheld source-disputed training targets and a disputed validation record. Comparisons combine paired errors at the district or single-facility level, so the effects are estimated on matched rows. The stricter pass removed 131 usable training labels from each treated arm. The observed test leader switched from newest plus NFHS at 134.99 to newest plus population at 136.59 beds. The validation-selected strict arm was ranked plus NFHS at 130.58 beds validation MAE. The comparison retained the intermediate released ranked plus population model. The report preserves test geography coverage and subgroup errors. [Factorial design, estimates and decision](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/max-supplemented-data/experiments/capacity/SOURCE_FEATURE_FACTORIAL.md)

In [ ]:
factorial_run = ml_run / "source-feature-factorial"
if capacity_full and rule_metrics:
    if not (factorial_run / "aggregate-results.json").is_file():
        capacity_subprocess([sys.executable, capacity_path / "source_feature_factorial.py",
                 "--archive", rule_archive, "--output", factorial_run])
    factorial_result = load_json(factorial_run / "aggregate-results.json")
    for policy, fields in factorial_result["results"].items():
        print(policy)
        display(pd.DataFrame(fields["arms"]).round(2))
    factorial_plots = factorial_run / "plots"
    run_cmd([sys.executable, capacity_path / "plot_source_feature_factorial.py", "--results",
             factorial_run / "aggregate-results.json", "--output", factorial_plots])
    for figure_path in sorted(factorial_plots.glob("*.png")):
        display(Image(filename=str(figure_path)))
else:
    for file in ("factorial-test-mae.png", "paired-changes.png", "errors-by-facility-size.png"):
        ml_picture(f"assets/source-feature-factorial/{file}")

### Corrected description pass and full-width release fit

The next pass screened descriptions for branch-specific whole-facility counts and evidence of service. After adding explicit historical and third-party scope guards, it recovered 13 service cells and zero new capacity labels for training. All four arms refit with 67 inputs before encoding. The 49 source fields exclude the bed target and the label-source description. Phone fields are reduced to listing flags. The additional fields use audited geography, documented services, socioeconomic indicators and district population.

The newest-source fit led validation at 112.49 beds MAE and test at 110.22 on 455 reported facilities. The publisher-ranked fit scored 114.33 and 112.96, with lower test RMSE and smaller large-facility misses. The three date-based arms were identical. The paired newest-minus-ranked MAE interval was −8.92 to +2.94 and the reused test cannot isolate how much of the improvement came from features, source cleaning or tuning. The released newest model was selected on validation. Nominal 90% intervals covered 89.9% of reported tests and had median width 353 beds. Test MAE was 364.55 for facilities above 200 beds. [Corrected description recovery, feature schema and result](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/max-supplemented-data/experiments/capacity/DESCRIPTION_RECOVERY_EVALUATION.md)

In [ ]:
description_wide_run = ml_run / "description-wide"
if capacity_full and rule_metrics and created_rule_outputs(rule_archive):
    if not (description_wide_run / "metrics.json").is_file():
        capacity_subprocess([sys.executable, capacity_path / "train_description_recovery.py",
                 "--archive", rule_archive, "--output", description_wide_run, "--as-of", "2026-11-22"])
    replayed_wide = load_json(description_wide_run / "metrics.json")
    display(pd.DataFrame([
        {"rule": rule, "validation_mae": arm["validation"]["mae"], **arm["test"],
         "new_training_bed_labels": arm["new_training_bed_labels"]}
        for rule, arm in replayed_wide["runs"].items()
    ]).round(2))
    wide_plots = description_wide_run / "plots"
    run_cmd([sys.executable, capacity_path / "report_description_recovery.py", "--results",
             description_wide_run / "metrics.json", "--plots", wide_plots])
    for figure_path in sorted(wide_plots.glob("*.png")):
        display(Image(filename=str(figure_path)))
else:
    if rule_metrics and not created_rule_outputs(rule_archive):
        print("Full-width replay needs the four archived iteration datasets as well as the saved training files.")
    for file in ("rule-scores.png", "validation-learning-curves.png", "error-by-facility-slice.png"):
        ml_picture(f"assets/description-recovery-wide/{file}")

## 04 · Capacity publication and confidence

The November 22 tagged publication selected the newest suitable dated branch-wide count, with direct decisions and holds taking precedence. On the separate wide description-feature experiment’s frozen cohort, the selected XGBoost model (depth 5, 111 boosting rounds, log target) reproduced 455 held-out predictions and scored 110.22 beds MAE. It supplied estimates only to routable facilities whose original and cleaned count were both missing. Estimates used a ±278 bed interval calibrated on 213 cases, clipped to zero. Coverage was 89.9% on historically reported test counts. Larger hospitals were harder to predict and missing labels may differ.

The final two CSVs preserve 10,088 original row positions. They contain 2,713 selected documentary counts, 6,833 estimates across 6,829 distinct facilities, and 542 blanks (503 identity holds, 26 unresolved reviews, one target-quality hold and 12 explicit zeros left unfilled). The Harsh Hospital value of 200,000 referred to patients. The original faulty value remains auditable.

The companion includes the release receipt, booster and encoder. The exact release CSV is retrieved from the Git object at `ff7dd5e`, byte-for-byte with the tagged release asset. Both audited branch tips have a different version of the checked-in CSV. Use the receipt-matching bytes for publication checks and the network caseload build. Use the corresponding branch’s snapshot for app code audit.

[Release evaluation, file format and exact replay](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/max-supplemented-data/experiments/capacity/NEWEST_CAPACITY_RELEASE.md) · [Tagged release assets](https://github.com/augustchu/datathon-F26/releases/tag/capacity-newest-2026-11-22)

In [ ]:
published = EXT["caseload_release"]
tagged_confidence = RELEASE / "facility_confidence.csv"
receipt_file = RELEASE / "delivery_receipt.json"
receipt = load_json(receipt_file) if receipt_file.is_file() else None

tips = {"tagged release via ff7dd5e": published,
        "main": ROOT / caseload_rel, "restored app": APP_ROOT / caseload_rel}
release_hashes = {label: digest(path) for label, path in tips.items() if path.is_file()}
display(pd.Series(release_hashes, name="SHA-256").rename_axis("CSV variant").reset_index())

if receipt:
    for name, path in (("facilities_newest_capacity.csv", published), ("facility_confidence.csv", tagged_confidence)):
        if path.is_file():
            assert digest(path) == receipt["files_sha256"][name], f"Byte check failed for {name}"
    for name, key in (("newest.ubj", "model_sha256"), ("newest-feature-encodings.json", "encoder_manifest_sha256")):
        if (RELEASE / name).is_file():
            assert digest(RELEASE / name) == receipt["model"][key], f"Byte check failed for {name}"

if published.is_file():
    state_fields = ["branch", "phone", "services", "beds", "doctors"]
    evidence_columns = [name + "_evidence_status" for name in state_fields]
    release = read_csv(published, usecols=["row_index", "facility_id", "unique_id", "capacity", "capacity_status",
                                        *evidence_columns])
    assert release["row_index"].tolist() == list(range(10088))
    credits = {"verified_unchanged": 20, "verified_corrected": 15, "verified_filled": 15,
               "corrected_unverified": 5, "populated_unverified": 5, "model_estimate": 0, "empty": 0}
    reconstructed = release[evidence_columns].apply(lambda column: column.map(credits))
    assert reconstructed.notna().all().all()
    confidence_scores = reconstructed.sum(axis=1)
    if tagged_confidence.is_file():
        companion = read_csv(tagged_confidence)
        assert list(companion.columns) == ["confidence_score"]
        assert companion["confidence_score"].equals(confidence_scores)
    if SRC["facilities"] is not None:
        original_ids = project_frame(SRC["facilities"])["unique_id"].astype(str).reset_index(drop=True)
        print("Workspace export matches the tagged release source-row order:",
              original_ids.equals(release["unique_id"].astype(str)))
    status = release["capacity_status"]
    estimated = status.eq("xgboost_estimate")
    assert estimated.sum() == 6833 and release.loc[estimated, "facility_id"].nunique() == 6829
    assert release["capacity"].isna().sum() == 542
    assert release.loc[estimated, "beds_evidence_status"].eq("model_estimate").all()
    chart_series(status.value_counts().sort_values(), "Tagged release capacity outcomes", "Original source rows")
    chart_series(confidence_scores.value_counts().sort_index(),
                 "Tagged release confidence score distribution", "Original source rows", color="#456a99")
    display(release.loc[release["capacity"].isna(), "capacity_status"].value_counts().rename("withheld_rows"))
else:
    print("Exact publication CSV unavailable. The read-only Git checkout needs history through ff7dd5e.")

**Decision.** The released five-cell confidence score summarizes branch/name/PIN, phone, service, beds and doctors on a fixed 0–100 scale. A verified unchanged cell earns 20 points, a verified correction or fill earns 15, an unverified populated/corrected cell earns five, and an estimate, hold or blank earns zero. The app later stopped using that record-wide score as a proxy for confidence in the requested service.

The strict publisher checks the training encoder, description-evidence ledger, archived test predictions and prior phone comparison before writing. The historical deliverable is reproduced only if those artifacts are present and aligned. A fresh model fit can regenerate the procedure on a distinct cohort and is evaluated as a new result. Set `capacity_archive` to the original private rule-cleaning archive if it is available, and optionally place its sibling `description-wide-v3` alongside it.

In [ ]:
rule_input = globals().get("rule_archive") or RULE_ARCHIVE
trained_input = globals().get("description_wide_run")
frozen_fit = RULE_ARCHIVE.parent / "description-wide-v3" if RULE_ARCHIVE else None
release_candidates = [path for path in (frozen_fit, trained_input) if isinstance(path, Path) and
                      all((path / filename).is_file() for filename in (
                          "newest.ubj", "metrics.json", "newest-description-evidence.jsonl",
                          "newest-feature-encodings.json", "test-predictions.csv"))]
if (FULL_REPLAY and isinstance(rule_input, Path) and release_candidates and
        (rule_input / "release/facilities_with_verification.csv").is_file()):
    trial = Path(tempfile.mkdtemp(prefix="publish-check-", dir=RUN))
    publisher = ROOT / "max-supplemented-data/experiments/capacity/publish_newest_capacity.py"
    replay = run_cmd([CPU_PYTHON, publisher, "--archive", rule_input, "--run", release_candidates[0],
                      "--audit-dir", trial / "audit", "--output", trial / "csv", "--as-of", "2026-11-22"],
                     cwd=ROOT, optional=True,
                     env={**os.environ, "KMP_DUPLICATE_LIB_OK": "FALSE", "OMP_NUM_THREADS": "2"})
    if replay.returncode == 0:
        replay_receipt = load_json(trial / "audit/delivery_receipt.json")
        regenerated_capacity_release = trial / "csv"
        display(pd.DataFrame({"replay": replay_receipt["files_sha256"],
                              "historical": receipt["files_sha256"] if receipt else {}}))
else:
    print("Publisher replay requires full mode and the trained model, evidence and verified phone artifacts.")

## 05 · Evidence-first lookup and decision audits

The research lookup had its own sequence of checkpoints alongside the dashboard. It preserved raw fields, linked facility branches, audited coordinates, and attached dated, quoted service claims and assessed quality where scope matched. One lookup compared confirmed evidence, a statistical count overlay and an ML count overlay. All three used the same service eligibility gate. A model-only service lead could be queued for follow-up. The dashboard evolved separately, so the research lookup replay below has its own evidence date and ranking behavior.

The decision to hold suspect points followed case reviews. The November 21 snapshot screened 41,489 linked branches and held 839 additional points, including Faridkot's Deep Blood Centre, whose recorded point had put it near Delhi. With the reviewed holds, the Delhi blood-bank candidate pool fell from 241 to 200. Jaipur's tested top blood banks and dialysis branch kept their order. India Post coordinates provide area checks and origin approximations. Branch points require branch evidence. [Coordinate and address audit](https://github.com/augustchu/datathon-F26/blob/af89658/non-ml/LOCATION_AUDIT.md)

The pilot checked public ratings in four Jaipur and Delhi service searches. Zero candidates had a comparable exact-branch, relevant-service, dated patient cohort. Selected testimonials and hospital-wide ratings stayed out of ranking. The lookup carried any qualifying assessments separately and could display source or review-search links. A caller's simulated “No” suppressed a branch for the current request across modes and could yield alternatives. Sending a report to private SQLite moderation required a separate step. [Patient rating pilot](https://github.com/augustchu/datathon-F26/blob/af89658/non-ml/PATIENT_REVIEW_PILOT.md) · [Facility assessment method](https://github.com/augustchu/datathon-F26/blob/af89658/non-ml/QUALITY_METHOD.md) · [Lookup and moderation workflow](https://github.com/augustchu/datathon-F26/blob/af89658/non-ml/README.md)

In [ ]:
from pathlib import Path
from unittest.mock import patch
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

side_workspace = RUN / 'modular'
side_src = ROOT / 'non-ml' / 'src'
if str(side_src) not in sys.path:
    sys.path.insert(0, str(side_src))
from facility_tool.workspace import Store, read_config

side_config = read_config(ROOT)
side_store = Store(ROOT, 'export', side_config['as_of'], output_dir=side_workspace)
side_mode = 'export'

def run_offline_lookup():
    # Run the original call-feedback notebook with its API toggle set to offline.
    notebook = ROOT / 'non-ml/notebooks/07_compare_and_feedback.ipynb'
    environment = {
        'DATATHON_REPO_ROOT': str(ROOT), 'DATATHON_OUTPUT_DIR': str(side_workspace),
        'DATATHON_MODE': side_mode, 'DATATHON_LOG_COUNT_MODELS': '0',
    }
    namespace = {}
    with patch.dict(os.environ, environment):
        for n, cell in enumerate(load_json(notebook)['cells']):
            if cell['cell_type'] != 'code':
                continue
            source = ''.join(cell['source']).replace(
                "USE_TRAVELTIME = config['traveltime']['enabled']", 'USE_TRAVELTIME = False',
            )
            source = source.replace(
                "if MODE == 'demo' and not eligible.empty:", 'if not eligible.empty:',
            ).replace(
                'Simulating a No on the first fictional facility.',
                'Simulating a No on the first listed branch. No facility was contacted.',
            )
            exec(compile(source, f'{notebook.name} cell {n + 1}', 'exec'), namespace)
    return namespace

print(f'Lookup replay input mode: {side_mode}. Checkpoints: {side_store.path}')

In [ ]:
# Summarize the source and placement checkpoints created by the earlier replay.
side_counts = side_store.read('source_counts', optional=True)
if not side_counts.empty:
    side_counts = side_counts.sort_values('rows_loaded')
    side_counts['rows_loaded'] = pd.to_numeric(side_counts['rows_loaded'])
    ax = side_counts.plot.barh(x='source', y='rows_loaded', legend=False, color='#397b7b',
                               title=f'Input records by source ({side_mode})')
    ax.set_xlabel('Source records before branch linkage')
    plt.tight_layout()
    plt.show()
else:
    print('The row-level lookup replay needs the checked organizer inputs.')
side_holds = side_store.read('location_entity_review', optional=True)
hold_column = 'location_routing_eligible'
if hold_column in side_holds:
    ax = side_holds[hold_column].map({True: 'Eligible', False: 'Held'}).value_counts().plot.bar(
        color='#397b7b', title='Branch location disposition'
    )
    ax.set_ylabel('Linked branches')
    ax.tick_params(axis='x', rotation=0)
    plt.tight_layout()
    plt.show()

### Count controls and preparation

The research count comparison authorizes predictors from training facilities, uses repeated grouped splits and an independent calibration partition, compares median, log-ridge and random forest, and assesses model support before exporting an overlay. Notebook 03 screens exact count labels, ambiguity, reporting gaps and predictors after the challenge schema's `countries` field. The separate model chapter covers the dashboard-oriented population, source and neural experiments. Here the count-model overlays are private scratch outputs of the research lookup. [Count EDA and evaluation design](https://github.com/augustchu/datathon-F26/blob/af89658/non-ml/PROJECT_SUMMARY.md)

In [ ]:
research_count_report = {
    'Beds': ('capacity', [150.57, 141.94, 141.41]),
    'Doctors': ('numberdoctors', [28.82, 27.83, 27.63]),
}
count_method_labels = {'median': 'Training median', 'regression': 'Log-ridge', 'ml': 'Random forest'}
for unit, (target, archived_mae) in research_count_report.items():
    metrics = side_store.read(f'count_models_{target}_metrics', optional=True)
    if metrics.empty:
        primary = pd.DataFrame({'method': list(count_method_labels), 'mae': archived_mae})
        metric_basis = 'archived project report'
    else:
        primary = metrics.loc[metrics['run'].eq('all_exact_labels'), ['method', 'mae']].copy()
        primary['mae'] = pd.to_numeric(primary['mae'])
        metric_basis = 'recomputed holdout'
    primary['method'] = primary['method'].map(count_method_labels)
    ax = primary.plot.bar(x='method', y='mae', legend=False, color='#397b7b',
                          title=f'{unit} mean absolute error ({metric_basis})')
    ax.set_ylabel(unit)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)
    plt.tight_layout()
    plt.show()

In [ ]:
# Run district context, produce stable lookup assets and compare searches offline.
if (side_store.path / 'field_updates.jsonl').is_file():
    for stage in ('06_district_context', '06b_prepare_lookup'):
        replay_repo_notebook(stage)
    side_lookup = run_offline_lookup()
    replay_repo_notebook('08_load_policy_scenarios')
else:
    print('Workspace lookup checkpoints are unavailable. Showing the recorded ratings pilot.')
side_pilot = pd.DataFrame({
    'query': ['Jaipur dialysis', 'Delhi dialysis', 'Jaipur blood bank', 'Delhi blood bank'],
    'eligible': [1, 1, 43, 2],
    'comparable ratings': [0, 0, 0, 0],
})
ax = side_pilot.set_index('query').plot.barh(color=['#397b7b', '#d27f47'],
                                          title='Recorded patient experience pilot')
ax.set_xlabel('Candidates per search')
plt.tight_layout()
plt.show()

### Live routing and tests

The live TravelTime check is separately opt-in and uses the credentials configured through the original runner. It exercises Jaipur driving and Gadchiroli driving and walking, with broad eligible-destination submission, route-derived ordering, and car-only alternative labels. The runner can build blood-bank lookup assets from the authorized Databricks catalog. This check consumes API quota and current responses can differ from the frozen app benchmark. The frozen benchmark is reproduced in the app chapter. [TravelTime notebook and runner](https://github.com/augustchu/datathon-F26/blob/af89658/non-ml/check_traveltime_live.py)

In [ ]:
from runpy import run_path

RUN_LIVE_TRAVELTIME = False
if RUN_LIVE_TRAVELTIME:
    if 'spark' not in globals() or 'dbutils' not in globals():
        raise RuntimeError('Attach this notebook to Databricks compute for the live test.')
    with patch.dict(os.environ, {'DATATHON_REPO_ROOT': str(ROOT),
                                 'DATATHON_OUTPUT_DIR': str(side_workspace)}):
        run_path(str(ROOT / 'non-ml/check_traveltime_live.py'))['run_databricks'](spark, dbutils)

for tests in ('non-ml/tests', 'non-ml/caseload'):
    run_cmd([
        sys.executable, '-c',
        'import sys,unittest; sys.path.insert(0, sys.argv[1]); '
        'suite = unittest.defaultTestLoader.discover(sys.argv[2]); '
        'sys.exit(not unittest.TextTestRunner(verbosity=1).run(suite).wasSuccessful())',
        str(side_src), tests,
    ], cwd=ROOT, optional=True)

## 06 · Side projects

### Population enrichment and early load simulator

The prototype built one point per PIN/state/district combination, assigned geoBoundaries ADM2 polygons to states from the PIN votes, and proposed district matches within state. Sparse and split votes went to review. Finalization used the reviewed crosswalk, summed the 2020 constrained WorldPop raster, scaled aggregate counts to published state projections and preserved unresolved polygons and district exclusions. Suppressed NFHS cells stayed null and parenthesized percentages carried low-confidence flags. Census 2011 name matches were candidates for a boundary-vintage check. District splits and changed names can invalidate comparisons across years. [Population pipeline](https://github.com/augustchu/datathon-F26/blob/af89658/load-balancing-pipeline/README.md) · [WorldPop source directory](https://data.worldpop.org/GIS/Population/Global_2000_2020_Constrained/2020/BSGM/IND/)

The simulator sampled synthetic districts with weights proportional to modeled population times relative NFHS burden. It assigned cases sequentially using service text, a travel measure and a soft capacity/load penalty for nonemergencies. Emergencies used the closest candidate over the evidence threshold. The initial approach demonstrated a routing policy with a declared proxy. Later planning experiments required workload and throughput in a comparable service scope, and the dashboard separated historical simulations from current availability. [Simulator assumptions](https://github.com/augustchu/datathon-F26/blob/af89658/load-balancing-pipeline/SIMULATOR.md)

In [ ]:
# Regenerate spatial proposals and finalize against the frozen reviewed crosswalk when inputs are available.
import shutil

population_source = ROOT / 'load-balancing-pipeline'
population_saved = population_source / 'outputs'
population_work = RUN / 'population'
population_work.mkdir(exist_ok=True)
(population_work / 'outputs').mkdir(exist_ok=True)

rebuild_population = FULL_REPLAY and bool(
    EXT.get('worldpop') and Path(EXT['worldpop']).is_file() and SRC.get('pin') and SRC.get('nfhs')
)
population_outputs = population_saved
if rebuild_population:
    shutil.copyfile(population_source / 'population_pipeline.py', population_work / 'population_pipeline.py')
    for key, filename in (('nfhs', 'NFHS-5 Dataset.csv'), ('pin', 'india_post_pincode_directory.csv')):
        project_frame(SRC[key]).to_csv(population_work / filename, index=False)
    for key, filename in (('census', 'india-districts-census-2011.csv'),
                          ('state_population', 'india_state_population_2020.csv')):
        source = Path(EXT.get(key) or (population_source / filename))
        shutil.copyfile(source, population_work / filename)
    spatial_boundaries = Path(EXT.get('boundaries') or (population_saved / 'geoBoundaries-IND-ADM2.geojson'))
    if spatial_boundaries.suffix == '.zip':
        shutil.copyfile(spatial_boundaries, population_work / 'geoBoundaries-IND-ADM2-all.zip')
    else:
        shutil.copyfile(spatial_boundaries, population_work / 'outputs/geoBoundaries-IND-ADM2.geojson')
    raster_link = population_work / 'ind_ppp_2020_UNadj_constrained.tif'
    if not raster_link.exists():
        try:
            raster_link.symlink_to(Path(EXT['worldpop']).resolve())
        except OSError:
            shutil.copyfile(EXT['worldpop'], raster_link)
    prepare_result = run_cmd([sys.executable, 'population_pipeline.py', 'prepare'],
                             cwd=population_work, optional=True)
    if prepare_result is not None and prepare_result.returncode == 0:
        proposals = population_work / 'outputs/district_crosswalk.csv'
        shutil.copyfile(proposals, population_work / 'outputs/district_crosswalk_proposals.csv')
        shutil.copyfile(population_saved / proposals.name, proposals)
        finalize_result = run_cmd([sys.executable, 'population_pipeline.py', 'finalize'],
                                  cwd=population_work, optional=True)
        if finalize_result is not None and finalize_result.returncode == 0:
            population_outputs = population_work / 'outputs'
print('Population evidence:', 'recomputed with the pinned review decisions'
      if population_outputs != population_saved else 'published aggregates')

In [ ]:
district_population = read_csv(population_outputs / 'nfhs_district_population_2020.csv')
scales = read_csv(population_outputs / 'state_scaling_diagnostics.csv')
census_candidates = read_csv(population_outputs / 'census_2011_validation_candidates.csv')
population_status = district_population['district_population_status'].value_counts()
display(population_status.rename('districts').to_frame())
display(scales[['state_scale_key', 'state_scale_factor', 'coverage_flag']].head(10))
display(census_candidates[['district_name', 'state_ut', 'validation_note']].head(8))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
population_status.sort_values().plot.barh(ax=axes[0], color='#397b7b', title='District coverage')
district_population.groupby('state_ut')['pop_2020'].sum(min_count=1).nlargest(10).sort_values().div(1e6).plot.barh(
    ax=axes[1], color='#397b7b', title='Largest mapped populations by state'
)
axes[0].set_xlabel('Districts')
axes[1].set_xlabel('Millions of people in mapped districts')
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
from importlib.util import find_spec

if find_spec("geopandas") is not None:
    # Run the prototype with seeded synthetic demand and a straight-line time proxy.
    if str(population_source) not in sys.path:
        sys.path.insert(0, str(population_source))
    from facility_mapping import assign_facility_regions, build_region_geometries, normalize_facilities
    from simulation import run_issue_scenario
    from traveltime_adapter import straight_line_minutes

    simulation_boundary = population_outputs / 'geoBoundaries-IND-ADM2.geojson'
    simulation_crosswalk = population_outputs / 'resolved_polygon_crosswalk.csv'
    simulation_geometry = build_region_geometries(simulation_boundary, simulation_crosswalk)
    simulation_districts = district_population.loc[district_population['pop_2020'].gt(0)].copy()
    simulation_nfhs = project_frame(SRC['nfhs']) if SRC.get('nfhs') else read_csv(population_source / 'NFHS-5 Dataset.csv')
    simulation_districts = simulation_districts.merge(
        simulation_nfhs, on=['district_name', 'state_ut'], how='left', validate='one_to_one',
        suffixes=('', '_nfhs'),
    )
    simulation_facilities_raw = (project_frame(SRC['facilities']) if SRC.get('facilities')
                                 else read_csv(population_source / 'facilities dataset.csv'))
    import json
    import numpy as np

    service_columns = [column for column in ('capability', 'procedure', 'specialties', 'equipment', 'description')
                       if column in simulation_facilities_raw]
    # The original prototype takes CSV strings. Preserve native arrays from Catalog and Parquet.
    for column in (*service_columns, 'source_urls'):
        simulation_facilities_raw[column] = simulation_facilities_raw[column].map(
            lambda value: json.dumps(list(value) if isinstance(value, np.ndarray) else value, default=str)
            if isinstance(value, (tuple, list, dict, np.ndarray)) else value
        )
    simulation_facilities = normalize_facilities(
        simulation_facilities_raw, 'name', 'latitude', 'longitude', service_columns,
        capacity_col='capacity', source_col='source_urls', facility_type_col='facilityTypeId',
        id_col='unique_id', city_col='address_city', state_col='address_stateOrRegion',
        postcode_col='address_zipOrPostcode',
    )
    simulation_facilities = assign_facility_regions(simulation_facilities, simulation_boundary, simulation_crosswalk)
    simulation_facilities = simulation_facilities.drop_duplicates('facility_id')
    def simulator_offline_travel(origin, facilities):
        return straight_line_minutes(origin, facilities), 'straight-line speed proxy'

    cases, assignments, loads = run_issue_scenario(
        simulation_districts, simulation_geometry, simulation_facilities, 'dialysis_need', 30,
        simulator_offline_travel, seed=7,
    )
    display(assignments[['case_id', 'state_ut', 'severity', 'facility_id', 'assignment_status']].head(10))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    cases['state_ut'].value_counts().head(10).sort_values().plot.barh(
        ax=axes[0], color='#397b7b', title='Seeded scenario demand by state'
    )
    loads.set_index('facility_id')['simulated_load_units'].nlargest(10).sort_values().plot.barh(
        ax=axes[1], color='#d27f47', title='Assigned resource units by facility ID'
    )
    axes[0].set_xlabel('Synthetic cases')
    axes[1].set_xlabel('Simulated units')
    plt.tight_layout()
    plt.show()
else:
    print("Install geospatial dependencies to rerun the seeded population simulator.")

### Caseload forecasts, throughput and load allocation

The team moved from a broad synthetic simulator to dated service activity with units and branch linkage. The typical-month estimator uses a rolling median for at least three usable months, a reported closed-period total divided by the covered months, or an explicitly published mean. Missing months stay missing and published zeros stay zero. It backtests rolling median against last observed using only earlier periods. The site-level evidence is a transcribed, cited snapshot. It is independent of the availability of the original report URLs. [Activity manifests and median specification](https://github.com/augustchu/datathon-F26/blob/af89658/non-ml/caseload/README.md)

A held-district and forward-period experiment tested XGBoost on 47 Himachal dialysis site-years and 94 Tamil Nadu maternity centres. The resource-scaled medians beat XGBoost on held-out districts, and a prior-year site average won the later known-site dialysis test. The project therefore used simple supported baselines, with five historical dialysis fills. Expanded replays preserved the reporting unit, service scope, period and independent flags for predicted caseload and predicted throughput. The public appointment audit found zero usable slot rows. The routine measured-availability interface needs a fresh provider export with like-for-like coverage windows. [XGBoost experiment](https://github.com/augustchu/datathon-F26/blob/af89658/non-ml/caseload/XGBOOST.md) · [Baseline coverage](https://github.com/augustchu/datathon-F26/blob/af89658/non-ml/caseload/BASELINES.md) · [Routine appointments audit](https://github.com/augustchu/datathon-F26/blob/af89658/non-ml/caseload/ROUTINE_APPOINTMENTS.md)

**Reproduction detail.** The caseload baseline, forecast builder and replays require the facility CSV with SHA-256 beginning `0aecdb730`. The file in commit `ff7dd5e` matches that digest. The same path at shared main `af89658` has different bytes (SHA-256 beginning `3aa26aae`). The code below checks the original file against the receipt and uses it when available. Otherwise the hash-gated experiments display their archived results. [Pinned caseload input](https://github.com/augustchu/datathon-F26/blob/ff7dd5e/max-supplemented-data/capacity_and_confidence/facilities_newest_capacity.csv) · [Release fingerprint](https://github.com/augustchu/datathon-F26/blob/af89658/non-ml/caseload/baseline_coverage.json)

In [ ]:
from hashlib import sha256

case_src = ROOT / 'non-ml' / 'caseload'
case_work = RUN / 'caseload'
case_work.mkdir(exist_ok=True)
case_pinned = Path(EXT['caseload_release']) if EXT.get('caseload_release') else None
case_expected_sha = load_json(case_src / 'baseline_coverage.json')['dataset']['sha256']
case_release_ok = bool(case_pinned and case_pinned.is_file()
                       and digest(case_pinned) == case_expected_sha)
print('Verified original caseload release bytes:', case_release_ok)

# Replay each historical script against the newly written upstream receipts.
# The pinned research tree remains untouched.
import shutil
case_replay_src = case_work / 'replay-source'
shutil.copytree(case_src, case_replay_src, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns('network_caseload.csv.gz', '__pycache__'))
case_replay_env = {**os.environ, 'PYTHONPATH': os.pathsep.join(filter(None, [
    str(ROOT / 'non-ml/src'), os.environ.get('PYTHONPATH'),
]))}
case_rebuilt = set()

def caseload_receipt(script, filename, cli_args=(), *, pin_required=False):
    dest = case_work / filename
    if not pin_required or case_release_ok:
        arguments = [sys.executable, str(case_replay_src / script), *map(str, cli_args)]
        if pin_required:
            arguments.extend(['--facilities', str(case_pinned)])
        result = run_cmd([*arguments, '--output', str(dest)], cwd=ROOT, optional=True, env=case_replay_env)
        if result is not None and result.returncode == 0:
            validated = load_json(dest)
            shutil.copyfile(dest, case_replay_src / filename)
            case_rebuilt.add(filename)
            print(f'{filename}: recomputed')
            return validated
    print(f'{filename}: archived pinned receipt')
    return load_json(case_src / filename)

typical_case = caseload_receipt('build.py', 'typical_caseload.json')
baseline_case = caseload_receipt('baseline_coverage.py', 'baseline_coverage.json', pin_required=True)
expanded_case = caseload_receipt('expand_coverage.py', 'expanded_coverage.json', pin_required=True)
xgb_case = caseload_receipt('xgboost_caseload.py', 'xgboost_results.json')

typical_frame = pd.DataFrame(typical_case['estimates'])
display(typical_frame[['facility', 'service', 'typical_month', 'unit', 'method', 'first_month', 'last_month']].head(12))
expanded_frame = pd.DataFrame(expanded_case['summary']).set_index('care_needed')
ax = expanded_frame[['previous_estimated_facilities', 'reported_facilities', 'estimated_facilities']].plot.barh(
    figsize=(9, 7), color=['#d27f47', '#397b7b', '#92b6ad'],
    title='Historical caseload coverage by care option',
)
ax.set_xlabel('Facilities in each evidence category')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
caseload_scores = pd.DataFrame([
    {'service': cohort['service'], 'partition': partition, 'model': model,
     'monthly_mae': metrics['mae'], 'monthly_unit': cohort['unit']}
    for cohort in xgb_case['cohorts'] for partition, test in cohort['evaluation'].items()
    if test for model, metrics in test['metrics'].items()
])
display(caseload_scores)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (service, scores) in zip(axes, caseload_scores.groupby('service', sort=False)):
    scores.pivot(index='partition', columns='model', values='monthly_mae').rename(
        index=lambda label: label.replace('_', ' '),
        columns=lambda label: label.replace('_', ' '),
    ).plot.barh(ax=ax, width=.8, title=service)
    ax.set_xlabel(f'Mean absolute error per month ({scores.monthly_unit.iloc[0]})')
    ax.set_ylabel('')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Same-period and typical-month replays

The earlier March dialysis replay compared reported sessions with predicted machine throughput, using the prior-period leave-site-out p90 sessions per functional machine. One hypothetical midpoint permitted a comparison under the detour cap and none of six origins changed first facility. The 13-option expansion aligned historical site-periods and found zero comparable origins in 34 checks, including an Odisha MRI pair without reviewed same-period throughput. This held evidence-dependent reranking and motivated an explicitly modeled scenario layer. [First dialysis replay](https://github.com/augustchu/datathon-F26/blob/af89658/non-ml/LOAD_BALANCING.md) · [Aligned replay](https://github.com/augustchu/datathon-F26/blob/af89658/non-ml/caseload/CARE_OPTIONS_REPLAY.md)

The later network matrix supplies 181,386 facility–service scenarios across 18 care options. It uses exact branch history when available and conditional peer medians otherwise. Most throughput values use peer p90 activity. Seven exact programme branches use resource-based throughput. The planning replay admits exact service-matched, routing-eligible branches and keeps time-critical services out of reranking. It screened 56 origins, compared 14, and changed the first result at three (one dialysis, two MRI). At two hypothetical origins, allocating 100 sequential synthetic cases moved 58 dialysis activities and all 100 MRI activities away from the nearest branch. Route detours and appointment availability were unmeasured in these stored straight-line scenarios. [Network estimates, provenance and replay](https://github.com/augustchu/datathon-F26/blob/af89658/non-ml/caseload/NETWORK_CASELOAD.md) · [Planning literature and evaluation design](https://github.com/augustchu/datathon-F26/blob/af89658/non-ml/LOAD_BALANCING_LITERATURE.md)

In [ ]:
historical_case = caseload_receipt('build_load_evidence.py', 'load_evidence.json')
dialysis_case = caseload_receipt('replay_load_planning.py', 'load_planning_replay.json', pin_required=True)
aligned_case = caseload_receipt('replay_care_options.py', 'care_options_replay.json', pin_required=True)
forecasts = case_src / 'network_caseload.csv.gz'
network_meta = load_json(case_src / 'network_caseload_receipt.json')
network_upstream = {'typical_caseload.json', 'baseline_coverage.json', 'load_evidence.json',
                    'care_options_replay.json'}
if case_release_ok and network_upstream <= case_rebuilt:
    candidate_forecasts = case_work / forecasts.name
    receipt_path = case_work / 'network_caseload_receipt.json'
    case_result = run_cmd([
        sys.executable, str(case_replay_src / 'build_network_caseload.py'),
        '--facilities', str(case_pinned), '--output', str(candidate_forecasts), '--receipt', str(receipt_path),
    ], cwd=ROOT, optional=True, env=case_replay_env)
    if case_result is not None and case_result.returncode == 0:
        forecasts = candidate_forecasts
        network_meta = load_json(receipt_path)
        for filename in network_upstream:
            assert network_meta['inputs'][filename] == digest(case_work / filename), filename
        print('Network forecast consumed:', ', '.join(sorted(network_upstream)))
if forecasts.parent == case_src:
    print('Using published forecast and receipt for the app handoff.')
network_case = caseload_receipt(
    'replay_network_care.py', 'network_care_replay.json',
    ('--forecasts', forecasts), pin_required=True,
)
display(pd.DataFrame([historical_case['summary'], dialysis_case['summary'], aligned_case['summary'],
                      network_case['summary']],
                     index=['Historical evidence', 'Dialysis replay', 'Aligned options', 'Network replay']))
network_coverage = pd.DataFrame(network_meta['usage_counts']).T.fillna(0)
display(network_coverage[['rows', 'reported_site', 'same_site_predicted',
                          'conditional_peer_predicted', 'predicted_capacity']])
replayed_services = pd.DataFrame(network_case['services']).set_index('care_needed')
ax = replayed_services[['origins', 'origins_compared', 'origins_with_top_change']].plot.barh(
    figsize=(9, 7), color=['#d4dfda', '#397b7b', '#d27f47'], title='First-result planning comparison'
)
ax.set_xlabel('Scenario origins')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
allocation_archive = load_json(case_src / 'network_planning_examples.json')['examples']
allocation_runs = []
for example in allocation_archive:
    service = example['service']
    if case_release_ok:
        destination = case_work / f'{service}-allocation.json'
        origin = example['origin']
        allocation_status = run_cmd([
            sys.executable, str(case_src / 'plan_network_load.py'),
            '--facilities', str(case_pinned), '--forecasts', str(forecasts), '--service', service,
            '--lat', str(origin['lat']), '--lon', str(origin['lon']), '--requests', '100',
            '--output', str(destination),
        ], cwd=ROOT, optional=True)
        if allocation_status is not None and allocation_status.returncode == 0:
            example = load_json(destination)
    allocation_runs.append(pd.DataFrame(example['replay']))
for replay in allocation_runs:
    ax = replay.set_index('name')[['baseline_suggestions', 'balanced_suggestions']].rename(columns={
        'baseline_suggestions': 'Nearest only', 'balanced_suggestions': 'Load balanced',
    }).plot.barh(color=['#397b7b', '#d27f47'], title=f"100 synthetic {replay.service.iloc[0]} activities")
    ax.set_xlabel('Assigned activities')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()

### Equity, access and maternity outcomes

The equity investigation considered favorable risk-adjusted clinical outcomes and favorable reports of respectful care together. A qualifying comparison requires the same service, period, cohort definition, measurements and adjustment method across branches, uncertainty bounds, and at least 30 observations per component. The tie-breaker keeps the same service evidence tier and caps extra travel at 1 km, with an additional cap of 2 minutes when route times are comparable. A move requires the lower score bound to exceed the other branch's upper bound. The current paired evidence file is empty. Four separately cited access records illustrate scheme eligibility questions, including Delhi JSSK maternity and Tamil Nadu and Assam dialysis programmes. [Equity ranking experiment](https://github.com/augustchu/datathon-F26/blob/af89658/possible-next-steps/EQUITY_RANK_FEATURE.md) · [Health equity research](https://github.com/augustchu/datathon-F26/blob/af89658/possible-next-steps/HEALTH_EQUITY_REPORT.md)

The maternity audit reviewed seven named hospitals. KIMS and Guru Teg Bahadur paired outcomes and feedback within dated COVID-positive cohorts. JIPMER paired hospital and dates, but excluded stillbirths, caesareans and selected complications from the survey cohort. Lok Nayak published same-cohort preterm and experience totals but the available tables cannot reveal clinical outcomes by those same demographics. Purnea reported general satisfaction, MediCiti reported surgical-site infections with no corresponding published respect survey, and the denominator for a JK Hospital subgroup ICU percentage was unresolved. None of the seven published both measures by the same demographic group in the same cohort. The pooled Uttar Pradesh report below allowed two descriptive shares by group, with both outcomes self-reported. Those pooled results stayed outside branch ranking. [Seven-hospital maternity audit](https://github.com/augustchu/datathon-F26/blob/af89658/possible-next-steps/MATERNITY_OUTCOMES_AUDIT.md) · [Uttar Pradesh cohort study](https://pubmed.ncbi.nlm.nih.gov/28676965/)

In [ ]:
from datetime import date
from facility_tool.equity import (
    EquityCandidate, load_paired_equity_evidence, rank_for_paired_equity, synthetic_equity_evidence,
)

equity_as_of = date(2026, 11, 21)
equity_pairs = load_paired_equity_evidence(ROOT / 'data/equity/paired_evidence.csv')
display(read_csv(ROOT / 'data/equity/access_evidence.csv')[['service', 'preference', 'source_title', 'source_url']])
equity_candidates = [
    EquityCandidate('fictional-a', 'same evidence tier', 3.1),
    EquityCandidate('fictional-b', 'same evidence tier', 3.8),
]
equity_demo = synthetic_equity_evidence('maternity_delivery', 'fictional-a', 'fictional-b', as_of=equity_as_of)
equity_empty = rank_for_paired_equity(
    equity_candidates, 'maternity_delivery', enabled=True, evidence=equity_pairs, as_of=equity_as_of
)
equity_ranked = rank_for_paired_equity(
    equity_candidates, 'maternity_delivery', enabled=True, evidence=equity_demo, as_of=equity_as_of
)
print(f'Published cohort rows: {len(equity_pairs)}. Published evidence behavior: {equity_empty.status}.')
print('Fictional ranked IDs:', [equity_candidates[i].facility_id for i in equity_ranked.order])
equity_points = pd.DataFrame([
    {'branch': equity_candidates[i].facility_id, 'score': score.value,
     'lower': score.lower, 'upper': score.upper}
    for i, score in equity_ranked.scores.items()
]).sort_values('branch')
fig, ax = plt.subplots(figsize=(7, 2.5))
ax.errorbar(equity_points['score'], equity_points['branch'],
            xerr=[equity_points['score'] - equity_points['lower'], equity_points['upper'] - equity_points['score']],
            fmt='o', capsize=4, color='#397b7b')
ax.set(xlim=(0, 100), xlabel='Mean favorable outcome and experience score', title='Fictional equity comparison')
plt.tight_layout()
plt.show()

In [ ]:
# Transcription of the published pooled UP cross-tabs cited in the maternity audit.
maternity_up = pd.DataFrame([
    ('Scheduled Caste or Scheduled Tribe', 800, 180, 350),
    ('Other Backward Classes', 1546, 328, 686),
    ('Other caste group', 293, 44, 173),
    ('Lowest wealth quintile', 528, 125, 246),
    ('Highest wealth quintile', 528, 98, 209),
], columns=['group', 'n', 'mistreatment', 'delivery complications'])
display(maternity_up)
maternity_shares = maternity_up.set_index('group')[['mistreatment', 'delivery complications']].div(
    maternity_up.set_index('group')['n'], axis=0
).mul(100)
ax = maternity_shares.plot.barh(figsize=(9, 4), color=['#397b7b', '#d27f47'],
                               title='Self-reported childbirth experience and complications, pooled study')
ax.set_xlabel('Percent within published group')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

### In The Loop verification and sampling

The project version supported screenshot-assisted comparisons, conflicting publisher claims and browser-local review exports. The standalone GitHub Pages application opens original source pages at text fragments or PDF locators, preserves the review identity, and freezes a uniform random sample for a threshold test. Both keep analyst decisions separate from the source claims. On the historical ten-branch desk review, 20 candidate rows yielded 2 accepted, 2 rejected and 16 inconclusive results. At branch level, only two exact whole-branch counts were resolved. The independent human worklist covered 130 of 195 competing-count branches and remained pending. The linked publisher queue has 161 claim rows across those branches, a different sampling frame. [Desk-review protocol and screenshots](https://github.com/augustchu/datathon-F26/tree/af89658/max-supplemented-data/experiments/verification) · [Standalone app and release](https://github.com/yana-157/in-the-loop-verification-tool/tree/fe2c9db) · [GitHub Pages application](https://yana-157.github.io/in-the-loop-verification-tool/)

Two-sided precision goals for the branch worklist and the output-first accuracy test answer different questions. The 161-row app setup sets a 90% accuracy floor, 98% planning accuracy, 5% significance and 80% target power. Its minimum sample is 39, with at least 38 correct required to pass. The standalone app defers the result when any sampled items stay inconclusive unless the reviewer explicitly counts them as wrong. App confidence bounds quantify copying accuracy within the submitted sheet. A separate source-first draw is needed to audit omissions.

The insurer scraper audit independently sampled precision and recall for beds and doctors for both insurers. All 48 sampled observations matched. Each combination had only six checks, giving an exact 95% lower bound near 54%. A separate targeted probe found a missed IFFCO value recorded with a leading zero, changing the parser. The archived browser workload paired six trials on 20 fictional disclosure pages, measuring page loading with and without full-page capture. Capture used roughly 7.8× as much browser CPU in that setup. The timing excludes researcher review and agent inference costs. [Scraper fidelity audit](https://github.com/augustchu/datathon-F26/tree/af89658/max-supplemented-data/experiments/verification/scraper-audit) · [Browser workload report](https://github.com/augustchu/datathon-F26/blob/af89658/In%20The%20Loop%20Verification%20Tool/REPORT.md)

In [ ]:
from IPython.display import Image

verification_experiment = ROOT / 'max-supplemented-data/experiments/verification'
desk_review = load_json(verification_experiment / 'result.json')
audit_metrics = pd.DataFrame(load_json(verification_experiment / 'scraper-audit/results.json')['results'])
audit_metrics['exact_95_lower'] = audit_metrics['exact_binomial_95_interval'].map(lambda bounds: bounds[0])
audit_metrics['exact_95_upper'] = audit_metrics['exact_binomial_95_interval'].map(lambda bounds: bounds[1])
display(audit_metrics[['publisher', 'metric', 'field', 'checks', 'matches', 'estimate',
                       'exact_95_lower', 'exact_95_upper']])
ax = pd.Series(desk_review['status_counts']).plot.bar(
    color=['#d4dfda', '#d27f47', '#397b7b'], title='Recorded verification desk review'
)
ax.set_ylabel('Candidate claims across 10 branches')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()
display(Image(filename=str(verification_experiment / 'screenshots/in-the-loop-desk-review.png'), width=970))

In [ ]:
# Test the project copy and standalone release when Node is available.
node_executable = shutil.which('node')
if node_executable:
    for verification_app in (ROOT / 'In The Loop Verification Tool', VERIFICATION_ROOT):
        if not (verification_app and Path(verification_app).is_dir()):
            continue
        verification_app = Path(verification_app)
        node_tests = sorted((verification_app / 'tests').glob('*.test.mjs'))
        run_cmd([node_executable, '--test', *map(str, node_tests)], cwd=verification_app, optional=True)
    if VERIFICATION_ROOT and (Path(VERIFICATION_ROOT) / 'lib/audit.js').is_file():
        audit_demo_script = '''
import { planAudit, evaluateAudit } from './lib/audit.js';
const p = planAudit(161, { alpha: .05, floor: .90, designAccuracy: .98, power: .80 });
const pending = evaluateAudit(p, {correct: 38, incorrect: 0, inconclusive: 1, remaining: 0});
const resolved = evaluateAudit(p, {correct: 38, incorrect: 1, inconclusive: 0, remaining: 0});
console.log(JSON.stringify({plan: p, pending, resolved}, null, 2));
'''
        run_cmd([node_executable, '--input-type=module', '-e', audit_demo_script], cwd=Path(VERIFICATION_ROOT), optional=True)
else:
    print('Node 22+ is needed for the JavaScript tests and live sampling calculation.')

In [ ]:
# Recompute browser timings where a compatible Chrome driver binary is available.
VERIFY_CHROME_BINARY = ''
verification_bench = ROOT / 'In The Loop Verification Tool' / 'benchmarks'
bench_path = verification_bench / 'browser-workload.json'
if VERIFY_CHROME_BINARY:
    fresh_bench = RUN / 'browser-workload.json'
    bench_run = run_cmd([
        sys.executable, str(verification_bench / 'browser_workload.py'),
        '--chrome', VERIFY_CHROME_BINARY, '--output', str(fresh_bench),
    ], cwd=ROOT, optional=True)
    if bench_run is not None and bench_run.returncode == 0:
        bench_path = fresh_bench
browser_workload = load_json(bench_path)
browser_frame = pd.DataFrame([
    {'workflow': workflow.replace('_', ' '), 'pair': index, **trial}
    for workflow, trials in browser_workload['samples'].items()
    for index, trial in enumerate(trials, start=1)
])
print('Benchmark source:', bench_path.name, '| browser:', browser_workload['browser'])
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, metric, label in zip(axes, ('chrome_cpu_seconds', 'elapsed_seconds'),
                             ('Browser CPU seconds', 'Elapsed seconds')):
    for _, values in browser_frame.groupby('pair'):
        ax.plot(values['workflow'], values[metric], color='#397b7b', alpha=.45, marker='o')
    ax.set_ylabel(label)
    ax.set_title('Six paired trials')
plt.tight_layout()
plt.show()

### Hindi IVR and future verification work

The call prototype confirms an exact branch, asks one service at a time and keeps offered, declined, unsure, unanswered and unasked statuses distinct. A staff member can opt into a short note about missing services. The caller flow generates a transcript or sample TwiML offline. The runbook describes prospective contact preparation, provider callbacks, human review and costs, but no live campaign or outcomes were recorded. A staff answer remains a dated facility self-report. Branch bed and doctor counts require a scoped administrator follow-up or source document. [IVR launch and pilot runbook](https://github.com/augustchu/datathon-F26/blob/af89658/non-ml/MASS_VERIFICATION.md)

The proposal documents prioritize source-grounded recovery, validation on independently recovered missing fields, and verification tasks chosen by their effect on a recommendation. The targeted review and a probability sample serve complementary purposes. Worth preserving from the candidate list are service-specific operational readiness, equity/access mapping, service-relevant ratings, external branch linkage and better retrospective diagnostic tests before larger models or nationwide collection. The proposals list evaluation criteria and resource trade-offs. [Brainstorm and historical rationale](https://github.com/augustchu/datathon-F26/tree/af89658/possible-next-steps)

In [ ]:
ivr_cases = [
    ('Confirmed branch, dialysis yes, MRI no, optional note', ['1', '1', '2', '3']),
    ('Confirmed branch, dialysis unsure, MRI no, finish', ['1', '0', '2', '4']),
    ('Wrong branch', ['2']),
]
ivr_results = []
for label, keys in ivr_cases:
    print(label)
    ivr_run = run_cmd([sys.executable, 'non-ml/preview_hindi_ivr.py', '--services', 'dialysis', 'mri',
                       '--keys', *keys], cwd=ROOT)
    receipt_start = ivr_run.stdout.rfind(chr(10) + '{')
    if receipt_start >= 0:
        receipt = json.loads(ivr_run.stdout[receipt_start + 1:])
        ivr_results.append({
            'case': label, 'branch_status': receipt['branch_status'],
            'note requested': receipt['note_requested'],
            **receipt['service_answers'],
        })
if ivr_results:
    display(pd.DataFrame(ivr_results))
run_cmd([sys.executable, 'non-ml/preview_hindi_ivr.py', '--services', 'dialysis', 'mri', '--twiml'], cwd=ROOT)

In [ ]:
# Frozen recommendation results, historical scorer files, and browser captures.
# The payload carries the pinned benchmark evidence.
import base64
import io
import zipfile

benchmark_payload = (
    '...'
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(benchmark_payload))) as history:
    history.extractall(BENCH_ROOT)
print(f"Benchmark evidence extracted to {BENCH_ROOT}")

## 07 · Facility Finder application audit

The intended final app is the restored **Facility Finder** on `restore-care-finder-map` at [d682872](https://github.com/augustchu/datathon-F26/commit/d68287276f3e9a275283eb005799c3be7db3bba5). Its manifest runs `classic_dashboard.py`. The default GitHub branch at [af89658](https://github.com/augustchu/datathon-F26/commit/af8965890e4de2e5a5f2a24177425e5fdb77352d) runs a parallel Streamlit prototype called **Find the right care**, via `dashboard.py`. The saved 16-query evaluation exercised the main-branch prototype at [18335aa](https://github.com/augustchu/datathon-F26/commit/18335aa43927425f5a2e2177c4cf45fe407d84fa), before its map and published-load changes.

| Decision | What the team shipped or kept |
|---|---|
| Map dashboard lineage | The selected restoration starts from `a141a75`, which removed the earlier in-app bed estimator, kept adjustable fit weights and reported beds, and presented service, location and options separately. It brings back the India overview, origin marker, numbered pins and matching order of cards. |
| User input | The restored page offers 18 services organized by 13 specialties or by body area, and accepts PIN, city, district or coordinates. The prototype offers nine services and an approximate city/district/PIN centroid. Both offer driving estimates and straight-line fallback. |
| Source of truth | Both prepare the latest cleaned release and compare its original capacity values with the Marketplace table by ID. Holdouts, filled values, estimates and estimator intervals have distinct provenance. The app reads a prepared table through Databricks SQL. |
| Ranking | Recommended combines travel with match and service-confidence adjustments. The classic app also offers distance, fit, recorded beds, recorded occupancy and advanced rank-order centroid priorities. |
| Projected demand | The older prototype's NFHS-scaled district population share split a hypothetical load evenly across candidates. Both published branches instead offer an opt-in branch/service comparison using the bundled monthly projections. It requires comparable units, scope, period and evidence, and allows one promotion within 10 additional straight-line km and, for routed comparisons, 15 additional driving minutes. |
| Equity | The restored app can show a fictional maternity comparison. Its bundled equity evidence has headers and zero real paired rows. The equity evidence and guardrails are covered in the side-project section. |

The restored app again prioritizes the contact block, reasons and caveats above evidence details. Its check of service confidence follows classic keyword-based candidate selection. Main has additional eligibility checks for audited branch points, explicit conflicts and service evidence. An app card can indicate documentary evidence or a modeled projection. Current hours, staffing and appointment availability require confirmation with the facility.

[Map restore import decisions](https://github.com/augustchu/datathon-F26/blob/d68287276f3e9a275283eb005799c3be7db3bba5/first-prototype/MAP_RESTORE.md) · [Prototype decisions](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/first-prototype/DECISIONS.md)

In [ ]:
import hashlib
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Image, display as show_rich_output

APP_RUN = RUN / "app-audit"
APP_RUN.mkdir(parents=True, exist_ok=True)
MAIN_APP = ROOT / "first-prototype"
CLASSIC_APP = APP_ROOT / "first-prototype"
BENCH_CODE = ROOT / "recommendation-benchmark"
APP_RELEASE = ROOT / "max-supplemented-data/capacity_and_confidence"
APP_REPLAYS = MAIN_APP / "replays/2026-11-22"
APP_POPULATION = ROOT / "load-balancing-pipeline/outputs/nfhs_district_population_2020.csv"

manifest_rows = []
for label, folder in [("Intended restored Facility Finder", CLASSIC_APP), ("Default-branch prototype", MAIN_APP)]:
    manifest = (folder / "app.yaml").read_text()
    manifest_rows.append({
        "implementation": label, "launch": manifest.splitlines()[0],
        "source files": sum(path.is_file() for path in folder.iterdir()),
        "has published load data": (folder / "projected_load.csv").is_file(),
        "ships audited point holds": (folder / "location_routing_holds.json").is_file(),
    })
display(pd.DataFrame(manifest_rows))

### Release integration and coordinate audit

The release's confidence table is paired to its capacity table by validated source-row order. The prepared snapshot then joins the release against the Marketplace `unique_id`. Geography attaches ADM2, NFHS and district population where the district/state key is unambiguous. The app's route provider uses a typical weekday-morning drive with a three-hour search limit. Approximate origins and straight-line distances are useful for offline reproduction, especially for comparing ranks. In a mountain catchment they can differ substantially from actual trips.

The alternate branch kept the classic candidate screen. The next cell matches every branch-coordinate hold from the main-branch audit back to the release and tests whether the classic bounding box would admit it. It also counts the projection rows shipped in both app directories. The app projection loaders match individual identifiers and evidence scope. The input-hash check belongs to the upstream caseload build, whose historical capacity file has a different hash from the currently published capacity CSV.

[Location audit](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/non-ml/LOCATION_AUDIT.md) · [Caseload methods](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/non-ml/caseload/NETWORK_CASELOAD.md)

In [ ]:
release_points = pd.read_csv(
    APP_RELEASE / "facilities_newest_capacity.csv",
    usecols=["facility_id", "latitude", "longitude", "routing_eligible"],
)
held_points = pd.DataFrame(load_json(MAIN_APP / "location_routing_holds.json")["holds"])
hold_comparison = held_points.merge(release_points, on="facility_id", suffixes=("_held", "_release"))
same_point = (
    np.isclose(hold_comparison.latitude_held, hold_comparison.latitude_release, atol=1e-7, rtol=0)
    & np.isclose(hold_comparison.longitude_held, hold_comparison.longitude_release, atol=1e-7, rtol=0)
)
classic_box = hold_comparison.latitude_release.between(6.5, 37.5) & hold_comparison.longitude_release.between(68, 97.5)
print(f"{len(held_points):,} audited hold rows. {(same_point & classic_box).sum():,} unchanged points inside the classic screening box.")
display(hold_comparison.loc[same_point & classic_box, ["reason"]].value_counts().rename("points").reset_index().head(8))

projections = pd.read_csv(CLASSIC_APP / "projected_load.csv")
fig, ax = plt.subplots(figsize=(8.4, 4.2))
projections["care_needed"].value_counts().sort_values().plot.barh(ax=ax, color="#288a7e")
ax.set(title="Published monthly projection rows shipped with Facility Finder", xlabel="Branch–service rows", ylabel="")
plt.tight_layout()
plt.show()

#### Projected-load promotion in the app

This small test uses two bundled CT projection rows from one comparable scope. The trip distances are deliberately synthetic. It exercises the implementation that ships with both apps, including the distance guard, without inferring their actual routes or appointments.

In [ ]:
load_demo_script = APP_RUN / "projected_load_demo.py"
load_demo_script.write_text("""
from pathlib import Path
import sys
import pandas as pd

app, out = map(Path, sys.argv[1:])
sys.path.insert(0, str(app))
from projected_load import load_projections, prefer_projected_load

projections = load_projections()
ct = projections.loc[projections.care_needed.eq("CT scan")].copy()
ct["pressure"] = ct.caseload.div(ct.caseload + ct.capacity)
comparable = max((group for _, group in ct.groupby(["unit", "scope", "period"])), key=len)
picked = comparable.loc[[comparable.pressure.idxmax(), comparable.pressure.idxmin()]].copy()
picked["role"] = ["Baseline", "Alternative"]
rows = picked[["facility_id"]].assign(
    service_evidence_kind="record_listing", distance_km=[6.0, 7.0],
    travel_time_minutes=[float("nan"), float("nan")], base_travel_cost=[6.0, 7.0],
)
promoted, count = prefer_projected_load(rows, "CT scan", routed=False)
assert count == 2 and promoted.iloc[0].facility_id == picked.iloc[1].facility_id
farther = rows.assign(distance_km=[6.0, 17.0], base_travel_cost=[6.0, 17.0])
guarded, count = prefer_projected_load(farther, "CT scan", routed=False)
assert count == 0 and guarded.iloc[0].facility_id == picked.iloc[0].facility_id
picked[["role", "caseload", "capacity", "pressure", "unit", "period", "scope"]].assign(
    synthetic_km=[6, 7], baseline_rank=[1, 2], rank_with_load=[2, 1],
).to_csv(out, index=False)
""", encoding="utf-8")
load_demo_path = APP_RUN / "projected_load_demo.csv"
run_cmd([sys.executable, load_demo_script, CLASSIC_APP, load_demo_path])
load_demo_table = pd.read_csv(load_demo_path)
display(load_demo_table.round(3))
ax = load_demo_table.set_index("role")[["caseload", "capacity"]].plot.bar(
    color=["#c3a04a", "#218476"], figsize=(7.5, 3.8), rot=0,
)
ax.set(title="Comparable CT activity and throughput in the bundled projection",
       ylabel=load_demo_table.unit.iloc[0], xlabel="Synthetic two-branch trip scenario")
plt.tight_layout()
plt.show()

All 509 audited branch-point pairs are unchanged in the release and pass the classic bounding-box screen. The restored branch can consequently admit held points to candidate and origin screening. Final rank still depends on the requested service, origin and score. The restored UI also applies the stricter evidence score after initial keyword eligibility, so a low or zero score increases the travel-equivalent penalty without removing the candidate. The benchmark below measures the resulting differences at the list level.

### Why the ranking changed

An early confidence measure scored five facility fields and applied 0, 40 or 80 km-equivalent penalties. It measured the completeness and checking status of a facility record. For recommendations the team switched to **confidence for the requested service**. Documentary support adds 80 points for corroborated official evidence, 75 for a hospital source, 70 for a reviewed directory, 50 for an explicit record listing or 15 for a passing mention. Branch identity adds up to 20. Explicit listings score 100 for service match and passing mentions 40. The main-branch candidate gate keeps active reviewed conflicts and unusable coordinates out.

The reviewed-only pass yielded three branch–service results. Adding record text yielded 8,000. Distinguishing listings from mentions raised confidence on 2,444 candidates without changing eligibility at that stage. The final 60-minute logistic curve uses midpoint 55 and width 3.5, allowing a typical explicit listing at 60 confidence to compete on travel and retaining a large penalty for weak mentions. Confidence 90 offsets about 11.6 additional driving minutes relative to confidence 60, with service match and other inputs held equal.

| Rank replay | What changed in 27 fixed routine searches |
|---|---|
| Legacy → service-specific | 23 winners and 24 top-three memberships changed. Direct-match winners increased from 8 to 24, with average winner distance falling from 19.10 to 9.39 km. |
| Linear cap 5 → 15 minutes | No winner changed. Three top-three memberships changed. |
| Linear → hour-scale piecewise | Three winners changed. The New Delhi NICU winner moved from 9.22 to 23.52 km while confidence increased from 60 to 95. |
| Piecewise → logistic at 75 | The winners stayed the same, with one change in top-three membership. |
| Logistic at 75 → steeper at 55 | One winner and five top-three lists changed. The New Delhi NICU winner moved back to 9.22 km. |

These runs used three city centroids, nine services, straight-line distance and zero population share. They test ranking sensitivity. Live trips, outcomes, demand and clinically acceptable detours were outside data coverage.

[Ranking tradeoff and receipts](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/first-prototype/DECISIONS.md#scoring-and-ranking-evolution)

In [ ]:
def logistic_app_penalty(value, center, width):
    curve = lambda x: 1 / (1 + np.exp((x - center) / width))
    return 60 * (curve(value) - curve(100)) / (curve(0) - curve(100))

score_changes = pd.read_csv(APP_REPLAYS / "service_score_changes.csv")
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.6), layout="constrained")
chart = score_changes.set_index("service")[["high_match_candidates", "passing_mention_candidates"]]
chart.plot.barh(stacked=True, color=["#218476", "#cba751"], ax=axes[0])
axes[0].set(title="What evidence admitted each candidate?", xlabel="Branch–service results", ylabel="")
axes[0].legend(["Direct match", "Passing mention"], frameon=False)
confidence = np.linspace(0, 100, 301)
axes[1].plot(confidence, 15 * (1 - confidence / 100), label="Linear, 15 min", color="#757e8d")
axes[1].plot(confidence, logistic_app_penalty(confidence, 75, 5), label="Logistic, midpoint 75", color="#cba751")
axes[1].plot(confidence, logistic_app_penalty(confidence, 55, 3.5), label="Logistic, midpoint 55", color="#218476")
axes[1].set(title="How confidence changes the travel-equivalent penalty",
            xlabel="Evidence confidence (0–100)", ylabel="Additional minute-equivalents")
axes[1].legend(frameon=False)
plt.show()

ranking_trials = {
    "ranking_update": "Legacy → service-specific",
    "confidence_weight": "Linear 5 → 15",
    "confidence_hour_piecewise": "Linear → piecewise",
    "confidence_hour_logistic": "Piecewise → logistic 75",
    "confidence_logistic55": "Logistic 75 → 55",
}
archived_rank_trials = []
for stem, label in ranking_trials.items():
    table = pd.read_csv(APP_REPLAYS / f"{stem}_summary.csv")
    archived_rank_trials.append({
        "comparison": label, "searches": len(table),
        "winner changed": int(table.winner_changed.sum()),
        "top-three membership changed": int(table.top_k_overlap.lt(table.top_k).sum()),
        "top-three order or membership changed": int(table.before_top.ne(table.after_top).sum()),
        "mean winner km before": table.before_winner_travel.mean(),
        "mean winner km after": table.after_winner_travel.mean(),
    })
display(pd.DataFrame(archived_rank_trials).round(2))

### Rebuild both app snapshots and run the ranking panel

These cells use the project-release facility/NFHS paths from the preflight and write replay assets in the run directory. They use the original cleaned-release capacity/confidence files and the historical review date, 2026-11-22. The separately tracked district population output lets us test the app's enrichment with its own artifact. Use the preflight GeoJSON/ZIP to recalculate the geographic check. Original archived inputs retain their hashes in the benchmark receipt. A workspace export can have different byte formatting even when its rows and values agree.

The classic benchmark uses City for the panel's `city, state` strings. It uses the classic service dictionary and score functions with no route credentials, load toggle or equity toggle. The main/prototype replay uses the benchmark's original call path. Both keep unsupported service/mode cases distinct from failed place resolution and ranked lists.

In [ ]:
APP_INPUTS_AVAILABLE = bool(SRC.get("facilities") and SRC.get("nfhs"))
if APP_INPUTS_AVAILABLE:
    def app_input_frame(path):
        path = Path(path)
        name = path.name.lower()
        if name.endswith((".csv", ".csv.gz")):
            return pd.read_csv(path, low_memory=False)
        if name.endswith((".jsonl", ".jsonl.gz", ".ndjson")):
            frame = pd.read_json(path, lines=True, dtype=False)
            if len(frame) and "columns" in frame and isinstance(frame.iloc[0]["columns"], list):
                frame = frame.iloc[1:].drop(columns="columns").reset_index(drop=True)
            return frame
        if name.endswith((".json", ".json.gz")):
            import gzip
            opener = gzip.open if name.endswith(".gz") else open
            with opener(path, "rt", encoding="utf-8") as handle:
                payload = json.load(handle)
            if isinstance(payload, dict) and {"columns", "rows"}.issubset(payload):
                columns = [item["name"] if isinstance(item, dict) else item for item in payload["columns"]]
                return pd.DataFrame(payload["rows"], columns=columns)
            return pd.DataFrame(payload)
        if name.endswith(".parquet") or path.is_dir():
            return pd.read_parquet(path)
        raise ValueError(f"Unsupported app audit workspace input: {path}")


    app_raw_frame = app_input_frame(SRC["facilities"])
    app_raw_frame = app_raw_frame.reindex(columns=["unique_id", "capacity"])
    app_raw_frame["unique_id"] = app_raw_frame["unique_id"].astype("string")
    assert app_raw_frame.unique_id.notna().all(), "Every Marketplace facility row needs a source ID."
    app_input_raw = APP_RUN / "facility_ids_and_original_capacity.jsonl"
    app_raw_frame.to_json(app_input_raw, orient="records", lines=True, force_ascii=False)
    app_input_nfhs = APP_RUN / "nfhs_table.json"
    app_nfhs_frame = app_input_frame(SRC["nfhs"])
    nfhs_payload = json.loads(app_nfhs_frame.to_json(orient="split", index=False))
    nfhs_payload["rows"] = nfhs_payload.pop("data")
    app_input_nfhs.write_text(json.dumps(nfhs_payload))
    app_geography = Path(SRC.get("boundaries") or EXT["boundaries"])
    if app_geography.suffix.casefold() == ".zip":
        from zipfile import ZipFile
        with ZipFile(app_geography) as archive:
            geojson_members = [name for name in archive.namelist() if name.lower().endswith(".geojson")]
            geojson_members = [name for name in geojson_members if "simplified" not in name.lower()] or geojson_members
            if not geojson_members:
                raise ValueError("The supplied archive has no GeoJSON member.")
            selected_member = max(geojson_members, key=lambda name: archive.getinfo(name).file_size)
            app_input_boundaries = APP_RUN / "organizer_adm2.geojson"
            app_input_boundaries.write_bytes(archive.read(selected_member))
    else:
        app_input_boundaries = app_geography
    print(f"Prepared app inputs: {len(app_raw_frame):,} source facility IDs and {len(app_nfhs_frame):,} NFHS rows.")
else:
    print("Set the checked facilities and NFHS paths to replay the app on workspace data.")

In [ ]:
app_runner = APP_RUN / "replay_app.py"
app_runner.write_text("""
import contextlib
from datetime import date
from hashlib import file_digest
import io
import json
import os
from pathlib import Path
import runpy
import sys
from time import perf_counter
from types import SimpleNamespace

import numpy as np
import pandas as pd

repo, alternate, archive, work, fdr, nfhs, boundary = map(Path, sys.argv[1:8])
version = sys.argv[8]
classic = version == "classic"
app = (alternate if classic else repo) / "first-prototype"
code = repo / "recommendation-benchmark"
sys.path[:0] = [str(app), str(code), str(repo / "first-prototype")]
import pipeline
import replay_service_confidence as replay
import run_first_prototype as benchmark
from traveltime_adapter import travel_time_matrix

def sha(path):
    with path.open("rb") as handle:
        return file_digest(handle, "sha256").hexdigest()

as_of = date(2026, 11, 22)
release = repo / "max-supplemented-data/capacity_and_confidence"
args = SimpleNamespace(
    capacity=release / "facilities_newest_capacity.csv", confidence=release / "facility_confidence.csv",
    raw_facilities=fdr, nfhs=nfhs, boundaries=boundary, as_of=as_of,
    population=repo / "load-balancing-pipeline/outputs/nfhs_district_population_2020.csv",
    location_holds=(repo / "first-prototype/location_routing_holds.json"),
    reviewed_evidence=app / "service_confidence_evidence.json",
)
queries = json.loads((code / "queries.json").read_text())
if version == "historical":
    pipeline = replay.load_pipeline(archive / "pipeline-18335aa.py", "historical_app_pipeline")

if classic:
    import logging
    for name in ("DATABRICKS_HTTP_PATH", "DATABRICKS_WAREHOUSE_HTTP_PATH", "DATABRICKS_WAREHOUSE_ID"):
        os.environ.pop(name, None)
    logging.getLogger("streamlit").setLevel(logging.CRITICAL)
    from classic_data import rank_classic_results, service_evidence, snapshot_to_classic
    from classic_ranking import score_recommended
    import classic_ranking
    from functools import partial
    # Keep the same review clock as the archived audit, including when run later.
    classic_ranking.assess_service_confidence = partial(classic_ranking.assess_service_confidence, as_of=as_of)
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        ui = runpy.run_path(str(app / "classic_dashboard.py"))
    capacity = pd.read_csv(args.capacity, dtype={"unique_id": "string"}, low_memory=False)
    confidence = pd.read_csv(args.confidence)
    release_source = pipeline.attach_capacity_release(replay.load_raw(fdr), capacity, confidence)
    frame = pipeline.prepare_facilities(release_source)
    # A classic search never consumes the geography/NFHS join or old synthetic load scenario.
    facilities = snapshot_to_classic(frame, ui["from_facility_table"])
    menu = benchmark.SERVICE_MENU
    weights = dict(capability=.40, availability=.25, quality=.20, confidence=.15)

    def run_query(q):
        started = perf_counter()
        result = {"query_id": q["id"], "results": []}
        selected = menu.get(q["service"])
        if selected not in ui["SERVICES"]:
            result.update(outcome="unsupported_query", reason="No exact service in the classic menu.")
        elif q["transport_mode"] != "driving":
            result.update(outcome="unsupported_query", reason="Routed transport in the classic app is driving.")
        else:
            try:
                loc, _, state = q["origin"].partition(",")
                lat, lon, _, _, _ = ui["resolve_location"](
                    "City", {"name": loc.strip(), "state": state.strip()}, facilities,
                )
                meta = ui["SERVICES"][selected]
                urgency = "emergency" if q["service"] == "emergency_c_section" else "urgent" if meta["urgent"] else "routine"
                candidates = ui["score_facilities"](facilities, selected, lat, lon, weights)
                ranked = rank_classic_results(candidates, selected, (lat, lon),
                                              population_share_pct=0, severity=urgency, use_traveltime=False).results
                score_recommended(ranked, selected, meta["aliases"])
                ranked = sorted(ranked, key=lambda item: (item["recommended"]["score"],
                                                         item["recommended"]["travel"], item["dist"]))
                for item in ranked[:5]:
                    row, claim = item["row"], item["service_confidence"]
                    record = service_evidence(row, item["matched_service"], (), ui["SERVICES"])
                    evidence = claim if claim.term else record
                    url = next(iter(claim.urls or record.urls), None)
                    result["results"].append({
                        "facility_id": row["facility_id"], "name": row["name"], "pin": row["pin"] or None,
                        "city": row["city"], "state": row["state"], "evidence_excerpt": evidence.excerpt or None,
                        "source_url": url, "citation_scope": "reviewed_service" if claim.urls else "facility_record" if url else None,
                        "service_evidence_kind": claim.kind, "service_match_score": claim.match_score,
                        "service_confidence_score": claim.score, "travel_basis": "straight_line",
                        "travel_mode": None, "travel_value": item["dist"], "score": item["recommended"]["score"],
                    })
                result.update(outcome="recommendations" if result["results"] else "abstain",
                              reason=None if result["results"] else "No classic keyword candidates passed the search.",
                              routing={"status": "offline", "origin_basis": "Classic exact city/state facility centroid",
                                       "severity": urgency, "candidates": len(ranked), "provider_reachable": None})
            except (ValueError, KeyError) as exc:
                result.update(outcome="error", reason=f"{type(exc).__name__}: {exc}")
        result["latency_ms"] = round((perf_counter() - started) * 1000, 2)
        return result
else:
    frame = benchmark.build_snapshot(args, pipeline, replay, pd)

    def run_query(q):
        return benchmark.run_query(q, frame, pipeline, travel_time_matrix, False)

hash_paths = {
    "raw_facilities": fdr, "capacity": args.capacity, "confidence": args.confidence,
    "reviewed_evidence": args.reviewed_evidence, "boundaries": boundary, "population": args.population, "nfhs": nfhs,
}
if not classic:
    hash_paths["location_holds"] = args.location_holds
code_sha = {
    "classic": "d68287276f3e9a275283eb005799c3be7db3bba5",
    "historical": "18335aa43927425f5a2e2177c4cf45fe407d84fa",
    "main": "af8965890e4de2e5a5f2a24177425e5fdb77352d",
}[version]
predictions = [run_query(q) for q in queries]
report = {"run": {
    "id": f"workspace-{version}", "code_commit": code_sha, "routing": "straight_line",
    "algorithm": "Restored classic Facility Finder Recommended" if classic else "Main-line prototype rank_candidates",
    "snapshot": f"Workspace replay with {len(frame)} source rows", "evidence_as_of": as_of.isoformat(),
    "execution": "offline workspace replay", "population_share_pct": 0,
    "input_sha256": {key: sha(path) for key, path in hash_paths.items()},
}, "predictions": predictions}
(work / f"{version}-predictions.json").write_text(json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False))
print(version, {outcome: sum(p["outcome"] == outcome for p in predictions)
                for outcome in sorted({p["outcome"] for p in predictions})})
""", encoding="utf-8")
if APP_INPUTS_AVAILABLE:
    for app_version in ("historical", "main", "classic"):
        run_cmd([sys.executable, app_runner, ROOT, APP_ROOT, BENCH_ROOT, APP_RUN, app_input_raw,
                 app_input_nfhs, app_input_boundaries, app_version])

In [ ]:
# Recompute each archived ranking pair from its exact implementation and current workspace rows.
app_ranking_runner = APP_RUN / "replay_ranking.py"
app_ranking_runner.write_text("""
from datetime import date
from hashlib import file_digest
import json
from pathlib import Path
import sys
import pandas as pd

root, archive, work, raw, boundaries = map(Path, sys.argv[1:])
prototype = root / "first-prototype"
sys.path.insert(0, str(prototype))
import replay_service_confidence as replay
import replay_service_ranking as comparison

archive_replays = prototype / "replays/2026-11-22"
def load_ranker(expected, role):
    source = archive / f"ranking-{expected[:12]}.py"
    if not source.exists() and expected.startswith("5cd2282ab2fb"):
        source = archive / "pipeline-18335aa.py"
    with source.open("rb") as handle:
        actual = file_digest(handle, "sha256").hexdigest()
    if actual != expected:
        raise ValueError(f"Unexpected ranking source bytes: {source.name}")
    return replay.load_pipeline(source, role)

ranking_names = ["ranking_update", "confidence_weight", "confidence_hour_piecewise",
                 "confidence_hour_logistic", "confidence_logistic55"]
current = replay.load_pipeline(archive / "pipeline-18335aa.py", "current_score_pipeline")
legacy = replay.load_pipeline(archive / "pipeline-b649ce3.py", "legacy_score_pipeline")
raw_frame = replay.load_raw(raw)
capacity_path = root / "max-supplemented-data/capacity_and_confidence"
capacity = pd.read_csv(capacity_path / "facilities_newest_capacity.csv",
                       dtype={"unique_id": "string"}, low_memory=False)
confidence = pd.read_csv(capacity_path / "facility_confidence.csv")
as_of = date(2026, 11, 22)
prepared = current.prepare_facilities(
    current.attach_capacity_release(raw_frame, capacity, confidence),
    reviewed_evidence=current.load_reviewed_service_evidence(prototype / "service_confidence_evidence.json"),
    audited_location_holds=current.load_audited_location_holds(prototype / "location_routing_holds.json"),
    as_of=as_of,
)
prepared = current.attach_regions(prepared, json.loads(boundaries.read_text()), pd.DataFrame())
before = legacy.prepare_facilities(legacy.attach_capacity_release(raw_frame, capacity, confidence))
service_scores, _ = replay.compare_services(legacy, before, current, prepared)
pd.DataFrame(service_scores).to_csv(work / "legacy_to_final_service_scores.csv", index=False)
for name in ranking_names:
    receipt = json.loads((archive_replays / f"{name}_receipt.json").read_text())
    expected = receipt["sha256"]
    # Earlier scores are evaluated on the same final cohort and origins.
    previous = load_ranker(expected["baseline_pipeline"], f"before_{name}")
    comparison.current = load_ranker(expected["current_pipeline"], f"after_{name}")
    summaries, results = [], []
    for case in receipt["rankings"]:
        row, details = comparison.compare_service(
            previous, prepared, case["service"], case["origin"],
            (case["origin_latitude"], case["origin_longitude"]), case["origin_basis"],
            receipt["severity"], None, receipt["top_k"],
        )
        summaries.append(row)
        results.extend(details)
    pd.DataFrame(summaries).to_csv(work / f"{name}_recomputed.csv", index=False)
    pd.DataFrame(results).to_csv(work / f"{name}_recomputed_top_three.csv", index=False)
    print(name, len(summaries), "fixed-cohort searches")
""", encoding="utf-8")
if APP_INPUTS_AVAILABLE:
    run_cmd([sys.executable, app_ranking_runner, ROOT, BENCH_ROOT, APP_RUN, app_input_raw, app_input_boundaries])

    trial_check = []
    for stem, label in ranking_trials.items():
        new = pd.read_csv(APP_RUN / f"{stem}_recomputed.csv")
        old = pd.read_csv(APP_REPLAYS / f"{stem}_summary.csv")
        fields = ["origin", "service", "before_winner_id", "after_winner_id", "before_top", "after_top"]
        trial_check.append({"comparison": label, "searches": len(new),
                            "matching recorded top-three lists": int(new[fields].fillna("").eq(old[fields].fillna("")).all(axis=1).sum())})
    display(pd.DataFrame(trial_check))
    display(pd.read_csv(APP_RUN / "legacy_to_final_service_scores.csv"))

### Score the recommendation benchmark

Sixteen fixed searches combine four project regression cases with twelve evidence-panel cases, balanced eight urban and eight rural. The reference file identifies documented branch/service positives and specific negative cases using source passages and known branch or location problems. It only supports judgments about the cases actually reviewed. The benchmark scores anchor retrieval, branch identity, citations, duplicates, transport consistency and output status separately.

The archived November 22 prototype run returned five lists, reported seven unsupported queries and failed to resolve four places. One of the five scored lists retrieved a documented reference branch in its first five slots. The other 24 returned branches had no benchmark judgment. All 25 result cards contained an excerpt and URL. One URL directly supported the requested service at the reviewed branch, and 24 links were general facility record sources.

- **Jaipur dialysis** retrieved the JLN Marg branch of Soni Hospital in first place. A third-ranked result labeled Jodhpur appeared 5.2 km from the Jaipur origin, a location discrepancy that remained unreviewed.
- **Pithoragarh dialysis** put its first result about 84 km away by straight line and missed the local reference hospital.
- **Lucknow emergency C-section** showed routine C-section and LSCS wording without the emergency qualifier across all five result excerpts.
- **Coverage gaps** included five blood-bank cases, one exact Level III NICU case, and one public-transport MRI case. The unresolved places were Khunti, Keonjhar, Sanchore and Dantewada.

On the frozen workspace inputs, both later branches returned the same five lists by query, seven unsupported queries, and four place-resolution errors. The default branch reproduced the archived rankings. The restored app hit a reference in two of five scored lists. Its additional anchor was Charak Hospital in the Lucknow emergency search, where two aliases of the same documented branch appeared. All five restored Lucknow cards had a selected-service confidence of zero. The independent benchmark credited branch identity. The card excerpts left emergency scope unsubstantiated. In Jaipur the restored city centroid shifted Soni Hospital to second, still within the first five. In Pithoragarh the first five branches remained the same.

These are 16 frozen offline searches. The reports and source catalog refer to other evaluation panels and workflows elsewhere, including the 27-search ranking sensitivity experiment. Interpret unsupported cases and unresolved locations separately from ranked-list misses.

[Benchmark fixtures and scoring rules](https://github.com/augustchu/datathon-F26/tree/af8965890e4de2e5a5f2a24177425e5fdb77352d/recommendation-benchmark)

In [ ]:
existing_benchmark = BENCH_ROOT / "evaluation.json"
frozen_benchmark = load_json(existing_benchmark)
score_runs = {}
for label, prediction in [
    ("18335aa archived", BENCH_ROOT / "predictions.json"),
    ("18335aa workspace", APP_RUN / "historical-predictions.json"),
    ("af89658 workspace", APP_RUN / "main-predictions.json"),
    ("d682872 workspace", APP_RUN / "classic-predictions.json"),
]:
    if not prediction.is_file():
        continue
    output = APP_RUN / f"{label.replace(' ', '-').lower()}-evaluation.json"
    run_cmd([sys.executable, BENCH_CODE / "evaluate.py", "--predictions", prediction, "--output", output])
    score_runs[label] = load_json(output)

assert score_runs["18335aa archived"]["sha256"] == frozen_benchmark["sha256"]
assert score_runs["18335aa archived"]["current"] == frozen_benchmark["current"]
statuses = []
for label, report in score_runs.items():
    for stratum in ("urban", "rural"):
        results = report["current"]["groups"][stratum]
        statuses.append({
            "run": label, "stratum": stratum, "ranked lists": results["outcomes"]["recommendations"],
            "unsupported queries": results["outcomes"]["unsupported_query"],
            "resolution or runtime errors": results["outcomes"]["error"],
            "abstentions": results["outcomes"]["abstain"],
            "reference hit within top five": results["anchor_hits"],
            "lists with scored reference": results["anchor_queries_scored"],
            "judged support among returned": results["result_judgments"]["supported"],
            "returned branches awaiting judgment": results["result_judgments"]["unreviewed"],
            "results with excerpt and link": results["citations_present"],
            "duplicate branch results": results["duplicate_results"],
        })
statuses = pd.DataFrame(statuses)
display(statuses)
ax = statuses.set_index(["run", "stratum"])[
    ["ranked lists", "unsupported queries", "resolution or runtime errors", "abstentions"]
].plot.barh(stacked=True, color=["#218476", "#c3a04a", "#bc6859", "#8792a3"], figsize=(11.5, 5.2))
ax.set(title="Urban and rural outcomes across the same 16 queries", xlabel="Queries per group", ylabel="")
ax.legend(frameon=False, bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

panel_rows = []
for label, report in score_runs.items():
    for case in report["current"]["queries"]:
        best = case["results"][0]["facility"] if case["results"] else {}
        panel_rows.append({
            "run": label, "case": case["query"]["id"], "search place": case["query"]["origin"],
            "service": case["query"]["service"], "outcome": case["outcome"], "reference": case["anchor_check"],
            "first facility": best.get("name"), "first point distance (km)": best.get("travel_value"),
            "first confidence": best.get("service_confidence_score"), "first excerpt": best.get("evidence_excerpt"),
            "reason": case["reason"],
        })
app_panel = pd.DataFrame(panel_rows)
changed = app_panel[app_panel["case"].isin(["urban-01", "urban-08", "rural-06"])]
display(changed)
display(app_panel[app_panel["run"].eq("d682872 workspace")])

### Historical browser captures

The following four images are browser screenshots of the saved 18335aa report cards generated from the benchmark predictions. They show what the archived code returned even if workspace data or source pages change. They are historical report captures. The next section reproduces the restored app’s actual interactive result map with synthetic data.

In [ ]:
captures = {
    "urban-01-jaipur-dialysis.png": "Jaipur dialysis · documented result first",
    "rural-06-pithoragarh-dialysis.png": "Pithoragarh dialysis · distance and local retrieval gap",
    "rural-02-gadchiroli-unsupported.png": "Gadchiroli blood bank · unsupported search",
    "urban-08-lucknow-emergency-scope-gap.png": "Lucknow emergency C-section · service scope gap",
}
for filename, caption in captures.items():
    print(caption)
    show_rich_output(Image(filename=str(BENCH_ROOT / "screenshots" / filename), width=780))

### App tests and interactive map

Run the shipped main-branch tests and the benchmark evaluator’s regression cases. The Streamlit test then runs the intended classic entry point, selects the included synthetic demo, searches for Dialysis near PIN 411001, and checks that five ranked facilities appear with the origin and five pins. It exports the app’s map HTML so it can be inspected in the notebook. Leaflet assets and map tiles are fetched by the viewing browser when available.

For an authenticated Databricks App, sync the selected full Git folder, then run its `first-prototype/prepare_snapshot.py` notebook with its facility, PIN and NFHS tables and the ADM2 volume ZIP. Its default writable schema is `additional_india_data.default`. It publishes facilities, PIN, NFHS and metadata tables with the `first_prototype_` prefix. Deploy the chosen `first-prototype/` folder and map the `sql-warehouse`, `facilities-table`, and `snapshot-table` app resource keys. The app manifest also maps the optional TravelTime ID/key through secrets. The notebook replay uses workspace inputs and local outputs. Authenticated deployment, live routes and confirmations from facilities are outside this validation.

[Databricks snapshot and app configuration](https://github.com/augustchu/datathon-F26/blob/d68287276f3e9a275283eb005799c3be7db3bba5/first-prototype/README.md)

In [ ]:
run_cmd([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=MAIN_APP)
run_cmd([sys.executable, "-m", "unittest", "discover", "-s", ".", "-p", "test_evaluate.py", "-v"], cwd=BENCH_CODE)

app_test_script = APP_RUN / "classic_app_smoke.py"
app_test_script.write_text("""
import os
from pathlib import Path
import re
import sys
from html import escape
from streamlit.testing.v1 import AppTest

app, output = map(Path, sys.argv[1:])
sys.path.insert(0, str(app))
for name in ("DATABRICKS_HTTP_PATH", "DATABRICKS_WAREHOUSE_HTTP_PATH", "DATABRICKS_WAREHOUSE_ID"):
    os.environ.pop(name, None)
test = AppTest.from_file(str(app / "classic_dashboard.py"), default_timeout=45).run()
assert len(test.exception) == 0
services = next(item for item in test.selectbox if item.label == "Service")
assert len(services.options) == 18
services.select("Dialysis").run()
next(item for item in test.text_input if item.label == "PIN code").input("411001").run()
next(item for item in test.button if item.label == "Find facilities").click().run()
assert len(test.exception) == 0 and len(test.error) == 0
assert next(item for item in test.selectbox if item.label == "Sort by").value == "Recommended"
markup = next(element.proto.srcdoc for element in test.get("iframe") if element.proto.srcdoc)
assert "Your location" in markup and markup.count("L.marker(") == 6
card_headings = [element.value.removeprefix("### ") for element in test.markdown
                 if re.match("^### [1-5][.] ", element.value)]
positions = [markup.find(f"<b>{escape(heading)}</b>") for heading in card_headings]
assert len(positions) == 5 and all(position >= 0 for position in positions) and positions == sorted(positions)
output.write_text(markup, encoding="utf-8")
print("Classic app smoke passed: 18 services, Recommended sort, origin and five ranked pins.")
""", encoding="utf-8")
map_preview = APP_RUN / "classic-dialysis-app-map.html"
run_cmd([sys.executable, app_test_script, CLASSIC_APP, map_preview], cwd=CLASSIC_APP)
main_or_databricks_display = globals().get("displayHTML")
if callable(main_or_databricks_display):
    main_or_databricks_display(map_preview.read_text())
else:
    show_rich_output(HTML(map_preview.read_text()))

### End-to-end handoff to the restored app

This replay takes the checked organizer facility and NFHS inputs, the verified original capacity and confidence pair, and the forecast rebuilt in the caseload section. A rerun of the capacity publisher can supply both files when their SHA-256 values match the audited publication. The file on disk retains its original bytes. App preparation removes a spreadsheet-protection apostrophe from an ID only when the exact unescaped ID exists in the organizer input. The app snapshot uses the pinned app's preparation functions for capacity, service evidence, district geography and population. The projection table includes only routable branch–service rows with documented activity in the forecast's scope. The broader network scenario stays available for research.

The receipt identifies all inputs and records the transitions from source rows to snapshot rows to app load projections. The Streamlit test opens the actual restored app, supplies the generated snapshot at its SQL read boundary and reads the generated projection CSV from disk. It verifies a routine MRI search with the projected-load control enabled. Outputs stay in the run directory. Live deployment and live routes are separate tests.

In [ ]:
handoff_dir = APP_RUN / "restored-app-handoff"
handoff_dir.mkdir(exist_ok=True)
if not APP_INPUTS_AVAILABLE or not case_release_ok:
    print("The restored app handoff needs the checked facility/NFHS inputs and original capacity release.")
else:
    delivery_confidence = RELEASE / "facility_confidence.csv"
    if not delivery_confidence.is_file():
        delivery_confidence = APP_RELEASE / "facility_confidence.csv"
    handoff_capacity = case_pinned
    handoff_confidence = delivery_confidence
    expected_confidence = "369bdab17319015850732bab8607afad4333789cf7799d42338a986854f4bc21"
    regenerated_folder = globals().get("regenerated_capacity_release")
    if regenerated_folder:
        regenerated_capacity = Path(regenerated_folder) / "facilities_newest_capacity.csv"
        regenerated_confidence = Path(regenerated_folder) / "facility_confidence.csv"
        if (regenerated_capacity.is_file() and regenerated_confidence.is_file()
                and digest(regenerated_capacity) == case_expected_sha
                and digest(regenerated_confidence) == expected_confidence):
            handoff_capacity, handoff_confidence = regenerated_capacity, regenerated_confidence

    if digest(handoff_capacity) != case_expected_sha or digest(handoff_confidence) != expected_confidence:
        raise ValueError("The app handoff requires the exact aligned capacity and confidence release pair.")
    if network_meta["release"]["sha256"] != case_expected_sha or digest(forecasts) != network_meta["forecast_sha256"]:
        raise ValueError("The forecast and its input release must match the caseload receipt.")

    handoff_population = globals().get("population_outputs", APP_POPULATION.parent) / APP_POPULATION.name
    handoff_script = handoff_dir / "run_handoff.py"
    handoff_script.write_text("""
from datetime import datetime, timezone
from hashlib import file_digest
from pathlib import Path
from unittest.mock import patch
import json
import os
import sys

import pandas as pd
from streamlit.testing.v1 import AppTest

app, out, raw_path, nfhs_path, boundaries_path, population_path, cap_path, confidence_path, forecast_path = (
    map(Path, sys.argv[1:10])
)
release_sha = sys.argv[10]
sys.path.insert(0, str(app))
import classic_data
import pipeline
import projected_load

def sha(path):
    with path.open("rb") as handle:
        return file_digest(handle, "sha256").hexdigest()

raw = pd.read_json(raw_path, lines=True, dtype={"unique_id": "string"})
capacity = pd.read_csv(cap_path, dtype={"unique_id": "string"}, low_memory=False)
confidence = pd.read_csv(confidence_path)
# Undo a CSV spreadsheet-protection apostrophe only when the exact unescaped ID exists
# in the validated organizer input and the escaped spelling does not.
raw_ids = set(raw["unique_id"].str.strip())
released_ids = capacity["unique_id"].str.strip()
unescaped = released_ids.str.removeprefix("'")
fixes = released_ids.str.startswith("'") & ~released_ids.isin(raw_ids) & unescaped.isin(raw_ids)
capacity.loc[fixes, "unique_id"] = unescaped.loc[fixes]
assert not capacity["unique_id"].str.strip().isna().any()

bound = pipeline.attach_capacity_release(raw, capacity, confidence)
prepared = pipeline.prepare_facilities(bound)
nfhs_payload = json.loads(nfhs_path.read_text(encoding="utf-8"))
nfhs = pd.DataFrame(nfhs_payload["rows"], columns=nfhs_payload["columns"])
prepared = pipeline.attach_regions(prepared, json.loads(boundaries_path.read_text(encoding="utf-8")), nfhs)
prepared = pipeline.attach_population(prepared, pd.read_csv(population_path, low_memory=False))
prepared["snapshot_prepared_at"] = datetime.now(timezone.utc).isoformat()
prepared["facility_display_source"] = "Verified original capacity release"
assert len(prepared) == len(raw) == len(capacity) == len(confidence)
assert prepared["facility_id"].is_unique
assert prepared["confidence_score"].tolist() == confidence["confidence_score"].tolist()
assert prepared.loc[
    prepared["capacity_release_status"].isin({"branch_identity_hold", "review_unresolved", "target_quality_hold"}),
    "beds_for_scenario"
].isna().all()
snapshot_path = out / "facility_snapshot.jsonl"
prepared.to_json(snapshot_path, orient="records", lines=True, force_ascii=False)

network = pd.read_csv(forecast_path, dtype=str, keep_default_na=False, low_memory=False)
assert network["release_sha256"].eq(release_sha).all()
selected = network.loc[network["routing_eligible"].str.casefold().eq("true") &
                       network["service_evidence"].eq("reported_activity_in_scope")].copy()
schema = pd.read_csv(app / "projected_load.csv", nrows=0).columns
selected["projection_facility_id"] = selected["facility_id"]
assert selected["facility_id"].isin(prepared["facility_id"]).all()
assert not selected.duplicated(["facility_id", "care_needed"]).any()
assert selected["unit"].ne("").all() and selected["scope"].ne("").all() and selected["period"].ne("").all()
assert pd.to_numeric(selected["caseload"]).ge(0).all() and pd.to_numeric(selected["capacity"]).gt(0).all()
projection_path = out / "projected_load.csv"
selected.loc[:, schema].to_csv(projection_path, index=False)
pd.testing.assert_frame_equal(pd.read_csv(projection_path), pd.read_csv(app / "projected_load.csv"))

snapshot = pd.read_json(snapshot_path, lines=True, dtype=False)
metadata = pd.DataFrame([{"prepared_at": prepared["snapshot_prepared_at"].iloc[0]}])
def read_local_snapshot(table, _http_path):
    return metadata if table.endswith("_snapshot_metadata") else snapshot.copy()
projected_load.PROJECTION_PATH = projection_path
projected_load.load_projections.cache_clear()
os.environ["DATABRICKS_WAREHOUSE_ID"] = "local-handoff-test"
for key in ("DATABRICKS_HTTP_PATH", "DATABRICKS_WAREHOUSE_HTTP_PATH"):
    os.environ.pop(key, None)
with patch.object(classic_data, "_load_snapshot", side_effect=read_local_snapshot) as sql_reads, \
        patch.object(classic_data, "prefer_projected_load",
                     wraps=projected_load.prefer_projected_load) as load_calls:
    ui = AppTest.from_file(str(app / "classic_dashboard.py"), default_timeout=90).run()
    assert not ui.exception and not ui.error and sql_reads.call_count >= 1
    next(item for item in ui.selectbox if item.label == "Service").select("MRI").run()
    next(item for item in ui.selectbox if item.label == "Location type").select("Coordinates (lat/lon)").run()
    next(item for item in ui.text_input if item.label == "Latitude").input("20.35231764")
    next(item for item in ui.text_input if item.label == "Longitude").input("85.83294129")
    next(item for item in ui.toggle if item.label == "Use TravelTime driving estimates").set_value(False)
    next(item for item in ui.toggle if item.label == "Consider projected load").set_value(True)
    ui.run()
    next(item for item in ui.button if item.label == "Find facilities").click().run()
    assert not ui.exception and not ui.error
    assert load_calls.call_count and load_calls.call_args.args[1] == "MRI"
    assert projected_load.load_projections.cache_info().misses >= 1
    markup = next(item.proto.srcdoc for item in ui.get("iframe") if item.proto.srcdoc)
    assert "Your location" in markup and "L.marker(" in markup
    (out / "restored_mri_map.html").write_text(markup, encoding="utf-8")

paths = {
    "organizer_facility_ids_and_capacity": raw_path, "nfhs": nfhs_path, "district_geometry": boundaries_path,
    "population": population_path, "release_capacity": cap_path, "release_confidence": confidence_path,
    "network_forecast": forecast_path, "app_snapshot": snapshot_path, "app_projection": projection_path,
}
receipt = {"sha256": {label: sha(path) for label, path in paths.items()},
           "organizer_rows": len(raw), "capacity_release_rows": len(capacity),
           "snapshot_rows": len(snapshot), "charted_district_join_rows": int(prepared["regional_population"].notna().sum()),
           "network_forecast_rows": len(network), "load_projection_rows": len(selected),
           "spreadsheet_escaped_ids_normalized": int(fixes.sum()),
           "app_test": "restored Streamlit MRI search read both generated files and rendered its map"}
(out / "handoff_receipt.json").write_text(json.dumps(receipt, indent=2), encoding="utf-8")
print(json.dumps({key: value for key, value in receipt.items() if key != "sha256"}, indent=2))
""", encoding="utf-8")
    run_cmd([sys.executable, handoff_script, CLASSIC_APP, handoff_dir, app_input_raw, app_input_nfhs,
             app_input_boundaries, handoff_population, handoff_capacity, handoff_confidence,
             forecasts, case_expected_sha], cwd=CLASSIC_APP)
    handoff_receipt = load_json(handoff_dir / "handoff_receipt.json")
    display(pd.Series({key: value for key, value in handoff_receipt.items() if key != "sha256"}, name="Result"))
    handoff_map = (handoff_dir / "restored_mri_map.html").read_text(encoding="utf-8")
    if callable(globals().get("displayHTML")):
        displayHTML(handoff_map)
    else:
        show_rich_output(HTML(handoff_map))

## 08 · Research and development document index

The repository logs record the code path. The organizer workspace backup, local reports and workbooks provide additional evidence where interactive outputs or local decisions were preserved. The archive retains the original versions for comparison.

| Document or artifact | What it contributes | Reading note |
|---|---|---|
| **Facility Finder Development Report** | Latest chronology of the restored app, score curve, map, TravelTime, load, equity, and verification | Identifies `d682872` as the current restored app. Its screenshots and benchmark references span different commits. |
| **Facility Search App Development Report** | Earlier implementation and search decisions | Treat the feature description as a snapshot of that report’s date. |
| **CMU Datathon Project Report** | Original sources, grouped model cohorts, EDA figures, service evidence and project handoff | Earlier count experiments use smaller, distinct test cohorts. |
| **Find the right care – CMU Datathon**, Word and PDF | User-oriented walkthrough, capacity and verification, pricing scenario and demo links | The launch address and phone plan were described as prospective. |
| **Facility Evidence Audit.xlsx** | External evidence coverage, comparisons, cases and identity leads | Historical review queue, before the subsequent sampling and reviews. |
| **Capacity from Facility Descriptions.xlsx** | Extracted text evidence and model comparisons | Distinguishes observed target values from candidate description-recovered values. |
| **Facility capacity and verification.xlsx** | Later verification and rule-comparison workbook | Local deliverable from the rule-comparison stage, before the final tagged newest-source delivery. |
| **Databricks backup** | Two exported training notebooks and inventory/model metadata | The backup records read-only Marketplace catalog metadata and two notebook exports. |

[Original action log](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/ACTION_LOG.md) · [Project summary](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/non-ml/PROJECT_SUMMARY.md) · [ML decision log](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/ML/MODEL_EVALUATION.md) · [Recommendations evaluation](https://github.com/augustchu/datathon-F26/blob/af8965890e4de2e5a5f2a24177425e5fdb77352d/recommendation-benchmark/EVALUATION_LOG.md)

In [ ]:
from xml.etree import ElementTree

reports = ASSETS / "documents"
document_index = []
for path in sorted(reports.rglob("*")):
    if not path.is_file():
        continue
    item = {"document": path.name, "bytes": path.stat().st_size, "location": str(path)}
    if path.suffix == ".xlsx":
        with pd.ExcelFile(path, engine="openpyxl") as workbook:
            item["contents"] = ", ".join(workbook.sheet_names)
    elif path.suffix == ".docx":
        with zipfile.ZipFile(path) as docx:
            document = ElementTree.fromstring(docx.read("word/document.xml"))
            namespaces = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}
            item["contents"] = f"{len(document.findall('.//w:p', namespaces))} paragraphs · " + \
                f"{sum(name.startswith('word/media/') for name in docx.namelist())} embedded media items"
    document_index.append(item)
if document_index:
    display(pd.DataFrame(document_index))
else:
    print("Load the companion archive to inspect the local originals and worksheet inventory.")

milestones = [
    ("9168412", "Databricks capacity, NFHS enrichment and interpretable model notebooks"),
    ("10673ea", "Modular source, geography, count and service preparation"),
    ("67a4464", "Documentary service checks and three-mode real-data replay"),
    ("193415c", "Hindi IVR and verification workflow"),
    ("d443323", "Branch-specific location holds"),
    ("ec15b98", "Bounded numeric preprocessing for rare-input extrapolation"),
    ("c0506ba", "Snapshot-bound source-grounded recovery"),
    ("8348383", "Dated external count evidence and identity/scope rules"),
    ("5e6ec32", "Separate frozen source-selection iterations"),
    ("3975b0c", "Independent population and sequential load simulator"),
    ("b3270ab", "Tagged newest-source model and aligned release"),
    ("3cc87f7", "Main service-evidence confidence ranker"),
    ("18335aa", "Earlier app scorer benchmark and research publication"),
    ("e26886b", "Main ordered Folium map"),
    ("af89658", "Main projected-load comparison"),
    ("d682872", "Restored Facility Finder service confidence"),
]
chronology = []
for revision, decision in milestones:
    checkout = APP_GIT if revision == "d682872" else MAIN_GIT
    try:
        when = git("show", "-s", "--format=%aI", revision, cwd=checkout).decode().strip()
    except RuntimeError:
        when = None
    chronology.append({"commit": revision, "at_et": pd.Timestamp(when).tz_convert("America/New_York")
                       if when else None, "decision": decision,
                       "source": f"https://github.com/augustchu/datathon-F26/commit/{revision}"})
display(pd.DataFrame(chronology).sort_values("at_et", na_position="last"))

from urllib.parse import quote

repository_documents = []
for repo, checkout, revision, namespace in [
    ("Datathon main", MAIN_GIT, MAIN_SHA, "augustchu/datathon-F26"),
    ("Restored app", APP_GIT, APP_SHA, "augustchu/datathon-F26"),
    ("Standalone verification", VERIFICATION_GIT, VERIFY_SHA, "yana-157/in-the-loop-verification-tool"),
]:
    entries = git("ls-tree", "-r", "--name-only", "-z", revision, cwd=checkout).decode().split("\0")
    for entry in entries:
        if Path(entry).suffix.casefold() in {".md", ".ipynb", ".txt"} and "/raw/" not in entry:
            repository_documents.append({"repo": repo, "path": entry,
                                         "link": f"https://github.com/{namespace}/blob/{revision}/{quote(entry)}"})
repository_index = pd.DataFrame(repository_documents)
repository_index.to_csv(RUN / "repository-document-index.csv", index=False)
display(repository_index.groupby("repo").size().rename("tracked documents and notebooks"))
print(f"Full repository document index: {RUN / 'repository-document-index.csv'}")

### Read the results at the right scope

- Treat the restored branch as the intended app and both branch tips as independently testable implementations. The 16 archived cases measure the earlier scorer’s ranking, supported options and origin resolution.
- Treat population pressure, network load, equity and IVR as separate experiments with their own inputs, coverage and activation conditions. Live routing and calls require their service credentials and current operating consent.
- Keep source-reported capacity, verified changes, XGBoost resource estimates, release-wide completeness and service-level confidence distinct. The displayed recommendation depends on the named service and branch. Current admission and appointment availability require a direct check.
- For an exact historical replay, retain the audit inputs, branch revision, model and feature hashes, source edition, outcome receipt and ordered test rows. The printed status tells which checks executed against the current workspace inputs and which results came from the saved project record.